# 4-estimators: compensation

The closing table of stage 4. Each estimator the stage built gets one row with its bias, its noise at a stated delay, its delay, its valid flag and the share of rows it holds, its start-up, the state it keeps, the constants it reads, the term it corrects, how it corrects it and the bound it leaves. Stage 3's closing table sits beside it row by row, so every term stage 3 handed on is either compensated here, designed around, left as a bound or carried to a later stage. The notebook has no capture of its own, runs no estimator and opens no telemetry. It reads the five stage 4 result files and stage 3's compensation result, and every number below comes from one of them with the file and key beside it.

## Purpose

- One row per estimator. The current zero, the position, the back-EMF speed per row, the velocity as the pot's slope and as the back-EMF speed, the five blends, the rest verdict in its two forms, and the winding temperature with R(T). A cell with no result behind it reads "not measured".
- Stage 3's closing table, every row in stage 3's order, with what stage 4 did with each, stage 3's number beside stage 4's, and whether stage 4's result agrees with what stage 3 said to do.
- The quantities two stage 4 notebooks both state, side by side with the conditions each was measured under, and the pairs that differ by more than their brackets, each with its candidate causes and a confidence. None is reconciled.
- The four open choices, the brush term's form, the blend, the rest verdict's form and the tap floor's rung, with what each option costs and what would decide it. None is chosen.
- The gaps of every notebook in one list, each with the capture that would close it and the deliverable it limits.
- What stages 5 to 7 read from here, the README's table with the numbers.

## Procedure

No sitting ran for this notebook. Each result file is read through `oscnb.results`, as json and as the csv tables beside it, and the speed notebook's `.noise.csv` and `.filters.csv` are the only tables read. A value marked "derived" is arithmetic on two cited keys, made in a cell and used only to compare, never to fill a cell of an estimator's row. Brackets are as the files give them, +/- one standard error unless the unit says otherwise. Text no key holds, the state each estimator keeps, the confidence words, the dispositions, the candidate causes, the gaps of the speed and velocity notebooks and the captures, is this notebook's own and is written once in the cell that builds its table. The method follows an analysis note that was reviewed and reproduced before this notebook was written.

## Data

Everything read is under `stages/`, for the pair `mg90-d__dev-v006-2A-D`.

- `3-servo/compensation/results/`, stage 3's closing table and its gaps, with the stage 3 numbers the rows quote.
- `4-estimators/current-zero/results/`, `position/results/`, `speed/results/`, `velocity/results/` and `winding-temperature/results/`, the five estimators' result files.
- `4-estimators/README.md`, whose last table section 7 fills.

The libraries, then the pair.

In [1]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from oscnb import boards, results, servos

pd.set_option("display.width", 200)    # the printed tables run wide

In [2]:
SERVO = servos.SERVOS["mg90-d"]         # the servo the files describe
BOARD = boards.BOARDS[boards.DEFAULT]   # the board they were measured on
PAIR = f"{SERVO.key}__{BOARD.key}"      # the pair, the result file's name
PAIR_DIR = Path("../../../telemetry") / PAIR   # the capture of every file
STAGE, EXPERIMENT = "4-estimators", "compensation"  # where the result goes
print(SERVO.key, "on", BOARD.key, "from", results.rel(PAIR_DIR.resolve()))

mg90-d on dev-v006-2A-D from telemetry/mg90-d__dev-v006-2A-D


## 1. What this notebook reads

Six result files, each under a short name that every citation below uses as `name:key`. A key path walks the file's nested dicts and lists with a slash, and a key that itself holds a slash is matched whole. A cell of a csv table beside a file is cited as `name:.table.csv[column=value,...].column`.

In [3]:
FILES = {                               # short name: (stage, experiment)
    "s3": ("3-servo", "compensation"),
    "cz": ("4-estimators", "current-zero"),
    "pos": ("4-estimators", "position"),
    "spd": ("4-estimators", "speed"),
    "vel": ("4-estimators", "velocity"),
    "wt": ("4-estimators", "winding-temperature")}
R = {k: results.read(s, e, pair=PAIR) for k, (s, e) in FILES.items()}
READ = []                               # one line per file, for the table
for k, (s, e) in FILES.items():
    p = results.path(s, e, pair=PAIR)
    beside = sorted(c.name.split(".", 1)[1]  # the csv tables next to it
                    for c in p.parent.glob(f"{p.stem}.*.csv"))
    READ.append({"name": k, "file": str(results.rel(p)),
                 "keys": len(R[k]), "csv beside it": ", ".join(beside),
                 "gaps key": "gaps" in R[k], "capture": R[k]["capture"]})
READ = pd.DataFrame(READ)
print(READ.to_string(index=False))
print("stage 3's closing table:", len(R["s3"]["closing_table"]), "rows;",
      "stage 3's gaps:", len(R["s3"]["gaps"]))

name                                                                       file  keys                                   csv beside it  gaps key                         capture
  s3             stages/3-servo/compensation/results/mg90-d__dev-v006-2A-D.json    70                                                      True telemetry/mg90-d__dev-v006-2A-D
  cz        stages/4-estimators/current-zero/results/mg90-d__dev-v006-2A-D.json    78                             alpha.csv, step.csv      True telemetry/mg90-d__dev-v006-2A-D
 pos            stages/4-estimators/position/results/mg90-d__dev-v006-2A-D.json    58                         filters.csv, lookup.csv      True telemetry/mg90-d__dev-v006-2A-D
 spd               stages/4-estimators/speed/results/mg90-d__dev-v006-2A-D.json    47                          filters.csv, noise.csv     False telemetry/mg90-d__dev-v006-2A-D
 vel            stages/4-estimators/velocity/results/mg90-d__dev-v006-2A-D.json    89 blends.csv, crossover.csv, rest.cs

Every file is one servo on one board and names the same capture folder. The speed and velocity files carry no `gaps` key, so their gaps are this notebook's text, taken from those notebooks' closing sections. Stage 3's closing table has 17 rows and its gaps 6.

### How a number is read

A quantity in a file is a value with its bounds and unit. `Q` prints it as `value [lo, hi] unit` with its citation, at a precision set by the value's size, the same rule for every cell below. `num` is that rule alone, for a value with no unit.

In [4]:
def node(f, key):
    """The value at a slash path of file `f`; a key holding a slash is
    matched whole, a list is indexed by number."""
    v, parts, i = R[f], key.split("/"), 0
    while i < len(parts):
        if isinstance(v, list):         # a list: the part is an index
            v, i = v[int(parts[i])], i + 1
            continue
        for j in range(len(parts), i, -1):   # the longest key first
            k = "/".join(parts[i:j])
            if k in v:
                v, i = v[k], j
                break
        else:
            raise KeyError((f, key, parts[i]))
    return v


def num(x):
    """A number at the precision every table here uses."""
    if x is None:
        return "-"
    if isinstance(x, (bool, str)):
        return str(x)
    if isinstance(x, int) or float(x).is_integer() and abs(x) < 1e9:
        return str(int(x))
    a = abs(x)
    if a >= 100:
        return f"{x:.1f}"
    if a >= 10:
        return f"{x:.2f}"
    if a >= 1:
        return f"{x:.3f}"
    return f"{x:.3g}"

In [5]:
def val(f, key):
    """The bare value of a quantity, or the node itself."""
    v = node(f, key)
    return v["value"] if isinstance(v, dict) and "value" in v else v


def se_of(f, key):
    """Half the bracket, the se where the unit says +/- se."""
    v = node(f, key)
    return (v["hi"] - v["lo"]) / 2


def Q(f, key, cite=True):
    """A quantity as 'value [lo, hi] unit <f:key>'."""
    v = node(f, key)
    tag = f" <{f}:{key}>" if cite else ""
    if isinstance(v, dict) and "value" in v:
        s = num(v["value"])
        if v.get("lo") is not None or v.get("hi") is not None:
            s += f" [{num(v.get('lo'))}, {num(v.get('hi'))}]"
        if v.get("unit"):
            s += f" {v['unit']}"
        extra = [f"{k} {num(x)}" for k, x in v.items()   # n, sd and such
                 if k not in ("value", "lo", "hi", "unit")
                 and not isinstance(x, (dict, list))]
        if extra:
            s += " (" + ", ".join(extra) + ")"
        return s + tag
    if isinstance(v, list):
        return "[" + ", ".join(num(x) for x in v) + "]" + tag
    return num(v) + tag

In [6]:
def find(f, key, **match):
    """The index of the first row of a list of dicts that matches."""
    for i, row in enumerate(node(f, key)):
        if all(row.get(k) == x for k, x in match.items()):
            return i
    raise KeyError((f, key, match))


def table(f, suffix):
    """The csv table `suffix` beside file `f`."""
    p = results.path(*FILES[f], pair=PAIR)
    return pd.read_csv(p.with_name(f"{p.stem}.{suffix}.csv"))


def csvval(f, suffix, col, **match):
    """One cell of a csv table, picked by the columns in `match`."""
    t = table(f, suffix)
    for k, x in match.items():
        t = t[t[k] == x]
    assert len(t) == 1, (f, suffix, match, len(t))
    return float(t.iloc[0][col])


def csvcell(f, suffix, col, **match):
    """The same cell as text with its citation."""
    where = ",".join(f"{k}={x}" for k, x in match.items())
    return (f"{num(csvval(f, suffix, col, **match))} "
            f"<{f}:.{suffix}.csv[{where}].{col}>")

Three helpers read the lists two notebooks keep for stage 5, the speed notebook's bias by band and brush form and the velocity notebook's table by regime. A derived difference of two quantities is stated in units of their combined standard error, and a pair is a finding from the threshold below, the one the stage 4 notebooks use.

In [7]:
SIGMA_FINDING = 2.0     # se: a difference at or over it is a finding
print(f"a finding at {SIGMA_FINDING:g} combined se or more")


def sb(set_, band):
    """One row of the speed notebook's bias by band and form."""
    i = find("spd", "stage5_bias_by_band_form", set=set_, band=band)
    r = node("spd", f"stage5_bias_by_band_form/{i}")
    return (f"{set_} {band} ({num(r['rev_s'])} rev/s, {num(r['mA'])} "
            f"mA): table {num(r['table'])}, product {num(r['product'])} "
            f"rev/s <spd:stage5_bias_by_band_form/{i}>")


def reg(i):
    """One row of the velocity notebook's table by regime."""
    r = node("vel", f"stage5_by_regime/{i}")
    return (f"{r['source']} in '{r['regime']}': noise "
            f"{num(r['noise_rev_s'])} rev/s, bias {num(r['bias_rev_s'])} "
            f"rev/s, delay {r['delay']} <vel:stage5_by_regime/{i}>")


def sigma(a, sa, b, sb_, label):
    """a less b over the combined se, as a derived line."""
    d, s = a - b, (sa ** 2 + sb_ ** 2) ** 0.5
    z = d / s if s else float("nan")
    return (f"derived: {label}: difference {num(d)}, combined se "
            f"{num(s)}, {abs(z):.2f} sigma")

a finding at 2 combined se or more


A table is shown as markdown with each citation in code type. `sources` collects the citations of a row's text, and every row of the result file carries that list.

In [8]:
CITE = re.compile(r"<((?:s3|cz|pos|spd|vel|wt):[^<>]*)>")  # <file:key>


def sources(*texts):
    """The citations in the texts, sorted, each once."""
    return sorted({m for t in texts for m in CITE.findall(str(t))})


def cell_md(text):
    """A table cell: citations in code type, pipes and brackets escaped."""
    parts = CITE.split(str(text))   # odd items are the citations
    out = []
    for i, p in enumerate(parts):
        if i % 2:
            out.append("`" + p.replace("|", "\\|") + "`")
        else:
            out.append(p.replace("|", "\\|").replace("<", "&lt;"))
    return "".join(out)


def show(rows, cols, heads=None):
    """Rows of dicts as one markdown table."""
    lines = ["| " + " | ".join(heads or cols) + " |",
             "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell_md(r.get(c, "")) for c in cols) + " |"
              for r in rows]
    display(Markdown("\n".join(lines)))

## 2. One row per estimator

Each row holds the fields the README's rules of the desk ask of an estimator, its bias, its noise at a stated delay, its delay, its valid flag and the share of rows it holds, its start-up, the state it keeps, the constants it reads, the term it corrects, how it corrects it, the bound it leaves and the confidence its notebook gave it. Every number is cited beside itself. The state kept, the term corrected, how, and the confidence are this notebook's text, from the notebooks' own sections. The fields follow in that order for every row.

In [9]:
FIELDS = ["estimator", "bias", "noise_at_delay", "delay", "valid_flag",
          "valid_share", "start_up", "state", "constants",
          "term_corrected", "how", "bound_left", "confidence"]
ROWS1 = []                              # the closing table, in order


def add_row(**fields):
    """One estimator's row, its citations gathered into `sources`."""
    assert list(fields) == FIELDS, [f for f in FIELDS if f not in fields]
    ROWS1.append({**fields, "sources": sources(*fields.values())})


SHIFT = next(k for k, v in node("cz", "alpha_scan").items()   # the alpha
             if v["tau_s"]["value"] == val("cz", "tau_s"))    # of record
HOLD_P90 = ", ".join(f"{Q('cz', f'hold_p90_counts/{k}')} at {k[:-1]} s"
                     for k in node("cz", "hold_p90_counts"))
print("the zero's alpha of record:", SHIFT, "| hold p90:", HOLD_P90)

the zero's alpha of record: shift_13 | hold p90: 0.589 [0.551, 0.616] counts <cz:hold_p90_counts/1s> at 1 s, 0.812 [0.737, 0.965] counts <cz:hold_p90_counts/10s> at 10 s, 0.946 [0.877, 0.992] counts <cz:hold_p90_counts/60s> at 60 s


In [10]:
CZ_DESIGN = "tracked_zero_cost_per_sweep/design/warm"   # spd keys of the
CZ_TRACKED = "tracked_zero_cost_per_sweep/tracked/warm"  # zero as a speed
ROW = dict(
    estimator="current zero (awake zero): stage 1's two-zero tracker with "
    "the start rule and the drive-feed constant",
    bias=f"warm by rail band: {Q('cz', 'stage5_table/0/value')} "
    f"({node('cz', 'stage5_table/0/note')}); stage 1's tracker alone warm "
    f"mean {Q('cz', 'warm_error_mean_counts')}; as speed: design "
    f"{Q('spd', CZ_DESIGN + '/mean')} over {Q('spd', CZ_DESIGN + '/sweeps')}"
    f", stage 1's tracker {Q('spd', CZ_TRACKED + '/mean')}",
    noise_at_delay=f"warm scatter {Q('cz', 'design_warm_rms_counts')} rms "
    f"at the truth's floor {Q('cz', 'truth_floor_counts')}; the zero's own "
    f"noise {Q('cz', f'alpha_scan/{SHIFT}/zero_noise_white_counts')} white,"
    f" {Q('cz', f'alpha_scan/{SHIFT}/zero_move_2s_coast_counts')} over a "
    f"2 s coast; the sample per tick "
    f"{Q('cz', 'trough_sd_per_tick_counts/asleep')} asleep, "
    f"{Q('cz', 'trough_sd_per_tick_counts/coast')} coast, "
    f"{Q('cz', 'trough_sd_per_tick_counts/narrow_drive')} narrow drive",
    delay=f"time constant {Q('cz', 'tau_s')} of feed; flat band of the "
    f"shift {Q('cz', 'alpha_flat_band_shift')}",
    valid_flag="fed or seeded since boot or a rail move, and inside the "
    "hold bound",
    valid_share=f"fed on {Q('cz', 'feed_share_awake_rows')} of awake rows "
    f"and {Q('cz', 'feed_share_drive_rows')} of drive rows; longest unfed "
    f"awake stretch {Q('cz', 'longest_unfed_awake_s')}; hold p90 {HOLD_P90}",
    start_up="to half a count in "
    f"{Q('cz', 'start_rule_settle_to_half_count_fed_s')} of feed (median, "
    f"hi the worst) against {Q('cz', 'settle_to_half_count_fed_s')} for "
    f"stage 1's tracker alone; start events "
    f"{Q('cz', 'design_events_rms_counts')} rms against "
    f"{Q('cz', 'stage1_events_rms_counts')}; boot error without the rule "
    f"{Q('cz', 'start_error_counts/boot')}",
    state="two zeros as 64-bit floats in counts, their fresh counters, the "
    "rail the awake zero was learned at, the last rail sample, the running "
    "mean of the drive-feed offset")

In [11]:
ROW.update(
    constants=f"alpha 2^-{SHIFT.split('_')[1]} <cz:alpha_scan/{SHIFT}> "
    f"(tau {Q('cz', 'tau_s')}), feed gate {Q('cz', 'feed_gate_pct')} "
    f"(clean to {Q('cz', 'feed_gate_clean_to_pct')}), step line "
    f"{Q('cz', 'awake_step_slope_counts_per_v')} and "
    f"{Q('cz', 'awake_step_intercept_counts')} or the curve of .step.csv, "
    f"seed threshold {Q('cz', 'seed_rail_threshold_v')}, debounce "
    f"{Q('cz', 'seed_debounce_ticks')}, drive-feed offset "
    f"{Q('cz', 'drive_feed_offset_counts')}",
    term_corrected="the current channel's zero while the bridge is awake "
    "and its step with the rail (stage 3 row 8)",
    how="learn the awake zero from trough rows of drives at or under the "
    "gate (the drive-feed offset taken off), settled coasts in the crest "
    "column and settled torque off; seed awake = asleep + step(rail) at "
    "boot and on a rail move past the threshold; subtract with its fraction",
    bound_left=f"all runs {Q('cz', 'design_all_runs_rms_counts')} rms, "
    f"worst {Q('cz', 'design_all_runs_worst_counts')} (stage 1's tracker "
    f"alone {Q('cz', 'stage1_all_runs_rms_counts')}, worst "
    f"{Q('cz', 'stage1_all_runs_worst_counts')}); stall holds on the proxy "
    f"truth {Q('cz', 'stall_hold_error_counts_rms')}; a count is "
    f"{Q('spd', 'rev_s_per_count')}",
    confidence="high on this servo; moderate for the step curve: "
    f"{node('cz', 'gaps/1/gap')} <cz:gaps/1/gap>")
add_row(**ROW)

In [12]:
ROW = dict(
    estimator="position: lookup indexed by counts, then a boxcar of 16 rows",
    bias=f"table term held out {Q('pos', 'table_term_heldout_deg_rms')} "
    f"({Q('pos', 'table_term_heldout_counts_rms')}); nothing out, stamp "
    f"lead applied {Q('pos', 'heldout_lead_applied_deg_rms')}, no lead "
    f"{Q('pos', 'heldout_no_lead_deg_rms')}; end lines "
    f"{Q('pos', 'end_residual_low_deg_rms')} low, "
    f"{Q('pos', 'end_residual_high_deg_rms')} high; by direction "
    f"{Q('pos', 'direction_offset_crawl_plus_deg')} and "
    f"{Q('pos', 'direction_offset_crawl_minus_deg')} on steady crawl "
    "holds, not carried",
    noise_at_delay=f"per row {Q('pos', 'pot_noise_row_counts')}; boxcar "
    f"16 {Q('pos', 'filter_boxcar16_noise_counts')} "
    f"({Q('pos', 'filter_boxcar16_noise_deg')}) at "
    f"{Q('pos', 'filter_boxcar16_delay_ticks')}; the other filters in "
    "the position file's .filters.csv",
    delay=f"lookup none; boxcar 16 {Q('pos', 'filter_boxcar16_delay_ticks')}"
    f", {Q('pos', 'filter_boxcar16_lag_pass_deg')} behind at the passes' "
    "speed",
    valid_flag="no flag",
    valid_share=f"valid range {Q('pos', 'stage5_table/9/value')} "
    f"({node('pos', 'stage5_table/9/note')}); share of rows: not measured",
    start_up=Q("pos", "stage5_table/10/value"))

In [13]:
ROW.update(
    state="the lookup none; the boxcar its last 16 rows",
    constants=f"{Q('pos', 'lookup_entries_guard_to_guard')} at pitch "
    f"{Q('pos', 'lookup_pitch_counts')} from "
    f"{Q('pos', 'lookup_knot_first_counts')} to "
    f"{Q('pos', 'lookup_knot_last_counts')} (the position file's "
    ".lookup.csv)",
    term_corrected="the pot's scale, smooth bend and fine texture (stage 3 "
    "rows 0 and 1); the board's tick pattern, period "
    f"{Q('pos', 'tick_pattern_period_ticks')}",
    how="linear interpolation between knots uniform in counts; the boxcar "
    f"of 16 nulls the pattern ({Q('pos', 'tick_pattern_pp_counts')} pp, "
    f"{Q('pos', 'tick_pattern_variance_share')} of the variance)",
    bound_left=f"direction {Q('pos', 'direction_bound_moving_deg')} in "
    f"motion, {Q('pos', 'direction_bound_rest_deg')} at rest; carries "
    f"blind over {Q('pos', 'carry_blind_span_deg')} deg at codes "
    f"{Q('pos', 'carry_codes')}; high end hole "
    f"{Q('pos', 'high_end_hole_deg')}; the stamp lead's share of the error "
    f"{Q('pos', 'share_stamp_lead')}",
    confidence="high for the lookup's pitch and the direction bound, "
    "moderate for the bound's size; one sitting")
add_row(**ROW)

In [14]:
ROW = dict(
    estimator="back-EMF speed per row (stage 3's four-term form, tracked "
    "zero, k_d = 0), brush term as the signed table or as the product",
    bias="; ".join([sb("ladder (third repeat)", "30-100"),
                    sb("ladder (third repeat)", "250-350"),
                    sb("holds (free, low)", "0-30"),
                    sb("run-up", "160-250"), sb("run-up", "350-450"),
                    sb("lifts", "0-30"), sb("lifts", "100-160"),
                    sb("load holds", "0-30")])
    + "; flat band over the ladder's rungs: table "
    f"{Q('spd', 'brush_flat_band_rev_s/table')}, product "
    f"{Q('spd', 'brush_flat_band_rev_s/product')}",
    noise_at_delay="per row (delay 0), table form: rest "
    f"{csvcell('spd', 'noise', 'table_sd', kind='rest', band='rest')}, "
    "crawl holds "
    f"{csvcell('spd', 'noise', 'table_sd', kind='hold', band='0-30')} and "
    f"{csvcell('spd', 'noise', 'table_sd', kind='hold', band='30-100')}, "
    "ladder "
    f"{csvcell('spd', 'noise', 'table_sd', kind='ladder', band='100-160')}"
    " to "
    f"{csvcell('spd', 'noise', 'table_sd', kind='ladder', band='250-350')}"
    f" rev/s; flat band {Q('spd', 'row_noise_flat_band')}; boxcar 32: "
    + csvcell("spd", "filters", "sd_table", kind="hold", band="0-30",
              filter="box 32") + " at a crawl, "
    + csvcell("spd", "filters", "sd_table", kind="ladder", band="250-350",
              filter="box 32") + " at 250-350 rev/s")

In [15]:
COV = "valid_coverage_by_set/all"       # the speed file's coverage, all sets
ROW.update(
    delay="none per row but L / R under acceleration; filters add (run-ups,"
    " rows): " + ", ".join(
        f"{k} " + csvcell("spd", "filters", "runup_delay_added",
                          kind="ladder", band="30-100", filter=k)
        for k in ("box 16", "box 32", "pole 32", "lock 1"))
    + "; measured less nominal "
    f"{Q('spd', 'filter_delay_flat_band_rows/box')}",
    valid_flag=f"on-time at least {Q('spd', 'valid_rule/tap_floor_ticks')}"
    f", {Q('spd', 'valid_rule/start_rows')} since torque on or a flip, not "
    f"a dropout row ({Q('spd', 'valid_rule/dropout_frac')} over "
    f"{Q('spd', 'valid_rule/dropout_reference_rows')})",
    valid_share=f"valid on {Q('spd', COV + '/valid_pct')} % of "
    f"{Q('spd', COV + '/driven_rows')} driven rows of "
    f"{Q('spd', COV + '/sweeps')} sweeps; under the floor "
    f"{Q('spd', COV + '/under_floor_pct')} %, start-up "
    f"{Q('spd', COV + '/start_up_pct')} %, dropout "
    f"{Q('spd', COV + '/dropout_pct')} %",
    start_up=f"{Q('spd', 'valid_rule/start_rows')}; the error settles by "
    f"row {Q('spd', 'startup_profile/settled_rows/none')}, decay "
    f"{Q('spd', 'startup_profile/decay_rows')}; row 0 reads "
    f"{Q('spd', 'startup_profile/median/row 0')} rev/s at the median",
    state="the tracked zero's state, the previous row's speed and "
    "direction (the table's newton steps), the filter's state, the valid "
    "rule's two row counters, its one-pole mean and flag count")

In [16]:
TOP = sb("ladder (third repeat)", "450-600")    # the unresolved top rungs
FLOOR_Q = node("spd", "valid_rule/tap_floor_ticks")
ROW.update(
    constants=", ".join(
        f"{k} {Q('spd', f'constants_used/{k}')}" for k in (
            "counts_per_a", "r_ohm", "t0_c", "v0_v", "g_s", "brake_b0_v",
            "brake_b1_ohm", "l_h", "ke_v_per_rev_s"))
    + "; the table <spd:brush_forms/table_signed_mv> or the product c0 "
    f"{Q('spd', 'brush_forms/product_ladder_alone/c0_mv')}, c_dir "
    f"{Q('spd', 'brush_forms/product_ladder_alone/c_dir_mv')}, b "
    f"{Q('spd', 'brush_forms/product_ladder_alone/b_mv_per_rev_s_a')}",
    term_corrected="the reading term, R(T) i, V0, the brush drop with its "
    "direction term and the low-speed offset (stage 3 rows 5, 6, 7, 10), "
    "on the tracked zero (row 8)",
    how="construction C per row; the product in closed form, the table by "
    f"{Q('spd', 'solve_recommendation/steps')} newton steps from the row "
    f"before (worst solve error "
    f"{Q('spd', 'solve_recommendation/worst_rms_over_noise')})",
    bound_left="stopped rotor under current: table "
    f"{Q('spd', 'rest_bias_by_current/table/a')} + "
    f"{Q('spd', 'rest_bias_by_current/table/b_per_a')}, mean "
    f"{Q('spd', 'rest_bias_by_current/mean/table')}; duty step direct path "
    f"{Q('spd', 'duty_step_response/share_rows_0_3')}, "
    f"{Q('spd', 'duty_step_response/share_rows_4_9')}, "
    f"{Q('spd', 'duty_step_response/share_rows_10_39')}; under the floor "
    f"not valid; top rungs not resolved ({TOP})",
    confidence="high on the free shaft below the 450-600 rev/s band; "
    "moderate on the lifts (pot truth); moderate for the floor at "
    f"{num(FLOOR_Q['value'])} ticks, high at {num(FLOOR_Q['hi'])} "
    "<spd:valid_rule/tap_floor_ticks>")
add_row(**ROW)
print(len(ROWS1), "rows so far")

3 rows so far


In [17]:
REG_POT_5, REG_POT_50 = 2, 3            # the velocity file's regime rows
CARRY_WIN = Q("vel", "carry_mask_until_window_ms")   # the skip's reach
ROW = dict(
    estimator="velocity as the pot's slope (two-point over a multiple of "
    "16 rows, or two boxcars of 16)",
    bias=f"{reg(REG_POT_5)}; {reg(REG_POT_50)}; pot5 at a crawl "
    f"{Q('vel', 'blend_bias_crawl_rev_s/pot5')}, pot25 "
    f"{Q('vel', 'blend_bias_crawl_rev_s/pot25')}",
    noise_at_delay="own noise at a crawl: "
    f"{node('vel', f'stage5_by_regime/{REG_POT_5}/delay')} "
    f"{Q('vel', f'stage5_by_regime/{REG_POT_5}/noise_rev_s')} rev/s, "
    f"{node('vel', f'stage5_by_regime/{REG_POT_50}/delay')} under the "
    f"floor {Q('vel', f'stage5_by_regime/{REG_POT_50}/noise_rev_s')} rev/s"
    f"; own sd pot5 {Q('vel', 'blend_own_sd_rev_s/crawl/pot5')}, pot25 "
    f"{Q('vel', 'blend_own_sd_rev_s/crawl/pot25')}; at rest: two-point "
    f"{Q('vel', 'pot_slope_rest_noise_5ms_deg_s/two_point')}, two "
    f"boxcars {Q('vel', 'pot_slope_rest_noise_5ms_deg_s/box16')}; angle "
    "error per window "
    f"{Q('vel', 'pot_slope_angle_error_per_window_deg/crawl 15-35')}",
    delay="half the window; measured on kick ends pot5 "
    f"{Q('vel', 'blend_delay_rows/kick end, hold over the floor/pot5')}, "
    "pot25 "
    f"{Q('vel', 'blend_delay_rows/kick end, hold over the floor/pot25')}; "
    "residual against nominal "
    f"{Q('vel', 'pot_slope_delay_resid_rows/up_to_128')}",
    valid_flag="no flag; the carry band's rows are skipped",
    valid_share=f"carry band +/-{Q('vel', 'carry_mask_half_counts')} "
    f"skipped under {CARRY_WIN}; a window holds a band row on "
    f"{Q('vel', 'carry_rows_touched_share_crawl/6.4 ms')} to "
    f"{Q('vel', 'carry_rows_touched_share_crawl/200 ms')}; the band itself "
    f"{Q('vel', 'carry_band_rows_share_crawl')}",
    start_up="not measured",
    state="the last n + 16 rows, or two running sums and the rows between, "
    "and the row marks of the carry band")

In [18]:
NEAR = "carry_near_rms_deg_s"           # the velocity file's carry scores
ROW.update(
    constants=f"the position lookup, the carries' codes "
    f"{Q('pos', 'carry_codes')} and missing runs "
    f"{Q('pos', 'carry_missing_codes')}, the window as a multiple of "
    f"{Q('vel', 'pot_slope_two_point_multiple_of_rows')} (off a multiple "
    f"{Q('vel', 'pot_slope_off_multiple_penalty')})",
    term_corrected="the pot's texture and the converter's carries in a "
    "velocity (stage 3 rows 1 and 2); the only source under the tap floor",
    how="difference of the lookup's angle over the window, the carry "
    f"band's rows left out under {CARRY_WIN} of window, never held",
    bound_left="near a carry, no mask: "
    f"{Q('vel', NEAR + '/none/near/25.6 ms')} against far "
    f"{Q('vel', NEAR + '/none/far/25.6 ms')}; skip12 "
    f"{Q('vel', NEAR + '/skip12/near/25.6 ms')}; hold12 "
    f"{Q('vel', NEAR + '/hold12/near/25.6 ms')}; over a kick the "
    "commutation count runs "
    f"{Q('vel', 'count_kick_start_145ms/ratio_count_pot')} of the pot's "
    "angle",
    confidence="moderate, one sitting, over the carries' crossings at a "
    "crawl")
add_row(**ROW)

In [19]:
REG_EMF = 0                             # the back-EMF's regime row
EMF_BOX = "emf_noise_boxcar_rev_s/crawl valid w_tab boxcar"
ROW = dict(
    estimator="velocity as the back-EMF speed through a boxcar (over the "
    "tap floor)",
    bias=f"crawl against the tag: table "
    f"{Q('vel', 'emf_bias_crawl_rev_s/table')}, product "
    f"{Q('vel', 'emf_bias_crawl_rev_s/product')}; against the tach: table "
    f"{Q('vel', 'emf_bias_crawl_tach_rev_s/table')}, product "
    f"{Q('vel', 'emf_bias_crawl_tach_rev_s/product')}; by direction table "
    f"{Q('vel', 'emf_bias_by_direction_rev_s/table/+')} and "
    f"{Q('vel', 'emf_bias_by_direction_rev_s/table/-')}, product "
    f"{Q('vel', 'emf_bias_by_direction_rev_s/product/+')} and "
    f"{Q('vel', 'emf_bias_by_direction_rev_s/product/-')}; pass: table "
    f"{Q('vel', 'emf_bias_pass_rev_s/table')} "
    f"({node('vel', 'emf_bias_pass_rev_s/confidence')}); under the floor "
    f"table {Q('vel', 'emf_bias_under_floor_rev_s/table')}, product "
    f"{Q('vel', 'emf_bias_under_floor_rev_s/product')}",
    noise_at_delay=f"{reg(REG_EMF)}; per row at a crawl "
    f"{Q('vel', 'emf_noise_row_rev_s/crawl valid')}; boxcar 32 "
    f"{Q('vel', EMF_BOX + ' 32')}; boxcar 200 {Q('vel', EMF_BOX + ' 200')}"
    f"; commutation-locked {Q('vel', 'emf_commutation_locked_share_crawl')}"
    ", left by a boxcar of 32 "
    f"{Q('vel', 'emf_commutation_locked_sd_rev_s/boxcar_32')}, of 100 "
    f"{Q('vel', 'emf_commutation_locked_sd_rev_s/boxcar_100')}",
    delay="half the boxcar; direct path from a duty step through a boxcar "
    f"of 32: {Q('vel', 'emf_direct_path_share/boxcar_32/r0_3')}, "
    f"{Q('vel', 'emf_direct_path_share/boxcar_32/r4_9')}, "
    f"{Q('vel', 'emf_direct_path_share/boxcar_32/r10_39')}",
    valid_flag="the speed notebook's rule",
    valid_share=f"valid on {Q('vel', 'emf_valid_share_camera')}; "
    f"{Q('vel', 'emf_crawl_rows_under_floor_share')} are under the floor",
    start_up=f"the speed rule's {Q('spd', 'valid_rule/start_rows')} plus "
    "the boxcar")

In [20]:
KICK_END = "kick_end_emf_minus_pot_rev_s"   # back-EMF less pot, by window
ROW.update(
    state="the speed estimator's state and the boxcar's rows",
    constants="the speed estimator's",
    term_corrected="none of its own; it carries the back-EMF speed's bias "
    "into the velocity",
    how="boxcar of the valid rows",
    bound_left="after a hard deceleration back-EMF less pot "
    f"{Q('vel', KICK_END + '/002-005/emf_minus_pot')}, "
    f"{Q('vel', KICK_END + '/010-020/emf_minus_pot')}, "
    f"{Q('vel', KICK_END + '/020-030/emf_minus_pot')}; stopped rotor "
    f"{Q('vel', 'emf_stopped_rotor_rev_s/table/line_a')} + "
    f"{Q('vel', 'emf_stopped_rotor_rev_s/table/line_b_per_a')}; direction "
    "split under 45 rev/s table "
    f"{Q('vel', 'emf_direction_split_rev_s/table/under_45')}, product "
    f"{Q('vel', 'emf_direction_split_rev_s/product/under_45')}",
    confidence="high for the crawl bias's sign, moderate for its size; "
    "one sitting")
add_row(**ROW)
print(len(ROWS1), "rows so far")

5 rows so far


The five blends make one velocity of the two sources. The switch, `sw5`, takes the back-EMF where it is valid and the pot's slope elsewhere. The complementary pair, `cp5` and `cp25`, takes the pot's slope below a fixed corner and the back-EMF above it. `swr5` and `cpr5` add the rest verdict. None is chosen.

In [21]:
BLENDS = {   # candidate: (label, its regime row, its seams)
    "sw5": ("the switch (back-EMF where valid, else the pot's 5 ms slope)",
            8, ("sw5_kick", "sw5_coast")),
    "cp5": ("the complementary pair, pot slope of 5 ms", 9,
            ("cp5_handover", "cp5_coast")),
    "cp25": ("the complementary pair, pot slope of 25 ms", 10,
             ("cp25_coast",))}
for c, (label, i, _) in BLENDS.items():  # each regime row names its blend
    assert node("vel", f"stage5_by_regime/{i}/source") == c, (c, i)


def seam(s):
    """A blend's seam: median, p90, the row it sits at and n."""
    k = f"blend_seam_rev_s/{s}"
    return (f"seam {s}: median {Q('vel', k + '/median')}, p90 "
            f"{Q('vel', k + '/p90')}, at {Q('vel', k + '/row')}, n "
            f"{Q('vel', k + '/n')}")


def share(c, parts):
    """A blend's share of rows by source."""
    return ", ".join(f"{p} {Q('vel', f'blend_share_of_rows/{c}/{p}')}"
                     for p in parts)

In [22]:
for c, (label, i, seams) in BLENDS.items():
    sw = c == "sw5"                     # the switch keeps no state
    add_row(
        estimator=f"velocity blend {c}: {label}",
        bias=f"crawl {Q('vel', f'blend_bias_crawl_rev_s/{c}')}; against "
        f"the tach {Q('vel', f'blend_bias_tach_crawl_rev_s/{c}')}" + (
            "" if c == "cp25" else "; stall over the floor "
            f"{Q('vel', f'blend_bias_stall_over_floor_rev_s/{c}')}"),
        noise_at_delay="own sd crawl "
        f"{Q('vel', f'blend_own_sd_rev_s/crawl/{c}')}, pass "
        f"{Q('vel', f'blend_own_sd_rev_s/pass/{c}')}, under the "
        f"floor {Q('vel', f'blend_own_sd_rev_s/under_floor/{c}')} "
        f"({reg(i)})",
        delay="against the pot's centered slope: kick "
        f"{Q('vel', f'blend_delay_rows/kick/{c}')}, kick end "
        f"{Q('vel', f'blend_delay_rows/kick end, hold over the floor/{c}')}"
        f", reversal {Q('vel', f'blend_delay_rows/reversal/{c}')}",
        valid_flag="none of its own; the back-EMF's valid rule sets the "
        "shares",
        valid_share="not measured" if c == "cp25" else share(
            c, ("pot", "back-EMF", "pair")),
        start_up="; ".join(seam(s) for s in seams),
        state="nothing of its own" if sw else "one one-pole (corner "
        f"{Q('vel', 'blend_corner_rows')}) and the back-EMF through the "
        "pot's kernel",
        constants="the two sources' and the valid rule's" if sw else
        "the two sources' and the fixed corner",
        term_corrected="the back-EMF's invalid rows under the floor" if sw
        else "the back-EMF's DC bias (reads the pot at DC) and its "
        "invalid rows",
        how="pick by the valid flag" if sw else "pot slope below the "
        "corner, back-EMF above it, fixed weights",
        bound_left=("direct path "
                    f"{Q('vel', 'emf_direct_path_share/boxcar_32/r10_39')}"
                    if sw else "direct path " + ", ".join(
                        Q("vel", f"emf_direct_path_share/pair_{c[2:]}/{r}")
                        for r in ("r10_39", "r40_199"))) + "; not chosen",
        confidence="moderate, one sitting")

In [23]:
for c, label in (("swr5", "the switch with the rest verdict"),
                 ("cpr5", "the pair with the rest verdict on pot rows")):
    add_row(
        estimator=f"velocity blend {c}: {label}",
        bias=f"crawl {Q('vel', f'blend_bias_crawl_rev_s/{c}')}; stall over "
        f"the floor {Q('vel', f'blend_bias_stall_over_floor_rev_s/{c}')}",
        noise_at_delay="own sd crawl "
        f"{Q('vel', f'blend_own_sd_rev_s/crawl/{c}')}, pass "
        f"{Q('vel', f'blend_own_sd_rev_s/pass/{c}')}",
        delay=f"kick {Q('vel', 'blend_delay_rows/kick/swr5')}"
        if c == "swr5" else "not measured",
        valid_flag="the rest verdict, then the blend's sources",
        valid_share=share(c, ("rest", "pot", "back-EMF", "pair")),
        start_up="seam at a kick: median "
        f"{Q('vel', f'blend_seam_rev_s/{c}_kick/median')}, p90 "
        f"{Q('vel', f'blend_seam_rev_s/{c}_kick/p90')}, at "
        f"{Q('vel', f'blend_seam_rev_s/{c}_kick/row')}" + (
            "; with the veto (swv5) "
            f"{Q('vel', 'blend_seam_rev_s/swv5_kick/median')} at "
            f"{Q('vel', 'blend_seam_rev_s/swv5_kick/row')}"
            if c == "swr5" else ""),
        state="the blend's and the verdict's row count",
        constants="the blend's and the verdict's",
        term_corrected="the pot slope's noise and the back-EMF's "
        "stopped-rotor reading at rest (reports zero)",
        how="zero while the verdict is set",
        bound_left="the seam at every start; not chosen",
        confidence="moderate, one sitting")
print(len(ROWS1), "rows so far")

10 rows so far


The rest verdict calls the horn at rest. Its two forms are a band in counts around an anchor and the pot's slope under a multiple of its own rest noise, each held for a hold time. They are equal alternatives in the velocity notebook, and neither is chosen.

In [24]:
REST = {   # form: (its key in the velocity file, its threshold as text)
    "band": ("band 2 T50", f"+/-{Q('vel', 'rest_verdict/band/counts')} "
             f"counts of an anchor = {Q('vel', 'rest_verdict/band/sigma16')}"),
    "slope": ("slope 192 T50", f"the {Q('vel', 'rest_verdict/slope/rows')}"
              "-row two-boxcar slope under "
              f"{Q('vel', 'rest_verdict/slope/threshold_x_own_sd')} x its "
              "rest sd = "
              f"{Q('vel', 'rest_verdict/slope/threshold_deg_s')}")}
FAR = "rest_min_speed_deg_s/50 ms"      # slowest motion told from rest
for form, (key, thr) in REST.items():
    k25, k50 = key.replace(" T50", " 25"), key.replace(" T50", " 50")
    fm, fm3 = (f"rest_false_motion_share/{key}",
               f"rest_false_motion_stage3_share/{key}")
    ROW = dict(
        estimator=f"rest verdict, {form} form "
        f"({node('vel', 'rest_verdict/note')})",
        bias="false rest at 10 to 35 deg/s: "
        f"{Q('vel', f'rest_false_rest_10_35_deg_s/{k50}')} at 50 ms, "
        f"{Q('vel', f'rest_false_rest_10_35_deg_s/{k25}')} at 25 ms (near "
        f"a carry {Q('vel', f'rest_false_rest_10_35_near_carry/{k25}')}) "
        f"over {Q('vel', 'rest_false_rest_rows_10_35')}; false motion: "
        f"baselines {Q('vel', fm + '/rest_base')}, coast tails "
        f"{Q('vel', fm + '/rest_coast')}, stalls "
        f"{Q('vel', fm + '/stall72')}, load holds "
        f"{Q('vel', fm3 + '/load holds')}",
        noise_at_delay=f"threshold {thr}; sigma16 "
        f"{Q('vel', 'rest_sigma16_counts')}",
        delay="declares "
        f"{Q('vel', f'rest_declare_ms/{key}/coast_down/median')} after a "
        "coast-down's stop (p90 "
        f"{Q('vel', f'rest_declare_ms/{key}/coast_down/p90')}), "
        f"{Q('vel', f'rest_declare_ms/{key}/stall_in_hold/median')} in a "
        f"stalled hold; drops {Q('vel', f'rest_drop_ms/{key}/median')} "
        f"after a start (p90 {Q('vel', f'rest_drop_ms/{key}/p90')}), with "
        "the back-EMF's veto "
        f"{Q('vel', f'rest_drop_ms/{key} + emf/median')}")
    REST[form] = (key, thr, ROW)

In [25]:
for form, (key, thr, ROW) in REST.items():
    band = form == "band"               # the two forms keep different state
    ROW.update(
        valid_flag="the verdict is itself a flag",
        valid_share="set on "
        f"{Q('vel', 'blend_share_of_rows/swr5/rest')} of the sitting's rows"
        f" (swr5); slowest motion told from rest at 50 ms "
        f"{Q('vel', FAR + '/far_from_carry')} far from a carry, "
        f"{Q('vel', FAR + '/on_widest_carry')} on the widest",
        start_up="the hold time "
        f"{Q('vel', f'rest_verdict/{form}/hold_ms')} ms; rule: "
        f"{node('vel', 'rest_hold_rule')} <vel:rest_hold_rule> "
        f"({Q('vel', 'rest_longest_missing_run_counts')})",
        state="the boxcar of 16, the anchor and a row count" if band
        else "the slope's rows and a row count",
        constants="sigma16 from a rest at identification, the longest "
        "missing run" if band else "the slope's rest sd, the longest "
        "missing run",
        term_corrected="the stopped rotor, which the back-EMF reads as a "
        "speed under current, and the pot slope's noise at rest (stage 3 "
        "rows 12 and 16 read it)",
        how="the pot inside the band (or its slope under the threshold) "
        "for the hold time; optional veto by the back-EMF off the stopped "
        "rotor's line",
        bound_left=f"veto: {Q('vel', 'rest_veto/margin_rev_s')} off "
        f"{Q('vel', 'rest_veto/line_a_rev_s')} + "
        f"{Q('vel', 'rest_veto/line_b_rev_s_per_a')}, fires on "
        f"{Q('vel', 'rest_veto/fires_on_stopped')} stopped rotors; not "
        f"told from rest under {Q('vel', FAR + '/far_from_carry')}; not "
        "chosen",
        confidence="moderate, one sitting")
    add_row(**ROW)
print(len(ROWS1), "rows so far")

12 rows so far


In [26]:
LOAD = "load_anchored_mean_pct"         # the winding file's bias under load
BINS = ("[0.25, 0.35)", "[0.35, 0.45)", "[0.45, 0.6)")   # its current bins
ROW = dict(
    estimator="winding temperature and R(T): the board NTC plus one heat "
    "node driven by the copper loss",
    bias="under load, anchored to the reads under 0.25 A: one node "
    + ", ".join(Q("wt", f"{LOAD}/{b}/b_one") for b in BINS)
    + " of R; the NTC alone "
    + ", ".join(Q("wt", f"{LOAD}/{b}/a_ntc") for b in BINS)
    + "; as speed for the NTC alone "
    + ", ".join(Q("wt", f"speed_bias_ntc_alone_rev_s/{b}") for b in BINS),
    noise_at_delay="held out, all kinds: one node "
    f"{Q('wt', 'heldout_rms_pct_all/b_one')}, NTC alone "
    f"{Q('wt', 'heldout_rms_pct_all/a_ntc')}, two nodes "
    f"{Q('wt', 'heldout_rms_pct_all/c_two')}; at one seat "
    f"{Q('wt', 'stage56_table/5/value')}; across seats "
    f"{Q('wt', 'stage56_table/6/value')}; floors "
    f"{Q('wt', 'floor_same_seat_pct')} same seat, "
    f"{Q('wt', 'floor_across_seats_pct')} across seats",
    delay=f"node time constant {Q('wt', 'one_node_tau_s')}; gain "
    f"{Q('wt', 'one_node_gain_c_per_w')}; heat capacity "
    f"{Q('wt', 'one_node_c_j_per_c')}",
    valid_flag="no flag",
    valid_share=f"graded to {Q('wt', 'graded_drive_s_max')} of drive from "
    f"a rest; plateau graded {Q('wt', 'plateau_graded')}, can node graded "
    f"{Q('wt', 'can_node_graded')}, ambient graded "
    f"{Q('wt', 'ambient_graded')}; longest drive "
    f"{Q('wt', 'longest_drive_s')}",
    start_up=f"state {node('wt', 'stage56_table/1/note')} "
    "<wt:stage56_table/1/note>; start-up error not measured; the NTC "
    f"itself climbs {Q('wt', 'ntc_self_heating_c')} after power-up",
    state=Q("wt", "stage56_table/1/value"))

In [27]:
ROW.update(
    constants=f"{Q('wt', 'stage56_table/2/value')}; "
    f"{Q('wt', 'stage56_table/3/value')} "
    f"({node('wt', 'stage56_table/3/note')})",
    term_corrected="R(T) in the R i term of the back-EMF speed and the "
    "stall line (stage 3 row 7)",
    how=Q("wt", "stage56_table/0/value"),
    bound_left=f"across seats {Q('wt', 'floor_across_seats_c')}; second "
    f"node as a ceiling {Q('wt', 'stage56_table/9/value')}; R after 3 s at "
    f"0.5 A {Q('wt', 'hold_pct_at_0p5a_3s_one_node')}; bridged seats read "
    f"{Q('wt', 'bridged_seat_ratio')} ({Q('wt', 'bridged_seat_holds')}); "
    "no graded basis past the graded drive",
    confidence=f"high to {Q('wt', 'graded_drive_s_max')} of drive at one "
    "seat; low past it")
add_row(**ROW)
CLOSING = pd.DataFrame(ROWS1)
print(len(CLOSING), "rows,", CLOSING["sources"].map(len).sum(),
      "citations; cells reading 'not measured':",
      int((CLOSING[FIELDS] == "not measured").sum().sum()))

13 rows, 362 citations; cells reading 'not measured': 3


### The rows

Each estimator's row as one table, the fields down the side.

In [28]:
NAMES = {"noise_at_delay": "noise at a stated delay", "valid_flag":
         "valid flag", "valid_share": "share of rows", "start_up":
         "start-up", "state": "state kept", "constants": "constants read",
         "term_corrected": "term it corrects", "bound_left":
         "bound it leaves"}


def show_row(i, r):
    """One estimator's row, the fields down the side."""
    body = [{"field": NAMES.get(f, f), "value": r[f]} for f in FIELDS[1:]]
    display(Markdown(f"**{i + 1}. {cell_md(r['estimator'])}**"))
    show(body, ["field", "value"])


for i, r in enumerate(ROWS1[:3]):       # the zero, the position, the speed
    show_row(i, r)

**1. current zero (awake zero): stage 1's two-zero tracker with the start rule and the drive-feed constant**

| field | value |
|---|---|
| bias | warm by rail band: 3.6 V -0.01, 4.9 V -0.01, 6.0-6.8 V +0.01, 7.0-7.6 V +0.20, >7.8 V +0.78 `cz:stage5_table/0/value` (the design; stage 1's tracker +0.54 to +0.76); stage 1's tracker alone warm mean 0.584 counts `cz:warm_error_mean_counts`; as speed: design 0.0486 [0.032, 0.0652] rev/s, +/- se `spd:tracked_zero_cost_per_sweep/design/warm/mean` over 134 sweeps `spd:tracked_zero_cost_per_sweep/design/warm/sweeps`, stage 1's tracker 0.302 [0.283, 0.321] rev/s, +/- se `spd:tracked_zero_cost_per_sweep/tracked/warm/mean` |
| noise at a stated delay | warm scatter 0.411 [0.373, 0.447] counts `cz:design_warm_rms_counts` rms at the truth's floor 0.396 counts `cz:truth_floor_counts`; the zero's own noise 0.00705 counts `cz:alpha_scan/shift_13/zero_noise_white_counts` white, 0.0903 counts `cz:alpha_scan/shift_13/zero_move_2s_coast_counts` over a 2 s coast; the sample per tick 1.061 [1.017, 1.114] counts `cz:trough_sd_per_tick_counts/asleep` asleep, 1.109 [1.038, 1.463] counts `cz:trough_sd_per_tick_counts/coast` coast, 1.161 [1.044, 1.436] counts `cz:trough_sd_per_tick_counts/narrow_drive` narrow drive |
| delay | time constant 0.41 s `cz:tau_s` of feed; flat band of the shift [12, 14] `cz:alpha_flat_band_shift` |
| valid flag | fed or seeded since boot or a rail move, and inside the hold bound |
| share of rows | fed on 0.911 share `cz:feed_share_awake_rows` of awake rows and 0.853 share `cz:feed_share_drive_rows` of drive rows; longest unfed awake stretch 2.400 s `cz:longest_unfed_awake_s`; hold p90 0.589 [0.551, 0.616] counts `cz:hold_p90_counts/1s` at 1 s, 0.812 [0.737, 0.965] counts `cz:hold_p90_counts/10s` at 10 s, 0.946 [0.877, 0.992] counts `cz:hold_p90_counts/60s` at 60 s |
| start-up | to half a count in 0.0625 [-, 0.303] s `cz:start_rule_settle_to_half_count_fed_s` of feed (median, hi the worst) against 0.781 [0.389, 1.768] s `cz:settle_to_half_count_fed_s` for stage 1's tracker alone; start events 0.361 [0.241, 0.475] counts `cz:design_events_rms_counts` rms against 2.070 counts `cz:stage1_events_rms_counts`; boot error without the rule 6.363 counts `cz:start_error_counts/boot` |
| state kept | two zeros as 64-bit floats in counts, their fresh counters, the rail the awake zero was learned at, the last rail sample, the running mean of the drive-feed offset |
| constants read | alpha 2^-13 `cz:alpha_scan/shift_13` (tau 0.41 s `cz:tau_s`), feed gate 30 % `cz:feed_gate_pct` (clean to 80 % `cz:feed_gate_clean_to_pct`), step line 1.667 [1.630, 1.704] counts/V `cz:awake_step_slope_counts_per_v` and -2.717 [-2.916, -2.518] counts `cz:awake_step_intercept_counts` or the curve of .step.csv, seed threshold 0.319 [0.2, 1] V `cz:seed_rail_threshold_v`, debounce 32 ticks `cz:seed_debounce_ticks`, drive-feed offset 0.667 [0.657, 0.698] counts `cz:drive_feed_offset_counts` |
| term it corrects | the current channel's zero while the bridge is awake and its step with the rail (stage 3 row 8) |
| how | learn the awake zero from trough rows of drives at or under the gate (the drive-feed offset taken off), settled coasts in the crest column and settled torque off; seed awake = asleep + step(rail) at boot and on a rail move past the threshold; subtract with its fraction |
| bound it leaves | all runs 0.408 counts `cz:design_all_runs_rms_counts` rms, worst 1.967 counts `cz:design_all_runs_worst_counts` (stage 1's tracker alone 0.883 counts `cz:stage1_all_runs_rms_counts`, worst 7.204 counts `cz:stage1_all_runs_worst_counts`); stall holds on the proxy truth 1.536 counts `cz:stall_hold_error_counts_rms`; a count is 0.471 [0.471, 0.517] rev/s per count of zero `spd:rev_s_per_count` |
| confidence | high on this servo; moderate for the step curve: the step's shape off 4.9 V rests on one day `cz:gaps/1/gap` |

**2. position: lookup indexed by counts, then a boxcar of 16 rows**

| field | value |
|---|---|
| bias | table term held out 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms` (3.399 [3.377, 3.424] counts `pos:table_term_heldout_counts_rms`); nothing out, stamp lead applied 0.41 [0.376, 0.446] deg `pos:heldout_lead_applied_deg_rms`, no lead 0.568 deg `pos:heldout_no_lead_deg_rms`; end lines 0.329 [0.329, 0.372] deg `pos:end_residual_low_deg_rms` low, 0.395 [0.395, 0.435] deg `pos:end_residual_high_deg_rms` high; by direction 0.274 [0.249, 0.3] deg `pos:direction_offset_crawl_plus_deg` and -0.227 [-0.257, -0.196] deg `pos:direction_offset_crawl_minus_deg` on steady crawl holds, not carried |
| noise at a stated delay | per row 1.031 [1.027, 1.160] counts `pos:pot_noise_row_counts`; boxcar 16 0.139 [0.139, 0.161] counts `pos:filter_boxcar16_noise_counts` (0.00733 deg `pos:filter_boxcar16_noise_deg`) at 7.500 ticks `pos:filter_boxcar16_delay_ticks`; the other filters in the position file's .filters.csv |
| delay | lookup none; boxcar 16 7.500 ticks `pos:filter_boxcar16_delay_ticks`, 0.0381 deg `pos:filter_boxcar16_lag_pass_deg` behind at the passes' speed |
| valid flag | no flag |
| share of rows | valid range lookup 384 to 3488 counts, fitted 576 to 3200, the crawl's coverage 790 to 2952 `pos:stage5_table/9/value` (past the start points no data); share of rows: not measured |
| start-up | the lookup none; the boxcar reads the mean of the rows it has until it holds 16, 0.80 ms `pos:stage5_table/10/value` |
| state kept | the lookup none; the boxcar its last 16 rows |
| constants read | 98 entries `pos:lookup_entries_guard_to_guard` at pitch 32 counts `pos:lookup_pitch_counts` from 384 counts `pos:lookup_knot_first_counts` to 3488 counts `pos:lookup_knot_last_counts` (the position file's .lookup.csv) |
| term it corrects | the pot's scale, smooth bend and fine texture (stage 3 rows 0 and 1); the board's tick pattern, period 16 ticks `pos:tick_pattern_period_ticks` |
| how | linear interpolation between knots uniform in counts; the boxcar of 16 nulls the pattern (3.014 [2.986, 3.025] counts `pos:tick_pattern_pp_counts` pp, 0.678 [0.52, 0.678] share `pos:tick_pattern_variance_share` of the variance) |
| bound it leaves | direction 0.5 deg `pos:direction_bound_moving_deg` in motion, 0.3 deg `pos:direction_bound_rest_deg` at rest; carries blind over [0.772, 0.139, 0.467, 0.0298, 0.279] `pos:carry_blind_span_deg` deg at codes [1023, 1279, 2047, 2303, 2559] `pos:carry_codes`; high end hole 7.966 deg `pos:high_end_hole_deg`; the stamp lead's share of the error 0.479 share `pos:share_stamp_lead` |
| confidence | high for the lookup's pitch and the direction bound, moderate for the bound's size; one sitting |

**3. back-EMF speed per row (stage 3's four-term form, tracked zero, k_d = 0), brush term as the signed table or as the product**

| field | value |
|---|---|
| bias | ladder (third repeat) 30-100 (94 rev/s, 73 mA): table -0.31, product 0.12 rev/s `spd:stage5_bias_by_band_form/0`; ladder (third repeat) 250-350 (303.3 rev/s, 127 mA): table 0.15, product -0.17 rev/s `spd:stage5_bias_by_band_form/3`; holds (free, low) 0-30 (23.40 rev/s, 39 mA): table -0.88, product 0.89 rev/s `spd:stage5_bias_by_band_form/6`; run-up 160-250 (218.9 rev/s, 655 mA): table 30, product -1.340 rev/s `spd:stage5_bias_by_band_form/11`; run-up 350-450 (408.7 rev/s, 349 mA): table 12.45, product 2.310 rev/s `spd:stage5_bias_by_band_form/13`; lifts 0-30 (11.50 rev/s, 195 mA): table 3.460, product 4.320 rev/s `spd:stage5_bias_by_band_form/15`; lifts 100-160 (111.5 rev/s, 218 mA): table 7.420, product 3.580 rev/s `spd:stage5_bias_by_band_form/17`; load holds 0-30 (0 rev/s, 198 mA): table 2.810, product 4.750 rev/s `spd:stage5_bias_by_band_form/18`; flat band over the ladder's rungs: table 0.252 [-1.414, 1.918] rev/s, range over rungs `spd:brush_flat_band_rev_s/table`, product -0.044 [-1.904, 1.816] rev/s, range over rungs `spd:brush_flat_band_rev_s/product` |
| noise at a stated delay | per row (delay 0), table form: rest 0.586 `spd:.noise.csv[kind=rest,band=rest].table_sd`, crawl holds 3.841 `spd:.noise.csv[kind=hold,band=0-30].table_sd` and 4.861 `spd:.noise.csv[kind=hold,band=30-100].table_sd`, ladder 9.471 `spd:.noise.csv[kind=ladder,band=100-160].table_sd` to 10.83 `spd:.noise.csv[kind=ladder,band=250-350].table_sd` rev/s; flat band 10.07 [9.146, 11.14] rev/s per row, 100 to 350 rev/s, range of 116 windows `spd:row_noise_flat_band`; boxcar 32: 3.690 `spd:.filters.csv[kind=hold,band=0-30,filter=box 32].sd_table` at a crawl, 0.928 `spd:.filters.csv[kind=ladder,band=250-350,filter=box 32].sd_table` at 250-350 rev/s |
| delay | none per row but L / R under acceleration; filters add (run-ups, rows): box 16 7.988 `spd:.filters.csv[kind=ladder,band=30-100,filter=box 16].runup_delay_added`, box 32 16.45 `spd:.filters.csv[kind=ladder,band=30-100,filter=box 32].runup_delay_added`, pole 32 32.82 `spd:.filters.csv[kind=ladder,band=30-100,filter=pole 32].runup_delay_added`, lock 1 6.381 `spd:.filters.csv[kind=ladder,band=30-100,filter=lock 1].runup_delay_added`; measured less nominal 0 [-3.318, 1.043] rows, measured added less nominal, range `spd:filter_delay_flat_band_rows/box` |
| valid flag | on-time at least 84 [72, 96] ticks; bounds: the failing rung below, the first rung with 10 runs or more `spd:valid_rule/tap_floor_ticks`, 16 rows, 4 L / R rounded up `spd:valid_rule/start_rows` since torque on or a flip, not a dropout row (0.25 of the one-pole mean `spd:valid_rule/dropout_frac` over 32 rows, the one-pole `spd:valid_rule/dropout_reference_rows`) |
| share of rows | valid on 85.21 `spd:valid_coverage_by_set/all/valid_pct` % of 10508431 `spd:valid_coverage_by_set/all/driven_rows` driven rows of 518 `spd:valid_coverage_by_set/all/sweeps` sweeps; under the floor 14.62 `spd:valid_coverage_by_set/all/under_floor_pct` %, start-up 0.144 `spd:valid_coverage_by_set/all/start_up_pct` %, dropout 0.0249 `spd:valid_coverage_by_set/all/dropout_pct` % |
| start-up | 16 rows, 4 L / R rounded up `spd:valid_rule/start_rows`; the error settles by row 14 rows `spd:startup_profile/settled_rows/none`, decay 3.439 rows, median, rows 2-12 `spd:startup_profile/decay_rows`; row 0 reads 142.9 `spd:startup_profile/median/row 0` rev/s at the median |
| state kept | the tracked zero's state, the previous row's speed and direction (the table's newton steps), the filter's state, the valid rule's two row counters, its one-pole mean and flag count |
| constants read | counts_per_a 1135.0 counts/A `spd:constants_used/counts_per_a`, r_ohm 4.177 ohm `spd:constants_used/r_ohm`, t0_c 26.26 C `spd:constants_used/t0_c`, v0_v 0.0332 V `spd:constants_used/v0_v`, g_s 0.00475 S `spd:constants_used/g_s`, brake_b0_v 0.00875 V `spd:constants_used/brake_b0_v`, brake_b1_ohm 0.24 ohm `spd:constants_used/brake_b1_ohm`, l_h 0.00082 H `spd:constants_used/l_h`, ke_v_per_rev_s 0.00728 V per rev/s `spd:constants_used/ke_v_per_rev_s`; the table `spd:brush_forms/table_signed_mv` or the product c0 -40.43 [-43.26, -37.60] mV, +/- se `spd:brush_forms/product_ladder_alone/c0_mv`, c_dir 12.39 [9.149, 15.63] mV, +/- se `spd:brush_forms/product_ladder_alone/c_dir_mv`, b 2.058 [2.010, 2.107] mV per (rev/s x A), +/- se `spd:brush_forms/product_ladder_alone/b_mv_per_rev_s_a` |
| term it corrects | the reading term, R(T) i, V0, the brush drop with its direction term and the low-speed offset (stage 3 rows 5, 6, 7, 10), on the tracked zero (row 8) |
| how | construction C per row; the product in closed form, the table by 2 `spd:solve_recommendation/steps` newton steps from the row before (worst solve error 0.0125 share of the row noise `spd:solve_recommendation/worst_rms_over_noise`) |
| bound it leaves | stopped rotor under current: table -4.663 [-5.633, -3.694] rev/s, +/- se `spd:rest_bias_by_current/table/a` + 43.03 [38.90, 47.16] rev/s per A, +/- se `spd:rest_bias_by_current/table/b_per_a`, mean 4.426 [3.747, 5.105] rev/s, +/- se `spd:rest_bias_by_current/mean/table`; duty step direct path 0.751 of the volts' step / Ke `spd:duty_step_response/share_rows_0_3`, 0.299 of the volts' step / Ke `spd:duty_step_response/share_rows_4_9`, 0.114 of the volts' step / Ke `spd:duty_step_response/share_rows_10_39`; under the floor not valid; top rungs not resolved (ladder (third repeat) 450-600 (456.9 rev/s, 272 mA): table 5.960, product -1.710 rev/s `spd:stage5_bias_by_band_form/5`) |
| confidence | high on the free shaft below the 450-600 rev/s band; moderate on the lifts (pot truth); moderate for the floor at 84 ticks, high at 96 `spd:valid_rule/tap_floor_ticks` |

In [29]:
for i, r in enumerate(ROWS1[3:5], 3):   # the velocity's two sources
    show_row(i, r)

**4. velocity as the pot's slope (two-point over a multiple of 16 rows, or two boxcars of 16)**

| field | value |
|---|---|
| bias | pot's two-point, 200 rows in 'crawl over the floor': noise 7.434 rev/s, bias 0 rev/s, delay 5 ms `vel:stage5_by_regime/2`; pot's two-point, 2000 rows in 'crawl under the floor': noise 1.470 rev/s, bias 0 rev/s, delay 50 ms `vel:stage5_by_regime/3`; pot5 at a crawl 0.000138 [-0.0445, 0.0447] rev/s, +/- se `vel:blend_bias_crawl_rev_s/pot5`, pot25 -0.0309 [-0.0714, 0.00966] rev/s, +/- se `vel:blend_bias_crawl_rev_s/pot25` |
| noise at a stated delay | own noise at a crawl: 5 ms 7.434 `vel:stage5_by_regime/2/noise_rev_s` rev/s, 50 ms under the floor 1.470 `vel:stage5_by_regime/3/noise_rev_s` rev/s; own sd pot5 6.988 rev/s `vel:blend_own_sd_rev_s/crawl/pot5`, pot25 2.594 rev/s `vel:blend_own_sd_rev_s/crawl/pot25`; at rest: two-point 6.112 [0, 9.705] deg/s rms at 100 rows of delay, bounds the windows' sd `vel:pot_slope_rest_noise_5ms_deg_s/two_point`, two boxcars 1.437 [2.4e-10, 2.301] deg/s rms at 100 rows of delay, bounds the windows' sd `vel:pot_slope_rest_noise_5ms_deg_s/box16`; angle error per window 0.12 [0.103, 0.129] deg, median, bounds the windows 25.6 to 400 ms `vel:pot_slope_angle_error_per_window_deg/crawl 15-35` |
| delay | half the window; measured on kick ends pot5 105.2 [104.4, 105.9] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick end, hold over the floor/pot5`, pot25 498.8 [495.0, 502.5] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick end, hold over the floor/pot25`; residual against nominal 1.734 [0.799, 1.734] rows, kick ends `vel:pot_slope_delay_resid_rows/up_to_128` |
| valid flag | no flag; the carry band's rows are skipped |
| share of rows | carry band +/-12 counts, the longest missing run `vel:carry_mask_half_counts` skipped under 50 ms, about twice the longest dwell at a crawl `vel:carry_mask_until_window_ms`; a window holds a band row on 0.0363 of moving rows `vel:carry_rows_touched_share_crawl/6.4 ms` to 0.332 of moving rows `vel:carry_rows_touched_share_crawl/200 ms`; the band itself 0.0695 of moving rows, +/-12 counts `vel:carry_band_rows_share_crawl` |
| start-up | not measured |
| state kept | the last n + 16 rows, or two running sums and the rows between, and the row marks of the carry band |
| constants read | the position lookup, the carries' codes [1023, 1279, 2047, 2303, 2559] `pos:carry_codes` and missing runs [12, 3, 11, 3, 7] `pos:carry_missing_codes`, the window as a multiple of 16 rows, the board's pattern `vel:pot_slope_two_point_multiple_of_rows` (off a multiple 1.803 [1.214, 2.392] x the 1 / n line, bounds over the four lengths `vel:pot_slope_off_multiple_penalty`) |
| term it corrects | the pot's texture and the converter's carries in a velocity (stage 3 rows 1 and 2); the only source under the tap floor |
| how | difference of the lookup's angle over the window, the carry band's rows left out under 50 ms, about twice the longest dwell at a crawl `vel:carry_mask_until_window_ms` of window, never held |
| bound it leaves | near a carry, no mask: 10.78 deg/s rms `vel:carry_near_rms_deg_s/none/near/25.6 ms` against far 5.852 deg/s rms `vel:carry_near_rms_deg_s/none/far/25.6 ms`; skip12 6.690 deg/s rms `vel:carry_near_rms_deg_s/skip12/near/25.6 ms`; hold12 31.41 deg/s rms `vel:carry_near_rms_deg_s/hold12/near/25.6 ms`; over a kick the commutation count runs 1.024 [1.018, 1.030] ratio, +/- 2 se, n 38 `vel:count_kick_start_145ms/ratio_count_pot` of the pot's angle |
| confidence | moderate, one sitting, over the carries' crossings at a crawl |

**5. velocity as the back-EMF speed through a boxcar (over the tap floor)**

| field | value |
|---|---|
| bias | crawl against the tag: table 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table`, product 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product`; against the tach: table 0.661 [0.571, 0.752] rev/s, +/- se over segments `vel:emf_bias_crawl_tach_rev_s/table`, product 1.504 [1.345, 1.662] rev/s, +/- se over segments `vel:emf_bias_crawl_tach_rev_s/product`; by direction table 0.891 [0.773, 1.009] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/table/+` and 0.109 [0.00347, 0.214] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/table/-`, product 0.219 [0.0885, 0.349] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/product/+` and 2.098 [1.984, 2.212] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/product/-`; pass: table -0.318 [-0.689, 0.0524] rev/s, +/- se over segments `vel:emf_bias_pass_rev_s/table` (not a finding); under the floor table -2.251 [-2.649, -1.854] rev/s, +/- se over segments `vel:emf_bias_under_floor_rev_s/table`, product -1.275 [-1.644, -0.906] rev/s, +/- se over segments `vel:emf_bias_under_floor_rev_s/product` |
| noise at a stated delay | back-EMF, table form, boxcar 200 rows in 'crawl over the floor': noise 1.718 rev/s, bias 0.54 rev/s, delay 5 ms `vel:stage5_by_regime/0`; per row at a crawl 3.896 rev/s per row, table `vel:emf_noise_row_rev_s/crawl valid`; boxcar 32 3.426 [2.266, 5.916] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_tab boxcar 32`; boxcar 200 1.914 [0.412, 5.021] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_tab boxcar 200`; commutation-locked 0.719 [0.701, 0.738] of the per-row variance, +/- se over segments `vel:emf_commutation_locked_share_crawl`, left by a boxcar of 32 2.882 rev/s `vel:emf_commutation_locked_sd_rev_s/boxcar_32`, of 100 1.057 rev/s `vel:emf_commutation_locked_sd_rev_s/boxcar_100` |
| delay | half the boxcar; direct path from a duty step through a boxcar of 32: 0.0591 [0.0438, 0.0745] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/boxcar_32/r0_3`, 0.108 [0.088, 0.128] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/boxcar_32/r4_9`, 0.117 [0.103, 0.131] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/boxcar_32/r10_39` |
| valid flag | the speed notebook's rule |
| share of rows | valid on 0.843 of driven rows `vel:emf_valid_share_camera`; 0.262 of crawl hold rows `vel:emf_crawl_rows_under_floor_share` are under the floor |
| start-up | the speed rule's 16 rows, 4 L / R rounded up `spd:valid_rule/start_rows` plus the boxcar |
| state kept | the speed estimator's state and the boxcar's rows |
| constants read | the speed estimator's |
| term it corrects | none of its own; it carries the back-EMF speed's bias into the velocity |
| how | boxcar of the valid rows |
| bound it leaves | after a hard deceleration back-EMF less pot -22.88 [-32.56, -13.21] rev/s, back-EMF less pot, +/- 2 se; cause open `vel:kick_end_emf_minus_pot_rev_s/002-005/emf_minus_pot`, -12.33 [-16.45, -8.209] rev/s, back-EMF less pot, +/- 2 se; cause open `vel:kick_end_emf_minus_pot_rev_s/010-020/emf_minus_pot`, -2.938 [-6.221, 0.345] rev/s, back-EMF less pot, +/- 2 se; cause open `vel:kick_end_emf_minus_pot_rev_s/020-030/emf_minus_pot`; stopped rotor -4.543 [-5.727, -3.358] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/line_a` + 42.67 [37.39, 47.95] rev/s per A, +/- se `vel:emf_stopped_rotor_rev_s/table/line_b_per_a`; direction split under 45 rev/s table 0.814 [0.746, 0.883] rev/s, +/- se `vel:emf_direction_split_rev_s/table/under_45`, product -1.828 [-1.896, -1.759] rev/s, +/- se `vel:emf_direction_split_rev_s/product/under_45` |
| confidence | high for the crawl bias's sign, moderate for its size; one sitting |

In [30]:
for i, r in enumerate(ROWS1[5:], 5):    # the blends, the verdict, R(T)
    show_row(i, r)

**6. velocity blend sw5: the switch (back-EMF where valid, else the pot's 5 ms slope)**

| field | value |
|---|---|
| bias | crawl 0.404 [0.278, 0.53] rev/s, +/- se `vel:blend_bias_crawl_rev_s/sw5`; against the tach 0.574 [0.468, 0.68] rev/s, +/- se `vel:blend_bias_tach_crawl_rev_s/sw5`; stall over the floor 1.226 rev/s `vel:blend_bias_stall_over_floor_rev_s/sw5` |
| noise at a stated delay | own sd crawl 3.426 rev/s `vel:blend_own_sd_rev_s/crawl/sw5`, pass 2.193 rev/s `vel:blend_own_sd_rev_s/pass/sw5`, under the floor 4.130 rev/s `vel:blend_own_sd_rev_s/under_floor/sw5` (sw5 in 'crawl, one velocity': noise 3.430 rev/s, bias 0.4 rev/s, delay -84 rows at a kick end `vel:stage5_by_regime/8`) |
| delay | against the pot's centered slope: kick -16.56 [-27.29, -5.825] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick/sw5`, kick end -83.46 [-90.87, -76.05] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick end, hold over the floor/sw5`, reversal 3.164 [-8.182, 14.51] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/reversal/sw5` |
| valid flag | none of its own; the back-EMF's valid rule sets the shares |
| share of rows | pot 0.381 share of rows `vel:blend_share_of_rows/sw5/pot`, back-EMF 0.619 share of rows `vel:blend_share_of_rows/sw5/back-EMF`, pair 0 share of rows `vel:blend_share_of_rows/sw5/pair` |
| start-up | seam sw5_kick: median 13.47 rev/s `vel:blend_seam_rev_s/sw5_kick/median`, p90 18.00 rev/s `vel:blend_seam_rev_s/sw5_kick/p90`, at 52 rows after the transition `vel:blend_seam_rev_s/sw5_kick/row`, n 38 `vel:blend_seam_rev_s/sw5_kick/n`; seam sw5_coast: median 6.815 rev/s `vel:blend_seam_rev_s/sw5_coast/median`, p90 10.14 rev/s `vel:blend_seam_rev_s/sw5_coast/p90`, at 3 rows after the transition `vel:blend_seam_rev_s/sw5_coast/row`, n 28 `vel:blend_seam_rev_s/sw5_coast/n` |
| state kept | nothing of its own |
| constants read | the two sources' and the valid rule's |
| term it corrects | the back-EMF's invalid rows under the floor |
| how | pick by the valid flag |
| bound it leaves | direct path 0.117 [0.103, 0.131] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/boxcar_32/r10_39`; not chosen |
| confidence | moderate, one sitting |

**7. velocity blend cp5: the complementary pair, pot slope of 5 ms**

| field | value |
|---|---|
| bias | crawl 0.0442 [9.35e-05, 0.0883] rev/s, +/- se `vel:blend_bias_crawl_rev_s/cp5`; against the tach 0.215 [0.114, 0.315] rev/s, +/- se `vel:blend_bias_tach_crawl_rev_s/cp5`; stall over the floor 0.0156 rev/s `vel:blend_bias_stall_over_floor_rev_s/cp5` |
| noise at a stated delay | own sd crawl 3.517 rev/s `vel:blend_own_sd_rev_s/crawl/cp5`, pass 2.470 rev/s `vel:blend_own_sd_rev_s/pass/cp5`, under the floor 4.130 rev/s `vel:blend_own_sd_rev_s/under_floor/cp5` (cp5 in 'crawl, one velocity': noise 3.520 rev/s, bias 0.04 rev/s, delay -69 rows at a kick end `vel:stage5_by_regime/9`) |
| delay | against the pot's centered slope: kick -14.88 [-24.20, -5.565] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick/cp5`, kick end -68.89 [-75.39, -62.39] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick end, hold over the floor/cp5`, reversal 4.443 [-2.742, 11.63] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/reversal/cp5` |
| valid flag | none of its own; the back-EMF's valid rule sets the shares |
| share of rows | pot 0.381 share of rows `vel:blend_share_of_rows/cp5/pot`, back-EMF 0.00521 share of rows `vel:blend_share_of_rows/cp5/back-EMF`, pair 0.614 share of rows `vel:blend_share_of_rows/cp5/pair` |
| start-up | seam cp5_handover: median 0.205 rev/s `vel:blend_seam_rev_s/cp5_handover/median`, p90 0.464 rev/s `vel:blend_seam_rev_s/cp5_handover/p90`, at 229 rows after the transition `vel:blend_seam_rev_s/cp5_handover/row`, n 38 `vel:blend_seam_rev_s/cp5_handover/n`; seam cp5_coast: median 5.408 rev/s `vel:blend_seam_rev_s/cp5_coast/median`, p90 10.08 rev/s `vel:blend_seam_rev_s/cp5_coast/p90`, at 3 rows after the transition `vel:blend_seam_rev_s/cp5_coast/row`, n 28 `vel:blend_seam_rev_s/cp5_coast/n` |
| state kept | one one-pole (corner 1030.5 rows, the pair's one-pole `vel:blend_corner_rows`) and the back-EMF through the pot's kernel |
| constants read | the two sources' and the fixed corner |
| term it corrects | the back-EMF's DC bias (reads the pot at DC) and its invalid rows |
| how | pot slope below the corner, back-EMF above it, fixed weights |
| bound it leaves | direct path 0.153 [0.125, 0.18] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_5/r10_39`, 0.0496 [0.00671, 0.0925] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_5/r40_199`; not chosen |
| confidence | moderate, one sitting |

**8. velocity blend cp25: the complementary pair, pot slope of 25 ms**

| field | value |
|---|---|
| bias | crawl 0.058 [0.00228, 0.114] rev/s, +/- se `vel:blend_bias_crawl_rev_s/cp25`; against the tach 0.244 [0.136, 0.352] rev/s, +/- se `vel:blend_bias_tach_crawl_rev_s/cp25` |
| noise at a stated delay | own sd crawl 3.421 rev/s `vel:blend_own_sd_rev_s/crawl/cp25`, pass 2.313 rev/s `vel:blend_own_sd_rev_s/pass/cp25`, under the floor 1.853 rev/s `vel:blend_own_sd_rev_s/under_floor/cp25` (cp25 in 'crawl, one velocity': noise 3.420 rev/s, bias 0.06 rev/s, delay -96 rows at a kick end `vel:stage5_by_regime/10`) |
| delay | against the pot's centered slope: kick -18.84 [-29.54, -8.144] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick/cp25`, kick end -96.07 [-103.1, -89.00] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/kick end, hold over the floor/cp25`, reversal 3.734 [-6.241, 13.71] rows against the pot's centered slope, +/- se `vel:blend_delay_rows/reversal/cp25` |
| valid flag | none of its own; the back-EMF's valid rule sets the shares |
| share of rows | not measured |
| start-up | seam cp25_coast: median 2.541 rev/s `vel:blend_seam_rev_s/cp25_coast/median`, p90 5.960 rev/s `vel:blend_seam_rev_s/cp25_coast/p90`, at 3 rows after the transition `vel:blend_seam_rev_s/cp25_coast/row`, n 28 `vel:blend_seam_rev_s/cp25_coast/n` |
| state kept | one one-pole (corner 1030.5 rows, the pair's one-pole `vel:blend_corner_rows`) and the back-EMF through the pot's kernel |
| constants read | the two sources' and the fixed corner |
| term it corrects | the back-EMF's DC bias (reads the pot at DC) and its invalid rows |
| how | pot slope below the corner, back-EMF above it, fixed weights |
| bound it leaves | direct path 0.154 [0.122, 0.186] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_25/r10_39`, 0.0463 [0.00284, 0.0897] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_25/r40_199`; not chosen |
| confidence | moderate, one sitting |

... (6 outputs left out)

**12. rest verdict, slope form (equal alternatives)**

| field | value |
|---|---|
| bias | false rest at 10 to 35 deg/s: 0 share of rows `vel:rest_false_rest_10_35_deg_s/slope 192 50` at 50 ms, 0.000641 share of rows `vel:rest_false_rest_10_35_deg_s/slope 192 25` at 25 ms (near a carry 0.0087 share of rows `vel:rest_false_rest_10_35_near_carry/slope 192 25`) over 480310 rows `vel:rest_false_rest_rows_10_35`; false motion: baselines 0 share of rows `vel:rest_false_motion_share/slope 192 T50/rest_base`, coast tails 0.00777 share of rows `vel:rest_false_motion_share/slope 192 T50/rest_coast`, stalls 0 share of rows `vel:rest_false_motion_share/slope 192 T50/stall72`, load holds 0.0011 share of rows `vel:rest_false_motion_stage3_share/slope 192 T50/load holds` |
| noise at a stated delay | threshold the 192 `vel:rest_verdict/slope/rows`-row two-boxcar slope under 5 `vel:rest_verdict/slope/threshold_x_own_sd` x its rest sd = 5.090 deg/s `vel:rest_verdict/slope/threshold_deg_s`; sigma16 0.137 [0, 0.165] counts, median [range] over the baselines `vel:rest_sigma16_counts` |
| delay | declares 56.10 ms, median `vel:rest_declare_ms/slope 192 T50/coast_down/median` after a coast-down's stop (p90 57.83 ms, p90 `vel:rest_declare_ms/slope 192 T50/coast_down/p90`), 58.73 ms, median `vel:rest_declare_ms/slope 192 T50/stall_in_hold/median` in a stalled hold; drops 6.500 ms, median `vel:rest_drop_ms/slope 192 T50/median` after a start (p90 9.595 ms, p90 `vel:rest_drop_ms/slope 192 T50/p90`), with the back-EMF's veto 5.750 ms, median `vel:rest_drop_ms/slope 192 T50 + emf/median` |
| valid flag | the verdict is itself a flag |
| share of rows | set on 0.327 share of rows `vel:blend_share_of_rows/swr5/rest` of the sitting's rows (swr5); slowest motion told from rest at 50 ms 2.018 deg/s `vel:rest_min_speed_deg_s/50 ms/far_from_carry` far from a carry, 14.12 deg/s `vel:rest_min_speed_deg_s/50 ms/on_widest_carry` on the widest |
| start-up | the hold time 50 `vel:rest_verdict/slope/hold_ms` ms; rule: T at least the longest carry dwell at the slowest speed to tell from rest: the longest missing run over the scale and that speed `vel:rest_hold_rule` (12 counts `vel:rest_longest_missing_run_counts`) |
| state kept | the slope's rows and a row count |
| constants read | the slope's rest sd, the longest missing run |
| term it corrects | the stopped rotor, which the back-EMF reads as a speed under current, and the pot slope's noise at rest (stage 3 rows 12 and 16 read it) |
| how | the pot inside the band (or its slope under the threshold) for the hold time; optional veto by the back-EMF off the stopped rotor's line |
| bound it leaves | veto: 10.88 rev/s `vel:rest_veto/margin_rev_s` off -4.543 [-5.727, -3.358] rev/s, +/- se `vel:rest_veto/line_a_rev_s` + 42.67 [37.39, 47.95] rev/s per A `vel:rest_veto/line_b_rev_s_per_a`, fires on 1 of 106 `vel:rest_veto/fires_on_stopped` stopped rotors; not told from rest under 2.018 deg/s `vel:rest_min_speed_deg_s/50 ms/far_from_carry`; not chosen |
| confidence | moderate, one sitting |

**13. winding temperature and R(T): the board NTC plus one heat node driven by the copper loss**

| field | value |
|---|---|
| bias | under load, anchored to the reads under 0.25 A: one node 0.481 [-0.916, 1.877] % `wt:load_anchored_mean_pct/[0.25, 0.35)/b_one`, 0.378 [-0.354, 1.110] % `wt:load_anchored_mean_pct/[0.35, 0.45)/b_one`, -0.0565 [-1.243, 1.130] % `wt:load_anchored_mean_pct/[0.45, 0.6)/b_one` of R; the NTC alone 2.001 [1.187, 2.815] % `wt:load_anchored_mean_pct/[0.25, 0.35)/a_ntc`, 2.946 [1.957, 3.935] % `wt:load_anchored_mean_pct/[0.35, 0.45)/a_ntc`, 3.865 [3.077, 4.652] % `wt:load_anchored_mean_pct/[0.45, 0.6)/a_ntc`; as speed for the NTC alone 3.344 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.25, 0.35)`, 6.668 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.35, 0.45)`, 10.72 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.45, 0.6)` |
| noise at a stated delay | held out, all kinds: one node 1.055 % `wt:heldout_rms_pct_all/b_one`, NTC alone 1.292 % `wt:heldout_rms_pct_all/a_ntc`, two nodes 1.002 % `wt:heldout_rms_pct_all/c_two`; at one seat 0.08 to 0.16 % of R rms (0.2 to 0.4 C), held out `wt:stage56_table/5/value`; across seats 1.25 % rms (3.2 C), the seat floor `wt:stage56_table/6/value`; floors 0.0678 [0.0341, 0.129] % `wt:floor_same_seat_pct` same seat, 0.604 [0.3, 1.087] % `wt:floor_across_seats_pct` across seats |
| delay | node time constant 2.375 [1.203, 3.227] s (fold_lo 1.400, fold_hi 2.699, stat_factor 1.044) `wt:one_node_tau_s`; gain 23.24 [14.27, 23.85] C/W (fold_lo 14.27, fold_hi 23.24, stat_factor 1.012) `wt:one_node_gain_c_per_w`; heat capacity 0.102 [0.0625, 0.135] J/C (fold_lo 0.0981, fold_hi 0.117) `wt:one_node_c_j_per_c` |
| valid flag | no flag |
| share of rows | graded to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` of drive from a rest; plateau graded False `wt:plateau_graded`, can node graded False `wt:can_node_graded`, ambient graded False `wt:ambient_graded`; longest drive 36.29 s `wt:longest_drive_s` |
| start-up | state zero after a power cycle `wt:stage56_table/1/note`; start-up error not measured; the NTC itself climbs 2.177 [0.294, 2.177] C `wt:ntc_self_heating_c` after power-up |
| state kept | one float per node, the rise over the NTC (C) `wt:stage56_table/1/value` |
| constants read | G 23.2 C/W (14.3 to 23.9), tau 2.38 s (1.20 to 3.23), C 0.102 J/C (0.063 to 0.135) `wt:stage56_table/2/value`; R0 4.1767 ohm at T0 26.26 C, alpha 0.39 %/C `wt:stage56_table/3/value` (R0 as a cold value open by 1.2 to 2.2 %) |
| term it corrects | R(T) in the R i term of the back-EMF speed and the stall line (stage 3 row 7) |
| how | T_w = T_ntc + x; x += (G P - x)(1 - exp(-dt / tau)); P = i_w^2 R(T_w); R(T) = R0 (1 + alpha (T - T0)) `wt:stage56_table/0/value` |
| bound it leaves | across seats 1.549 [0.769, 2.786] C `wt:floor_across_seats_c`; second node as a ceiling C2 0.51 J/C, tau2 at least 35 s `wt:stage56_table/9/value`; R after 3 s at 0.5 A 7.121 [5.351, 7.121] % `wt:hold_pct_at_0p5a_3s_one_node`; bridged seats read 0.774 [0.746, 0.803] of R0 `wt:bridged_seat_ratio` (3 holds `wt:bridged_seat_holds`); no graded basis past the graded drive |
| confidence | high to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` of drive at one seat; low past it |

### The blends and the verdict side by side

The same keys as the rows above, one column per quantity so the candidates read across. Each header names its key, with the candidate or the form in its place. A blank is a key the velocity file does not hold, which reads "not measured".

In [31]:
def get(f, key):
    """A node, or None where the file holds no such key."""
    try:
        return node(f, key)
    except (KeyError, IndexError):
        return None


def pm_txt(f, key):
    """'value +/- se' of a quantity, or 'not measured'."""
    v = get(f, key)
    if v is None:
        return "not measured"
    if v.get("lo") is None:
        return num(v["value"])
    return f"{num(v['value'])} +/- {num(se_of(f, key))}"


def v_txt(f, key):
    """The bare value of a quantity, or 'not measured'."""
    v = get(f, key)
    return "not measured" if v is None else num(v["value"])

In [32]:
SEAMS = {"sw5": ("sw5_kick", "sw5_coast"), "cp5": ("cp5_handover",
         "cp5_coast"), "cp25": ("cp25_coast",), "swr5": ("swr5_kick",
         "swv5_kick"), "cpr5": ("cpr5_kick",)}
BL = []
for c, seams in SEAMS.items():
    sh = get("vel", f"blend_share_of_rows/{c}") or {}
    BL.append({
        "blend": c,
        "crawl": pm_txt("vel", f"blend_bias_crawl_rev_s/{c}"),
        "stall": v_txt("vel", f"blend_bias_stall_over_floor_rev_s/{c}"),
        "sd": " / ".join(v_txt("vel", f"blend_own_sd_rev_s/{g}/{c}")
                         for g in ("crawl", "pass", "under_floor")),
        "kick_end": v_txt("vel", "blend_delay_rows/kick end, hold over "
                          f"the floor/{c}"),
        "share": ", ".join(f"{k} {num(v['value'])}" for k, v in sh.items())
        or "not measured",
        "seam": ", ".join(f"{s} {v_txt('vel', f'blend_seam_rev_s/{s}/median')}"
                          f" (p90 {v_txt('vel', f'blend_seam_rev_s/{s}/p90')})"
                          for s in seams)})
show(BL, ["blend", "crawl", "stall", "sd", "kick_end", "share", "seam"], [
    "blend", "crawl bias, rev/s `vel:blend_bias_crawl_rev_s/b`",
    "stall over the floor `vel:blend_bias_stall_over_floor_rev_s/b`",
    "own sd crawl / pass / under the floor `vel:blend_own_sd_rev_s/g/b`",
    "delay at a kick end, rows `vel:blend_delay_rows/kick end, hold over "
    "the floor/b`", "share of rows `vel:blend_share_of_rows/b`",
    "seam median `vel:blend_seam_rev_s/s/median`"])
print("the pair's corner:", Q("vel", "blend_corner_rows"))

| blend | crawl bias, rev/s `vel:blend_bias_crawl_rev_s/b` | stall over the floor `vel:blend_bias_stall_over_floor_rev_s/b` | own sd crawl / pass / under the floor `vel:blend_own_sd_rev_s/g/b` | delay at a kick end, rows `vel:blend_delay_rows/kick end, hold over the floor/b` | share of rows `vel:blend_share_of_rows/b` | seam median `vel:blend_seam_rev_s/s/median` |
|---|---|---|---|---|---|---|
| sw5 | 0.404 +/- 0.126 | 1.226 | 3.426 / 2.193 / 4.130 | -83.46 | pot 0.381, back-EMF 0.619, pair 0, rest 0 | sw5_kick 13.47 (p90 18.00), sw5_coast 6.815 (p90 10.14) |
| cp5 | 0.0442 +/- 0.0441 | 0.0156 | 3.517 / 2.470 / 4.130 | -68.89 | pot 0.381, back-EMF 0.00521, pair 0.614, rest 0 | cp5_handover 0.205 (p90 0.464), cp5_coast 5.408 (p90 10.08) |
| cp25 | 0.058 +/- 0.0557 | not measured | 3.421 / 2.313 / 1.853 | -96.07 | not measured | cp25_coast 2.541 (p90 5.960) |
| swr5 | 0.404 +/- 0.126 | 0 | 3.426 / 2.193 / 4.130 | -83.46 | pot 0.0906, back-EMF 0.583, pair 0, rest 0.327 | swr5_kick 44.69 (p90 61.97), swv5_kick 18.02 (p90 18.34) |
| cpr5 | 0.0442 +/- 0.0441 | 0.0156 | 3.517 / 2.470 / 4.130 | not measured | pot 0.0906, back-EMF 0.00521, pair 0.614, rest 0.29 | cpr5_kick 14.39 (p90 16.88) |

the pair's corner: 1030.5 rows, the pair's one-pole <vel:blend_corner_rows>


In [33]:
def rest_line(what, key_of):
    """One quantity of the rest verdict for both forms."""
    return {"quantity": what, **{f: key_of(k) for f, (k, _, _) in
                                 REST.items()}}


RV = [
    rest_line("threshold `vel:rest_verdict`", lambda k: REST[
        "band" if k.startswith("band") else "slope"][1]),
    rest_line("declare after a coast-down, median (p90), ms "
              "`vel:rest_declare_ms/k/coast_down`", lambda k: (
                  f"{v_txt('vel', f'rest_declare_ms/{k}/coast_down/median')}"
                  f" ({v_txt('vel', f'rest_declare_ms/{k}/coast_down/p90')})")),
    rest_line("declare in a stalled hold, ms "
              "`vel:rest_declare_ms/k/stall_in_hold/median`", lambda k:
              v_txt("vel", f"rest_declare_ms/{k}/stall_in_hold/median")),
    rest_line("drop after a start, median (p90), ms `vel:rest_drop_ms/k`",
              lambda k: f"{v_txt('vel', f'rest_drop_ms/{k}/median')} "
              f"({v_txt('vel', f'rest_drop_ms/{k}/p90')})"),
    rest_line("drop with the back-EMF veto, ms "
              "`vel:rest_drop_ms/k + emf/median`", lambda k:
              v_txt("vel", f"rest_drop_ms/{k} + emf/median")),
    rest_line("false rest at 10 to 35 deg/s, at 50 ms / 25 ms "
              "`vel:rest_false_rest_10_35_deg_s`", lambda k: " / ".join(
                  v_txt("vel", "rest_false_rest_10_35_deg_s/"
                        + k.replace(" T50", t)) for t in (" 50", " 25"))),
    rest_line("false motion, baselines / coast tails / stalls / load holds "
              "`vel:rest_false_motion_share/k`, "
              "`vel:rest_false_motion_stage3_share/k/load holds`",
              lambda k: " / ".join(
                  [v_txt("vel", f"rest_false_motion_share/{k}/{s}")
                   for s in ("rest_base", "rest_coast", "stall72")]
                  + [v_txt("vel", "rest_false_motion_stage3_share/"
                           f"{k}/load holds")]))]
show(RV, ["quantity", "band", "slope"],
     ["quantity", f"band, `{REST['band'][0]}`",
      f"slope, `{REST['slope'][0]}`"])

| quantity | band, `band 2 T50` | slope, `slope 192 T50` |
|---|---|---|
| threshold `vel:rest_verdict` | +/-2 `vel:rest_verdict/band/counts` counts of an anchor = 14.64 sigma16 `vel:rest_verdict/band/sigma16` | the 192 `vel:rest_verdict/slope/rows`-row two-boxcar slope under 5 `vel:rest_verdict/slope/threshold_x_own_sd` x its rest sd = 5.090 deg/s `vel:rest_verdict/slope/threshold_deg_s` |
| declare after a coast-down, median (p90), ms `vel:rest_declare_ms/k/coast_down` | 56.35 (64.53) | 56.10 (57.83) |
| declare in a stalled hold, ms `vel:rest_declare_ms/k/stall_in_hold/median` | 55.33 | 58.73 |
| drop after a start, median (p90), ms `vel:rest_drop_ms/k` | 7.725 (10.14) | 6.500 (9.595) |
| drop with the back-EMF veto, ms `vel:rest_drop_ms/k + emf/median` | 5.750 | 5.750 |
| false rest at 10 to 35 deg/s, at 50 ms / 25 ms `vel:rest_false_rest_10_35_deg_s` | 0 / 0.000641 | 0 / 0.000641 |
| false motion, baselines / coast tails / stalls / load holds `vel:rest_false_motion_share/k`, `vel:rest_false_motion_stage3_share/k/load holds` | 0 / 0.0193 / 0 / 0.0058 | 0 / 0.00777 / 0 / 0.0011 |

The two lists behind several cells, as their files hold them. The speed notebook's bias by band and brush form, rev/s against each set's truth, and the velocity notebook's table by regime.

In [34]:
BF = pd.DataFrame(node("spd", "stage5_bias_by_band_form"))
print("<spd:stage5_bias_by_band_form>")
print(BF.to_string())
VR = pd.DataFrame(node("vel", "stage5_by_regime"))
print("\n<vel:stage5_by_regime>")
print(VR.to_string())

<spd:stage5_bias_by_band_form>
                      set     band  rev_s   mA  table  product
0   ladder (third repeat)   30-100   94.0   73  -0.31     0.12
1   ladder (third repeat)  100-160  137.5   85  -0.03     0.01
2   ladder (third repeat)  160-250  211.8  103   0.06    -0.08
3   ladder (third repeat)  250-350  303.3  127   0.15    -0.17
4   ladder (third repeat)  350-450  413.2  204  -2.81     1.14
5   ladder (third repeat)  450-600  456.9  272   5.96    -1.71
6       holds (free, low)     0-30   23.4   39  -0.88     0.89
7       holds (free, low)   30-100   35.6   42  -0.55     0.73
8       holds (free, low)  160-250  210.4   98   1.12     1.35
9       holds (free, low)  250-350  253.6  105  -0.32    -0.40
10                   refs  100-160  141.4   81   0.54     0.84
11                 run-up  160-250  218.9  655  30.00    -1.34
12                 run-up  250-350  306.2  512  27.10    -1.92
13                 run-up  350-450  408.7  349  12.45     2.31
14                 run-u

## 3. Stage 3's closing table, row by row

Every row stage 3 handed on, in its order, with its size and its instruction as stage 3's file gives them. The disposition is one of compensated (here, by the named estimator), designed around (by a rule), bound (a number carried as a limit) or carried (to a later stage, not stage 4's). `agrees` says whether stage 4's result agrees with what stage 3 said to do, as yes, changed, incomplete, no or carried, with what stage 4 says instead beside it. Stage 3's number and stage 4's sit side by side, each with its key.

In [35]:
S3 = node("s3", "closing_table")
ROWS3 = []                              # stage 3's rows, in its order
S3_FIELDS = ["row", "effect", "stage3_size", "stage3_how", "owner",
             "disposition", "disposition_how", "estimator", "residual",
             "agrees", "agrees_how", "stage3_number", "stage4_number"]


def s3row(i, disposition, disposition_how, estimator, stage3_number,
          stage4_number, residual, agrees, agrees_how):
    """One row of stage 3's table with what stage 4 did with it."""
    assert len(ROWS3) == i, (len(ROWS3), i)      # in stage 3's order
    assert agrees in ("yes", "changed", "incomplete", "no", "carried")
    r = S3[i]
    row = {"row": i, "effect": r["effect"],
           "stage3_size": f"{r['size (mg90-d)']} <s3:closing_table/{i}>",
           "stage3_how": r["how"], "owner": r["owner"],
           "disposition": disposition, "disposition_how": disposition_how,
           "estimator": estimator, "residual": residual, "agrees": agrees,
           "agrees_how": agrees_how, "stage3_number": stage3_number,
           "stage4_number": stage4_number}
    ROWS3.append({**row, "sources": sources(*row.values())})


XO = "crossover_ms"                     # stage 3's crossover by band, form
XW = "crossover_window_ms"              # stage 4's crossover windows
print(len(S3), "rows to place")

17 rows to place


In [36]:
s3row(0, "compensated", "compensated here (position)", "position",
      "the angle form at 1.5 deg knots, held out in stage 4: "
      f"{Q('pos', 'angle_form_1p5_deg_heldout_deg_rms')}",
      f"lookup indexed by counts, pitch {Q('pos', 'lookup_pitch_counts')}, "
      f"{Q('pos', 'lookup_entries_guard_to_guard')}; table term held out "
      f"{Q('pos', 'table_term_heldout_deg_rms')}; a 64-count pitch "
      f"{Q('pos', 'lookup_64_counts_heldout_deg_rms')}",
      f"table term held out {Q('pos', 'table_term_heldout_deg_rms')}",
      "yes", "agrees on a table; the form changed from knots in angle to "
      "knots uniform in counts (no search)")
s3row(1, "compensated (position); designed around (velocity)",
      "position: compensated here (position), residual stated; velocity: "
      "designed around (velocity), the rule changed",
      "position; velocity as the pot's slope",
      f"position in-sample {Q('s3', 'position_resid_rms_deg')}; the "
      "crossover against the signed table "
      f"{Q('s3', XO + '/0-30/w_signed / pot_two')} and "
      f"{Q('s3', XO + '/0-30/w_signed / pot_line')} ms at 0-30 rev/s, "
      f"{Q('s3', XO + '/30-100/w_signed / pot_two')} and "
      f"{Q('s3', XO + '/30-100/w_signed / pot_line')} at 30-100, against "
      f"the product {Q('s3', XO + '/0-30/w_product / pot_two')} and "
      f"{Q('s3', XO + '/0-30/w_product / pot_line')}",
      f"position held out {Q('pos', 'table_term_heldout_deg_rms')} with "
      "segment offsets out, "
      f"{Q('pos', 'heldout_lead_applied_deg_rms')} with nothing out but "
      f"the lead; crossover, causal: crawl rms {Q('vel', XW + '/crawl_rms')}"
      f", line {Q('vel', XW + '/crawl_rms_line')}, product "
      f"{Q('vel', XW + '/crawl_rms_product')}, own noise "
      f"{Q('vel', XW + '/crawl_own_noise')}, steps "
      f"{Q('vel', XW + '/steps_rms')}, pass {Q('vel', XW + '/pass_rms')}; "
      "at 5 ms of delay the pot's own noise "
      f"{Q('vel', 'stage5_by_regime/2/noise_rev_s')} against the "
      f"back-EMF's {Q('vel', 'stage5_by_regime/0/noise_rev_s')} rev/s",
      f"position {Q('pos', 'table_term_heldout_deg_rms')}; velocity: the "
      f"crossover's bracket {Q('vel', XW + '/crawl_rms')}",
      "changed", "position agrees. velocity: the direction agrees, the "
      "window moved (crawl earlier, pass later, brackets wide) and the rule "
      "is restated: the pot's slope where the back-EMF is not valid or "
      f"where {node('vel', 'stage5_by_regime/3/delay')} and more of delay "
      "is acceptable <vel:stage5_by_regime/3>, not stage 3's fixed window")

In [37]:
NV = "carry_near_rms_deg_s"             # velocity's two-point near a carry
SPAN = node("pos", "carry_blind_span_deg")
WIDEST = node("pos", "carry_codes")[SPAN.index(max(SPAN))]  # by blind span
s3row(2, "designed around", "designed around (position: no mask; "
      "velocity: skip, never hold)", "position; velocity as the pot's slope",
      "the size and the mask above",
      f"carries at {Q('pos', 'carry_codes')}, blind over "
      f"{Q('pos', 'carry_blind_span_deg')} deg, missing runs "
      f"{Q('pos', 'carry_missing_codes')}. position: near a carry "
      f"{Q('pos', 'carry_near_rms_deg')}, far {Q('pos', 'carry_far_rms_deg')}"
      f", with a hold of the band {Q('pos', 'carry_hold_near_rms_deg')} and "
      f"{Q('pos', 'carry_hold_added_delay_ms')} of added delay. velocity, "
      "two-point near a carry at 12.8 ms: none "
      f"{Q('vel', NV + '/none/near/12.8 ms')}, hold12 "
      f"{Q('vel', NV + '/hold12/near/12.8 ms')}, skip12 "
      f"{Q('vel', NV + '/skip12/near/12.8 ms')}, far "
      f"{Q('vel', NV + '/none/far/12.8 ms')}; at 51.2 ms none "
      f"{Q('vel', NV + '/none/near/51.2 ms')}, skip12 "
      f"{Q('vel', NV + '/skip12/near/51.2 ms')}; mask rule "
      f"{node('vel', 'carry_mask')} <vel:carry_mask> "
      f"+/-{Q('vel', 'carry_mask_half_counts')} under {CARRY_WIN}",
      f"position blind over {Q('pos', 'carry_blind_span_deg')} deg",
      "incomplete", "the mask holds only as rows skipped in a slope under "
      f"{CARRY_WIN} of window, "
      f"+/-{Q('vel', 'carry_mask_half_counts')}; holding the pot through "
      "the band makes both the position and the velocity worse. The widest "
      f"carry by blind span is {WIDEST} <pos:carry_codes>, not the code "
      "stage 3's size names (a different construction, the position "
      "notebook's open discrepancy)")

In [38]:
s3row(3, "bound", "left as a bound (position), not compensated", "position",
      f"{Q('s3', 'position_resid_by_dir_crawl_deg/+1')} and "
      f"{Q('s3', 'position_resid_by_dir_crawl_deg/-1')} by direction at a "
      "crawl",
      "on steady crawl holds "
      f"{Q('pos', 'direction_offset_crawl_plus_deg')} and "
      f"{Q('pos', 'direction_offset_crawl_minus_deg')}; clip means from "
      f"{Q('pos', 'direction_offset_clip_min_deg')} to "
      f"{Q('pos', 'direction_offset_clip_max_deg')}; at rest "
      f"{Q('pos', 'rest_direction_split_deg')}; position-controlled "
      f"contrast {Q('pos', 'direction_contrast_in_bin_deg')}, speed-free "
      f"half width {Q('pos', 'speed_free_half_width_deg')}, timing slope "
      f"{Q('pos', 'timing_slope_ms')}; a direction source, if ever: "
      f"hysteresis {Q('pos', 'direction_source_pot_hysteresis_counts')}, "
      f"late {Q('pos', 'direction_source_late_ms')}",
      f"bound {Q('pos', 'direction_bound_moving_deg')} in motion, "
      f"{Q('pos', 'direction_bound_rest_deg')} at rest",
      "no", "the fixed direction-signed offset stage 3 proposed does not "
      "hold across the sitting's clips: it changes sign between clips, and "
      "the position carries a bound instead")

In [39]:
PL, PR = ("brush_forms/product_ladder_alone",   # stage 4's two fits of the
          "brush_forms/product_ladder_runups")  # product form, k_d = 0
s3row(4, "bound", "left as a bound, stage 7's (no stage 4 result)", "none",
      f"{Q('s3', 'force_term_push_deg')}; "
      f"{Q('s3', 'force_term_bound_deg_per_50ma')}", "not measured",
      "stage 3's numbers, carried", "carried", "carried unchanged")
s3row(5, "compensated", "compensated here in two forms (speed), the form "
      "not chosen", "back-EMF speed per row",
      "product form: c0 " + Q("s3", "v_b_product_form/c0_mv") + ", c_dir "
      + Q("s3", "v_b_product_form/c_dir_mv") + ", b "
      + Q("s3", "v_b_product_form/b_mv_per_rev_s_a"),
      "per row, k_d = 0: ladder alone c0 " + Q("spd", PL + "/c0_mv")
      + ", c_dir " + Q("spd", PL + "/c_dir_mv") + ", b "
      + Q("spd", PL + "/b_mv_per_rev_s_a") + "; ladder + run-ups c0 "
      + Q("spd", PR + "/c0_mv") + ", c_dir " + Q("spd", PR + "/c_dir_mv")
      + ", b " + Q("spd", PR + "/b_mv_per_rev_s_a") + ". bias: "
      + "; ".join(sb(s, b) for s, b in (
          ("run-up", "160-250"), ("run-up", "250-350"),
          ("run-up", "350-450"), ("run-up", "450-600"), ("lifts", "0-30"),
          ("lifts", "30-100"), ("lifts", "100-160"))),
      "the run-ups and the lifts under both forms, the bias column",
      "yes", "agrees that the speed-only table fails under current and the "
      "product holds on the run-ups; b is " + num(100 * (
          val("spd", PL + "/b_mv_per_rev_s_a")
          / val("s3", "v_b_product_form/b_mv_per_rev_s_a") - 1))
      + " % over stage 3's (derived from <spd:" + PL + "/b_mv_per_rev_s_a>"
      " and <s3:v_b_product_form/b_mv_per_rev_s_a>); c_dir with the "
      f"run-ups in the fit moved from {Q('s3', 'v_b_product_form/c_dir_mv')}"
      f" to {Q('spd', PR + '/c_dir_mv')} with k_d = 0 (the speed "
      "notebook's second gap); both forms read high on the lifts; the form "
      "stays open")
s3row(6, "compensated", "compensated here (speed), unchanged",
      "back-EMF speed per row", "the size above",
      f"g {Q('spd', 'constants_used/g_s')} and the brake line "
      f"{Q('spd', 'constants_used/brake_b0_v')}, "
      f"{Q('spd', 'constants_used/brake_b1_ohm')} through the chain; the "
      "per-row port reproduces stage 3's window scores within "
      f"{Q('spd', 'port_gate_flat_band_rev_s')}",
      f"the port's gate {Q('spd', 'port_gate_flat_band_rev_s')}", "yes",
      "agrees")

In [40]:
SL = ("stall_line_cold", "stall_line_warm_quoted")   # stage 3's two lines
DUTIES = ("40 %", "50 %")               # the duties stage 3's size quotes
WARM_CUT = {d: num(100 * (1 - val("s3", f"{SL[1]}/{d}")
                          / val("s3", f"{SL[0]}/{d}")))
            for d in DUTIES}    # derived: warm under cold, % of the current
COLD_OVER = {d: num(100 * (val("s3", f"{SL[0]}/{d}")
                           / val("s3", f"{SL[1]}/{d}") - 1))
             for d in DUTIES}   # derived: cold over warm less one
print("derived from stage 3's stall lines, warm under cold:", WARM_CUT,
      "%; cold over warm less one:", COLD_OVER, "%")
GRADED = Q("wt", "graded_drive_s_max")
s3row(7, "compensated", f"compensated here to {GRADED} of drive from a "
      "rest (winding-temperature); a bound past that",
      "winding temperature and R(T)",
      "stall lines " + "; ".join(
          f"at {d} {Q('s3', f'{SL[0]}/{d}')} cold and "
          f"{Q('s3', f'{SL[1]}/{d}')} warm" for d in DUTIES)
      + f" (derived: warm under cold by {', '.join(WARM_CUT.values())} % of "
      f"the current, cold over warm less one {', '.join(COLD_OVER.values())}"
      " %)",
      f"one node {Q('wt', 'one_node_gain_c_per_w')}, "
      f"{Q('wt', 'one_node_tau_s')}, {Q('wt', 'one_node_c_j_per_c')}; held "
      f"out {Q('wt', 'heldout_rms_pct_all/b_one')} against "
      f"{Q('wt', 'heldout_rms_pct_all/a_ntc')} for the NTC alone; the NTC "
      "alone under load "
      f"{Q('wt', 'load_anchored_mean_pct/[0.45, 0.6)/a_ntc')} of R = "
      f"{Q('wt', 'speed_bias_ntc_alone_rev_s/[0.45, 0.6)')}; one node "
      f"{Q('wt', 'load_anchored_mean_pct/[0.45, 0.6)/b_one')}; NTC span "
      f"{Q('wt', 'ntc_span_c')}",
      f"R after 3 s at 0.5 A {Q('wt', 'hold_pct_at_0p5a_3s_one_node')}; "
      f"graded to {GRADED}",
      "yes", "agrees on the method (R(T) from a thermal estimate); the "
      "stage 3 row said there was no lever, stage 4 found one in the holds' "
      "own heating. The speed and velocity estimators read R(T) from the "
      "NTC alone. The warm stall line is not re-derived; stage 3's warm "
      "line sits above the one node's bracket at 0.5 A (the "
      "winding-temperature notebook's open discrepancy)")

derived from stage 3's stall lines, warm under cold: {'40 %': '7.840', '50 %': '8.843'} %; cold over warm less one: {'40 %': '8.507', '50 %': '9.701'} %


In [41]:
CZ_LINE = "awake_step_line"             # stage 3's step line
s3row(8, "compensated", "compensated here (current-zero), the tracker "
      "extended", "current zero",
      f"line {Q('s3', CZ_LINE + '/per_v')}, {Q('s3', CZ_LINE + '/c0')}, rms "
      f"{Q('s3', CZ_LINE + '/rms')}, above 6 V "
      f"{Q('s3', CZ_LINE + '/per_v_above_6v')}; trough over tail "
      f"{Q('s3', 'trough_over_tail_counts')}; a count "
      f"{Q('s3', 'count_in_rev_s')}",
      f"line {Q('cz', 'awake_step_slope_counts_per_v')}, "
      f"{Q('cz', 'awake_step_intercept_counts')}, held out "
      f"{Q('cz', 'step_line_heldout_rms_counts')}, the curve "
      f"{Q('cz', 'step_curve_heldout_rms_counts')} (held out by "
      f"{node('cz', 'step_heldout_by')} <cz:step_heldout_by>); stage 1's "
      f"tracker fed at {Q('cz', 'feed_gate_pct')} alone: warm "
      f"{Q('cz', 'warm_error_counts_rms')} rms, mean "
      f"{Q('cz', 'warm_error_mean_counts')}, rail moved "
      f"{Q('cz', 'rail_moved_error_counts_rms')}, cold "
      f"{Q('cz', 'cold_error_counts_rms')}, worst "
      f"{Q('cz', 'worst_error_counts')}; drive-feed offset "
      f"{Q('cz', 'drive_feed_offset_counts')}; the design warm "
      f"{Q('cz', 'design_warm_rms_counts')}, events "
      f"{Q('cz', 'design_events_rms_counts')}, worst "
      f"{Q('cz', 'design_all_runs_worst_counts')}; a count "
      f"{Q('spd', 'rev_s_per_count')}",
      f"the design's worst {Q('cz', 'design_all_runs_worst_counts')}",
      "incomplete", "stage 1's tracker with the trough fed at or under the "
      "gate carries the trough's offset under a drive as a warm bias and "
      "starts several counts off after a boot or a rail move; stage 4 adds "
      "the drive-feed constant and the start rule. The step is a curve, "
      "not one line")

In [42]:
DR = "dropout_rule_vs_stage3/reference 32 rows"   # the per-row rule's hit
HL = "dropout_hold_last_effect"         # hold-last at the top band
s3row(9, "designed around", "designed around (speed): k_d = 0, a per-row "
      "dropout rule in the valid flag; the rate flag itself not built",
      "back-EMF speed per row, its valid rule",
      f"k_d {Q('s3', 'k_d_recommended')}; the rate flag "
      f"{Q('s3', 'dropout_flag_rule/reference')}, excess "
      f"{Q('s3', 'dropout_flag_rule/excess')}, span "
      f"{Q('s3', 'dropout_flag_rule/span')}",
      "k_d = 0 kept; the per-row rule against stage 3's marks: hit rate "
      f"{Q('spd', DR + '/hit_rate')}, precision {Q('spd', DR + '/precision')}"
      f", n {Q('spd', DR + '/n')}, flags {Q('spd', DR + '/flags')}; dropout "
      f"rows {Q('spd', COV + '/dropout_pct')} % of driven rows; hold-last "
      "at 450-600 rev/s moves the bias from "
      f"{Q('spd', HL + '/none/450-600/bias')} to "
      f"{Q('spd', HL + '/hold-last/450-600/bias')}; stage 3's rate flag: "
      "not measured",
      f"precision {Q('spd', DR + '/precision')}", "yes",
      "agrees on k_d = 0; the fault-side rate flag is carried from stage 3 "
      "unrun")

In [43]:
TS = node("spd", "brush_forms/table_signed_mv")   # the signed table, mV
s3row(10, "compensated", "compensated here (speed: the signed table or "
      "the product's c0), residual stated; cause still open",
      "back-EMF speed per row",
      "at a crawl against the tag, the full construction "
      f"{Q('s3', 'speed_by_band_clips_tag/full/0-30/mean')}, with the "
      f"signed table {Q('s3', 'speed_by_band_clips_tag/signed/0-30/mean')}",
      "signed table at its first two knots "
      "<spd:brush_forms/table_signed_mv>: +1 "
      f"{num(TS['+1'][0])}, {num(TS['+1'][1])} mV, -1 {num(TS['-1'][0])}, "
      f"{num(TS['-1'][1])} mV; crawl on the camera sitting: table "
      f"{Q('vel', 'emf_bias_crawl_rev_s/table')}, product "
      f"{Q('vel', 'emf_bias_crawl_rev_s/product')}; free low holds: "
      f"{sb('holds (free, low)', '0-30')}; "
      f"{sb('ladder (third repeat)', '30-100')}; direction split under 45 "
      f"rev/s table {Q('vel', 'emf_direction_split_rev_s/table/under_45')},"
      f" product {Q('vel', 'emf_direction_split_rev_s/product/under_45')};"
      f" under the floor {Q('vel', 'emf_bias_under_floor_rev_s/table')}",
      f"crawl {Q('vel', 'emf_bias_crawl_rev_s/table')} (table), "
      f"{Q('vel', 'emf_bias_crawl_rev_s/product')} (product)", "yes",
      "agrees: stage 3's crawl offset is closed to "
      f"{Q('vel', 'emf_bias_crawl_rev_s/table')} by the table and "
      f"{Q('vel', 'emf_bias_crawl_rev_s/product')} by the product; a "
      "direction split the forms do not carry remains, and the table "
      "form's crawl bias differs between the speed and velocity notebooks "
      "(section 4)")

In [44]:
NA = "not measured"
s3row(11, "carried", "not stage 4's (owner 5); carried", "none",
      f"ripple {Q('s3', 'crawl_speed_ripple_sd_over_mean')} of the mean",
      f"{NA}. Related: mechanical time constant "
      f"{Q('vel', 'mechanical_tau_ms')}; the crawl's motion locked to the "
      f"horn's angle {Q('vel', 'crawl_angle_locked_motion_rev_s/sd')}; the "
      "commutation-locked part of the back-EMF "
      f"{Q('vel', 'emf_commutation_locked_share_crawl')}, of which motion "
      f"at most {Q('vel', 'emf_commutation_motion_bound_rev_s')}",
      "stage 3's numbers, carried", "carried", "carried unchanged; stage 4 "
      "adds that most of the back-EMF's ripple at a crawl is electrical and "
      "not motion")
s3row(12, "designed around", "designed around (velocity: the rest "
      "verdict); the drift stays unmeasured", "rest verdict",
      f"pos sd {Q('s3', 'hold_pos_sd_counts')}; hold drift measured "
      f"{Q('s3', 'hold_drift_measured')}",
      f"at rest per row {Q('pos', 'pot_noise_row_counts')}, boxcar 16 "
      f"{Q('vel', 'rest_sigma16_counts')}; the verdict holds stage 3's "
      "load holds as motion on "
      f"{Q('vel', 'rest_false_motion_stage3_share/band 2 T50/load holds')} "
      "(band) and "
      f"{Q('vel', 'rest_false_motion_stage3_share/slope 192 T50/load holds')}"
      " (slope)", "the drift over seconds: not measured", "yes",
      "agrees; the drift over seconds is still not in the data")
s3row(13, "carried", "not stage 4's (owner 6); carried", "none",
      "the size above", NA, "stage 3's numbers, carried", "carried",
      "carried unchanged")
s3row(14, "carried", "not stage 4's (owner 6); carried", "none",
      "the size above", f"{NA}; the rows carry no rail channel "
      f"({Q('spd', 'nan_row_columns/vbus')} all nan), so a rail at rest is "
      "not in the stage 4 rows", "stage 3's numbers, carried", "carried",
      "carried unchanged; stage 4's zero start rule needs the same rail "
      "read")

In [45]:
TD = "tap_deficit_by_ticks"             # the taps' deficit by on-time
TS3 = "valid_rule/tap_floor_stage3_closing_ticks"
s3row(15, "designed around", "designed around (speed): the estimate "
      "invalid under the floor, the pot's slope the source there",
      "back-EMF speed per row, its valid rule",
      f"{Q('s3', 'tap_floor_half_ticks')}, {Q('s3', 'tap_floor_v_low')}; the"
      f" closing table's {Q('spd', TS3)}",
      f"floor {Q('spd', 'valid_rule/tap_floor_ticks')}; deficit at 72 ticks "
      f"{Q('spd', TD + '/72-72/deficit_mv')} (n {Q('spd', TD + '/72-72/n')},"
      f" window_valid {Q('spd', TD + '/72-72/window_valid_share')}), at 84 "
      f"{Q('spd', TD + '/84-84/deficit_mv')} (n {Q('spd', TD + '/84-84/n')})"
      f", at 96 {Q('spd', TD + '/96-96/deficit_mv')} (n "
      f"{Q('spd', TD + '/96-96/n')}); as speed "
      f"{Q('spd', TD + '/72-72/as_rev_s')} and "
      f"{Q('spd', TD + '/84-84/as_rev_s')} rev/s; rows under the floor "
      f"{Q('spd', COV + '/under_floor_pct')} %, under 96 ticks "
      f"{Q('spd', COV + '/under_96_ticks_pct')} %, window_valid 0 on "
      f"{Q('spd', COV + '/window_valid_0_pct')} %",
      f"rows under the floor {Q('spd', COV + '/under_floor_pct')} %, not "
      "valid there", "yes",
      "agrees that window_valid does not cover the taps; the number moved "
      f"from {Q('spd', TS3)} to {num(FLOOR_Q['value'])} "
      f"<spd:valid_rule/tap_floor_ticks> on "
      f"{Q('spd', TD + '/84-84/n')} runs, with {num(FLOOR_Q['hi'])} "
      "<spd:valid_rule/tap_floor_ticks> the first well-populated rung (an "
      "open choice, section 5). 'V_on from the rail' was not tried: the rows "
      "carry no rail")
s3row(16, "carried", "not stage 4's (owner 6); carried", "none",
      "the size above", f"{NA}. Related: a stalled hold is declared at rest "
      f"in {Q('vel', 'rest_declare_ms/band 2 T50/stall_in_hold/median')} "
      "(band) and "
      f"{Q('vel', 'rest_declare_ms/slope 192 T50/stall_in_hold/median')} "
      f"(slope); R(T) is graded to {GRADED} only",
      "stage 3's numbers, carried", "carried", "carried unchanged; the "
      "fault 'from R(T)' has no graded basis past the graded drive")
STAGE3 = pd.DataFrame(ROWS3)
print(len(STAGE3), "rows;", STAGE3["disposition"].value_counts().to_dict())
print(STAGE3["agrees"].value_counts().to_dict())

17 rows; {'compensated': 6, 'designed around': 4, 'carried': 4, 'bound': 2, 'compensated (position); designed around (velocity)': 1}
{'yes': 8, 'carried': 5, 'incomplete': 2, 'changed': 1, 'no': 1}


In [46]:
show(ROWS3, ["row", "effect", "stage3_how", "disposition_how",
             "estimator", "agrees", "agrees_how"],
     ["#", "effect", "stage 3 said", "stage 4 did", "by", "agrees",
      "what stage 4 says"])

| # | effect | stage 3 said | stage 4 did | by | agrees | what stage 4 says |
|---|---|---|---|---|---|---|
| 0 | pot scale and smooth bend | Compensate in firmware, the table at 1.5 deg knots | compensated here (position) | position | yes | agrees on a table; the form changed from knots in angle to knots uniform in counts (no search) |
| 1 | pot fine texture | Compensate in firmware for position, the table. Design around for velocity, pot windows of 200 ms and longer and the EMF under that | position: compensated here (position), residual stated; velocity: designed around (velocity), the rule changed | position; velocity as the pot's slope | changed | position agrees. velocity: the direction agrees, the window moved (crawl earlier, pass later, brackets wide) and the rule is restated: the pot's slope where the back-EMF is not valid or where 50 ms and more of delay is acceptable `vel:stage5_by_regime/3`, not stage 3's fixed window |
| 2 | converter's wide and missing codes | Design around, mask +/-12 counts of a carry in the velocity. Bound the position at 0.71 deg there | designed around (position: no mask; velocity: skip, never hold) | position; velocity as the pot's slope | incomplete | the mask holds only as rows skipped in a slope under 50 ms, about twice the longest dwell at a crawl `vel:carry_mask_until_window_ms` of window, +/-12 counts, the longest missing run `vel:carry_mask_half_counts`; holding the pot through the band makes both the position and the velocity worse. The widest carry by blind span is 1023 `pos:carry_codes`, not the code stage 3's size names (a different construction, the position notebook's open discrepancy) |
| 3 | lash | Compensate in firmware, a direction-signed 0.17 deg offset on the position. Bound the rest on the hold | left as a bound (position), not compensated | position | no | the fixed direction-signed offset stage 3 proposed does not hold across the sitting's clips: it changes sign between clips, and the position carries a bound instead |
| 4 | pot force term | Bound, the board does not see it. The push block and a tag under torque measure it | left as a bound, stage 7's (no stage 4 result) | none | carried | carried unchanged |
| 5 | brush drop and its direction term | Compensate in firmware. A speed-only table holds at the free current only. Under current the product form c0 + c_dir + b w i fits this servo, three constants and a sign, the ladder alone giving b to 2.6 %. The matched-speed two-current experiment decides the form | compensated here in two forms (speed), the form not chosen | back-EMF speed per row | yes | agrees that the speed-only table fails under current and the product holds on the run-ups; b is 5.234 % over stage 3's (derived from `spd:brush_forms/product_ladder_alone/b_mv_per_rev_s_a` and `s3:v_b_product_form/b_mv_per_rev_s_a`); c_dir with the run-ups in the fit moved from 14.37 [12.80, 15.95] mV, +/- se `s3:v_b_product_form/c_dir_mv` to 36.02 [34.22, 37.82] mV, +/- se `spd:brush_forms/product_ladder_runups/c_dir_mv` with k_d = 0 (the speed notebook's second gap); both forms read high on the lifts; the form stays open |
| 6 | reading term | Compensate in firmware, as construction C with the meter-confirmed g | compensated here (speed), unchanged | back-EMF speed per row | yes | agrees |
| 7 | R(T) | Compensate in firmware, R(T) from the thermal estimate and the stall fault from the warm line | compensated here to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` of drive from a rest (winding-temperature); a bound past that | winding temperature and R(T) | yes | agrees on the method (R(T) from a thermal estimate); the stage 3 row said there was no lever, stage 4 found one in the holds' own heating. The speed and velocity estimators read R(T) from the NTC alone. The warm stall line is not re-derived; stage 3's warm line sits above the one node's bracket at 0.5 A (the winding-temperature notebook's open discrepancy) |
| 8 | current zero's awake step | Compensate in firmware, stage 1's two-zero tracker with the trough fed at &lt;= 30 %. A constant costs 0.2 to 1.5 rev/s rms and a -30 mV rail split | compensated here (current-zero), the tracker extended | current zero | incomplete | stage 1's tracker with the trough fed at or under the gate carries the trough's offset under a drive as a warm bias and starts several counts off after a boot or a rail move; stage 4 adds the drive-feed constant and the start rule. The step is a curve, not one line |
| 9 | dropout rate | Flag in firmware, the rate against the reference + max(3 se, 100/s) over >= 50 ms, no live term | designed around (speed): k_d = 0, a per-row dropout rule in the valid flag; the rate flag itself not built | back-EMF speed per row, its valid rule | yes | agrees on k_d = 0; the fault-side rate flag is carried from stage 3 unrun |
| 10 | low-speed offset of the construction | Compensate in firmware, the signed table (-24 to -14 mV at 50 to 100 rev/s) or a V0 under rotation lowered by as much, the product form's c0 of -35.6 mV the same term. The cause is open (the ammeter experiment) | compensated here (speed: the signed table or the product's c0), residual stated; cause still open | back-EMF speed per row | yes | agrees: stage 3's crawl offset is closed to 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table` by the table and 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product` by the product; a direction split the forms do not carry remains, and the table form's crawl bias differs between the speed and velocity notebooks (section 4) |
| 11 | train friction pair and stiction | Compensate in firmware, the feed-forward 47 mA + 0.26 mA per rev/s (friction). Design around, the loop and not a duty carries a crawl under the floor's speed | not stage 4's (owner 5); carried | none | carried | carried unchanged; stage 4 adds that most of the back-EMF's ripple at a crawl is electrical and not motion |
| 12 | hold deadband | Design around, integral action or dither in the position loop. A streamed hold measures the drift | designed around (velocity: the rest verdict); the drift stays unmeasured | rest verdict | yes | agrees; the drift over seconds is still not in the data |
| 13 | Kt through the gears and its side split | Compensate in firmware, the one Kt through the gears for the torque limit. The dip sits inside the static friction band, design around | not stage 4's (owner 6); carried | none | carried | carried unchanged |
| 14 | sag cap | Compensate in firmware, the cap from the rail at rest, read to 0.5 % | not stage 4's (owner 6); carried | none | carried | carried unchanged; stage 4's zero start rule needs the same rail read |
| 15 | tap floor | Design around, V_on from the rail under 93 half-ticks or the estimate invalid there. The board's window_valid has to cover the taps | designed around (speed): the estimate invalid under the floor, the pot's slope the source there | back-EMF speed per row, its valid rule | yes | agrees that window_valid does not cover the taps; the number moved from 93 ticks `spd:valid_rule/tap_floor_stage3_closing_ticks` to 84 `spd:valid_rule/tap_floor_ticks` on 3 `spd:tap_deficit_by_ticks/84-84/n` runs, with 96 `spd:valid_rule/tap_floor_ticks` the first well-populated rung (an open choice, section 5). 'V_on from the rail' was not tried: the rows carry no rail |
| 16 | current ceiling and stall fault | Compensate in firmware, the limit as the ceiling and the fault from the warm line or R(T) | not stage 4's (owner 6); carried | none | carried | carried unchanged; the fault 'from R(T)' has no graded basis past the graded drive |

The numbers of each row, stage 3's beside stage 4's, and the bound or residual the row leaves.

In [47]:
show(ROWS3, ["row", "stage3_size", "stage3_number", "stage4_number",
             "residual"],
     ["#", "stage 3's size", "stage 3's number", "stage 4's number",
      "residual or bound"])

| # | stage 3's size | stage 3's number | stage 4's number | residual or bound |
|---|---|---|---|---|
| 0 | 18.9 counts/deg, 50 counts pp over the travel `s3:closing_table/0` | the angle form at 1.5 deg knots, held out in stage 4: 0.174 deg `pos:angle_form_1p5_deg_heldout_deg_rms` | lookup indexed by counts, pitch 32 counts `pos:lookup_pitch_counts`, 98 entries `pos:lookup_entries_guard_to_guard`; table term held out 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms`; a 64-count pitch 0.186 deg `pos:lookup_64_counts_heldout_deg_rms` | table term held out 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms` |
| 1 | 6.0 counts rms (0.32 deg). 2.8 to 5.2 % on a 100 ms pot speed at a spot, 0.22 rev/s x (1 s / W) tiling the travel `s3:closing_table/1` | position in-sample 0.4 deg `s3:position_resid_rms_deg`; the crossover against the signed table 134 `s3:crossover_ms/0-30/w_signed / pot_two` and 146 `s3:crossover_ms/0-30/w_signed / pot_line` ms at 0-30 rev/s, 119 `s3:crossover_ms/30-100/w_signed / pot_two` and 92 `s3:crossover_ms/30-100/w_signed / pot_line` at 30-100, against the product 77 `s3:crossover_ms/0-30/w_product / pot_two` and 100 `s3:crossover_ms/0-30/w_product / pot_line` | position held out 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms` with segment offsets out, 0.41 [0.376, 0.446] deg `pos:heldout_lead_applied_deg_rms` with nothing out but the lead; crossover, causal: crawl rms 103.0 [66.15, 115.2] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms`, line 127.4 [120.0, 137.0] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms_line`, product 50.81 [38.96, 64.82] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms_product`, own noise 141.8 [115.0, 221.8] ms of window, 5-95 % of 200 resamplings, never 13% `vel:crossover_window_ms/crawl_own_noise`, steps 120.2 [85.80, 202.0] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/steps_rms`, pass 200.4 [114.7, 224.8] ms of window, 5-95 % of 200 resamplings, never 20% `vel:crossover_window_ms/pass_rms`; at 5 ms of delay the pot's own noise 7.434 `vel:stage5_by_regime/2/noise_rev_s` against the back-EMF's 1.718 `vel:stage5_by_regime/0/noise_rev_s` rev/s | position 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms`; velocity: the crossover's bracket 103.0 [66.15, 115.2] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms` |
| 2 | 0.71 deg at 2047, 0.20 to 0.62 at the other carries `s3:closing_table/2` | the size and the mask above | carries at [1023, 1279, 2047, 2303, 2559] `pos:carry_codes`, blind over [0.772, 0.139, 0.467, 0.0298, 0.279] `pos:carry_blind_span_deg` deg, missing runs [12, 3, 11, 3, 7] `pos:carry_missing_codes`. position: near a carry 0.201 deg `pos:carry_near_rms_deg`, far 0.176 deg `pos:carry_far_rms_deg`, with a hold of the band 0.698 deg `pos:carry_hold_near_rms_deg` and 29.57 [-, 108.3] ms `pos:carry_hold_added_delay_ms` of added delay. velocity, two-point near a carry at 12.8 ms: none 19.50 deg/s rms `vel:carry_near_rms_deg_s/none/near/12.8 ms`, hold12 35.51 deg/s rms `vel:carry_near_rms_deg_s/hold12/near/12.8 ms`, skip12 7.504 deg/s rms `vel:carry_near_rms_deg_s/skip12/near/12.8 ms`, far 8.181 deg/s rms `vel:carry_near_rms_deg_s/none/far/12.8 ms`; at 51.2 ms none 4.942 deg/s rms `vel:carry_near_rms_deg_s/none/near/51.2 ms`, skip12 5.220 deg/s rms `vel:carry_near_rms_deg_s/skip12/near/51.2 ms`; mask rule skip `vel:carry_mask` +/-12 counts, the longest missing run `vel:carry_mask_half_counts` under 50 ms, about twice the longest dwell at a crawl `vel:carry_mask_until_window_ms` | position blind over [0.772, 0.139, 0.467, 0.0298, 0.279] `pos:carry_blind_span_deg` deg |
| 3 | play 0.39 to 1.12 deg. The pot leads the horn by +0.17 / -0.18 deg by direction at a crawl `s3:closing_table/3` | 0.166 deg `s3:position_resid_by_dir_crawl_deg/+1` and -0.177 deg `s3:position_resid_by_dir_crawl_deg/-1` by direction at a crawl | on steady crawl holds 0.274 [0.249, 0.3] deg `pos:direction_offset_crawl_plus_deg` and -0.227 [-0.257, -0.196] deg `pos:direction_offset_crawl_minus_deg`; clip means from -0.194 deg `pos:direction_offset_clip_min_deg` to 0.528 deg `pos:direction_offset_clip_max_deg`; at rest -0.0526 [-0.0754, -0.0298] deg `pos:rest_direction_split_deg`; position-controlled contrast -0.0551 [-0.103, -0.00676] deg `pos:direction_contrast_in_bin_deg`, speed-free half width 0.192 [0.145, 0.238] deg `pos:speed_free_half_width_deg`, timing slope -5.880 [-6.797, -4.963] ms `pos:timing_slope_ms`; a direction source, if ever: hysteresis 4 [4, 6] counts `pos:direction_source_pot_hysteresis_counts`, late 8.399 [0, 8.399] ms `pos:direction_source_late_ms` | bound 0.5 deg `pos:direction_bound_moving_deg` in motion, 0.3 deg `pos:direction_bound_rest_deg` at rest |
| 4 | 0.51 deg under a push, relaxing 0.31. At most 0.33 deg per 50 mA stalled `s3:closing_table/4` | 0.509 deg each way, quoted `s3:force_term_push_deg`; 0.334 deg, quoted, a bound `s3:force_term_bound_deg_per_50ma` | not measured | stage 3's numbers, carried |
| 5 | X = -35.6 + 14.4 [+1] + 1.96 w i mV on this servo. The speed table reads +26.4 to +29.5 rev/s on the 100 % run-ups at 0.48 to 0.70 A `s3:closing_table/5` | product form: c0 -35.57 [-37.24, -33.89] mV, +/- se `s3:v_b_product_form/c0_mv`, c_dir 14.37 [12.80, 15.95] mV, +/- se `s3:v_b_product_form/c_dir_mv`, b 1.956 [1.943, 1.969] mV per (rev/s x A), +/- se `s3:v_b_product_form/b_mv_per_rev_s_a` | per row, k_d = 0: ladder alone c0 -40.43 [-43.26, -37.60] mV, +/- se `spd:brush_forms/product_ladder_alone/c0_mv`, c_dir 12.39 [9.149, 15.63] mV, +/- se `spd:brush_forms/product_ladder_alone/c_dir_mv`, b 2.058 [2.010, 2.107] mV per (rev/s x A), +/- se `spd:brush_forms/product_ladder_alone/b_mv_per_rev_s_a`; ladder + run-ups c0 -46.45 [-48.37, -44.54] mV, +/- se `spd:brush_forms/product_ladder_runups/c0_mv`, c_dir 36.02 [34.22, 37.82] mV, +/- se `spd:brush_forms/product_ladder_runups/c_dir_mv`, b 2.052 [2.037, 2.067] mV per (rev/s x A), +/- se `spd:brush_forms/product_ladder_runups/b_mv_per_rev_s_a`. bias: run-up 160-250 (218.9 rev/s, 655 mA): table 30, product -1.340 rev/s `spd:stage5_bias_by_band_form/11`; run-up 250-350 (306.2 rev/s, 512 mA): table 27.10, product -1.920 rev/s `spd:stage5_bias_by_band_form/12`; run-up 350-450 (408.7 rev/s, 349 mA): table 12.45, product 2.310 rev/s `spd:stage5_bias_by_band_form/13`; run-up 450-600 (474.4 rev/s, 249 mA): table 3.330, product 7.690 rev/s `spd:stage5_bias_by_band_form/14`; lifts 0-30 (11.50 rev/s, 195 mA): table 3.460, product 4.320 rev/s `spd:stage5_bias_by_band_form/15`; lifts 30-100 (58.60 rev/s, 205 mA): table 5.220, product 3.730 rev/s `spd:stage5_bias_by_band_form/16`; lifts 100-160 (111.5 rev/s, 218 mA): table 7.420, product 3.580 rev/s `spd:stage5_bias_by_band_form/17` | the run-ups and the lifts under both forms, the bias column |
| 6 | g 4.75 mS of (1 - D) V_on off the current. Without it -15.1 / -12.2 rev/s at 94 / 137 rev/s and a -79 mV rail split `s3:closing_table/6` | the size above | g 0.00475 S `spd:constants_used/g_s` and the brake line 0.00875 V `spd:constants_used/brake_b0_v`, 0.24 ohm `spd:constants_used/brake_b1_ohm` through the chain; the per-row port reproduces stage 3's window scores within 0.15 [-0.15, 0.107] rev/s `spd:port_gate_flat_band_rev_s` | the port's gate 0.15 [-0.15, 0.107] rev/s `spd:port_gate_flat_band_rev_s` |
| 7 | 0.39 %/C. No lever here, the NTC at 24.5 to 27.5 C. The warm 3 s stall 7.8 to 8.8 % under cold at 40 to 50 % `s3:closing_table/7` | stall lines at 40 % 449.7 mA `s3:stall_line_cold/40 %` cold and 414.4 mA `s3:stall_line_warm_quoted/40 %` warm; at 50 % 564.1 mA `s3:stall_line_cold/50 %` cold and 514.2 mA `s3:stall_line_warm_quoted/50 %` warm (derived: warm under cold by 7.840, 8.843 % of the current, cold over warm less one 8.507, 9.701 %) | one node 23.24 [14.27, 23.85] C/W (fold_lo 14.27, fold_hi 23.24, stat_factor 1.012) `wt:one_node_gain_c_per_w`, 2.375 [1.203, 3.227] s (fold_lo 1.400, fold_hi 2.699, stat_factor 1.044) `wt:one_node_tau_s`, 0.102 [0.0625, 0.135] J/C (fold_lo 0.0981, fold_hi 0.117) `wt:one_node_c_j_per_c`; held out 1.055 % `wt:heldout_rms_pct_all/b_one` against 1.292 % `wt:heldout_rms_pct_all/a_ntc` for the NTC alone; the NTC alone under load 3.865 [3.077, 4.652] % `wt:load_anchored_mean_pct/[0.45, 0.6)/a_ntc` of R = 10.72 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.45, 0.6)`; one node -0.0565 [-1.243, 1.130] % `wt:load_anchored_mean_pct/[0.45, 0.6)/b_one`; NTC span 26.60 [24.54, 28.66] C `wt:ntc_span_c` | R after 3 s at 0.5 A 7.121 [5.351, 7.121] % `wt:hold_pct_at_0p5a_3s_one_node`; graded to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` |
| 8 | 1.82 counts/V -3.76, +0.05 counts/V above 6 V. 1 count = 0.51 rev/s `s3:closing_table/8` | line 1.817 counts/V `s3:awake_step_line/per_v`, -3.759 counts `s3:awake_step_line/c0`, rms 0.841 counts `s3:awake_step_line/rms`, above 6 V 0.054 counts/V `s3:awake_step_line/per_v_above_6v`; trough over tail 0.74 [0.318, 1.161] counts, +/- sd `s3:trough_over_tail_counts`; a count 0.505 rev/s per count of zero `s3:count_in_rev_s` | line 1.667 [1.630, 1.704] counts/V `cz:awake_step_slope_counts_per_v`, -2.717 [-2.916, -2.518] counts `cz:awake_step_intercept_counts`, held out 0.733 counts `cz:step_line_heldout_rms_counts`, the curve 0.437 counts `cz:step_curve_heldout_rms_counts` (held out by capture set `cz:step_heldout_by`); stage 1's tracker fed at 30 % `cz:feed_gate_pct` alone: warm 0.73 [0.684, 0.778] counts `cz:warm_error_counts_rms` rms, mean 0.584 counts `cz:warm_error_mean_counts`, rail moved 2.237 [1.260, 3.017] counts `cz:rail_moved_error_counts_rms`, cold 1.675 [0.796, 2.176] counts `cz:cold_error_counts_rms`, worst 7.204 counts `cz:worst_error_counts`; drive-feed offset 0.667 [0.657, 0.698] counts `cz:drive_feed_offset_counts`; the design warm 0.411 [0.373, 0.447] counts `cz:design_warm_rms_counts`, events 0.361 [0.241, 0.475] counts `cz:design_events_rms_counts`, worst 1.967 counts `cz:design_all_runs_worst_counts`; a count 0.471 [0.471, 0.517] rev/s per count of zero `spd:rev_s_per_count` | the design's worst 1.967 counts `cz:design_all_runs_worst_counts` |
| 9 | 0 below 420 rev/s. 26 -> 71 per s at 100 % over two days. The data want k_d about 0, -0.024 rev/s per dropout/s without it `s3:closing_table/9` | k_d 0 mV per dropout/s `s3:k_d_recommended`; the rate flag 25.77 [16.66, 34.88] per s `s3:dropout_flag_rule/reference`, excess 100 per s, max(3 se, 100) `s3:dropout_flag_rule/excess`, span 50 ms, at least `s3:dropout_flag_rule/span` | k_d = 0 kept; the per-row rule against stage 3's marks: hit rate 1 `spd:dropout_rule_vs_stage3/reference 32 rows/hit_rate`, precision 0.957 `spd:dropout_rule_vs_stage3/reference 32 rows/precision`, n 45 `spd:dropout_rule_vs_stage3/reference 32 rows/n`, flags 47 `spd:dropout_rule_vs_stage3/reference 32 rows/flags`; dropout rows 0.0249 `spd:valid_coverage_by_set/all/dropout_pct` % of driven rows; hold-last at 450-600 rev/s moves the bias from 9.455 [8.116, 10.79] rev/s, +/- se `spd:dropout_hold_last_effect/none/450-600/bias` to 8.527 [7.354, 9.701] rev/s, +/- se `spd:dropout_hold_last_effect/hold-last/450-600/bias`; stage 3's rate flag: not measured | precision 0.957 `spd:dropout_rule_vs_stage3/reference 32 rows/precision` |
| 10 | X -28 to -15 mV under 100 rev/s, -2.8 rev/s at 94 rev/s and -2.6 at a crawl, rail-flat `s3:closing_table/10` | at a crawl against the tag, the full construction -2.581 [-2.674, -2.487] rev/s, +/- se `s3:speed_by_band_clips_tag/full/0-30/mean`, with the signed table 0.382 [0.285, 0.48] rev/s, +/- se `s3:speed_by_band_clips_tag/signed/0-30/mean` | signed table at its first two knots `spd:brush_forms/table_signed_mv`: +1 -23.95, -13.74 mV, -1 -18.84, -14.57 mV; crawl on the camera sitting: table 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table`, product 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product`; free low holds: holds (free, low) 0-30 (23.40 rev/s, 39 mA): table -0.88, product 0.89 rev/s `spd:stage5_bias_by_band_form/6`; ladder (third repeat) 30-100 (94 rev/s, 73 mA): table -0.31, product 0.12 rev/s `spd:stage5_bias_by_band_form/0`; direction split under 45 rev/s table 0.814 [0.746, 0.883] rev/s, +/- se `vel:emf_direction_split_rev_s/table/under_45`, product -1.828 [-1.896, -1.759] rev/s, +/- se `vel:emf_direction_split_rev_s/product/under_45`; under the floor -2.251 [-2.649, -1.854] rev/s, +/- se over segments `vel:emf_bias_under_floor_rev_s/table` | crawl 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table` (table), 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product` (product) |
| 11 | Tc 55 uN m (47 mA), b 0.26 mA per rev/s. Stiction >= 50 to 60 uN m driven, a bound, and 81 to 117 uN m back-driven, measured (load 11.7). The floor 0.34 V effective at 20 rev/s `s3:closing_table/11` | ripple 0.366 ratio `s3:crawl_speed_ripple_sd_over_mean` of the mean | not measured. Related: mechanical time constant 18.29 [11.94, 26.66] ms, median [10-90 %] of the kept steps `vel:mechanical_tau_ms`; the crawl's motion locked to the horn's angle 2.236 rev/s `vel:crawl_angle_locked_motion_rev_s/sd`; the commutation-locked part of the back-EMF 0.719 [0.701, 0.738] of the per-row variance, +/- se over segments `vel:emf_commutation_locked_share_crawl`, of which motion at most 0.183 [0.155, 0.281] rev/s sd, [range by tile] `vel:emf_commutation_motion_bound_rev_s` | stage 3's numbers, carried |
| 12 | A ratchet, follows a rise of 5 to 10 % of duty and holds through a cut of 20 to 25 % (load 11.4), measured. Pos sd 0.87 counts at rest under torque. The drift over seconds is unmeasured `s3:closing_table/12` | pos sd 0.871 [0.521, 1.524] counts `s3:hold_pos_sd_counts`; hold drift measured False `s3:hold_drift_measured` | at rest per row 1.031 [1.027, 1.160] counts `pos:pot_noise_row_counts`, boxcar 16 0.137 [0, 0.165] counts, median [range] over the baselines `vel:rest_sigma16_counts`; the verdict holds stage 3's load holds as motion on 0.0058 share of rows `vel:rest_false_motion_stage3_share/band 2 T50/load holds` (band) and 0.0011 share of rows `vel:rest_false_motion_stage3_share/slope 192 T50/load holds` (slope) | the drift over seconds: not measured |
| 13 | One for both directions, 225.7 mN m/A (221 to 242) over three sittings. The split as a ratio not established. The rests dip 3 to 6 mN m at 41.5 to 49.5 deg on both sides, cause open (rig or train) `s3:closing_table/13` | the size above | not measured | stage 3's numbers, carried |
| 14 | 100 x 5 / rail, 59.5 to 83.5 %, 1.42 to 2.79 points per 0.2 V `s3:closing_table/14` | the size above | not measured; the rows carry no rail channel (22853600 rows `spd:nan_row_columns/vbus` all nan), so a rail at rest is not in the stage 4 rows | stage 3's numbers, carried |
| 15 | V_on 0.33 V low at 72 half-ticks with window_valid 1. The flag follows the current's 64-tick floor, the taps need 93 `s3:closing_table/15` | 72 half-ticks, V_on low with window_valid 1 `s3:tap_floor_half_ticks`, 0.333 V `s3:tap_floor_v_low`; the closing table's 93 ticks `spd:valid_rule/tap_floor_stage3_closing_ticks` | floor 84 [72, 96] ticks; bounds: the failing rung below, the first rung with 10 runs or more `spd:valid_rule/tap_floor_ticks`; deficit at 72 ticks 482.5 [341.8, 504.8] mV, median [10-90 %] `spd:tap_deficit_by_ticks/72-72/deficit_mv` (n 15 `spd:tap_deficit_by_ticks/72-72/n`, window_valid 1 `spd:tap_deficit_by_ticks/72-72/window_valid_share`), at 84 10.50 [9.413, 29.59] mV, median [10-90 %] `spd:tap_deficit_by_ticks/84-84/deficit_mv` (n 3 `spd:tap_deficit_by_ticks/84-84/n`), at 96 -22.81 [-29.34, -10.51] mV, median [10-90 %] `spd:tap_deficit_by_ticks/96-96/deficit_mv` (n 16 `spd:tap_deficit_by_ticks/96-96/n`); as speed -3.975 `spd:tap_deficit_by_ticks/72-72/as_rev_s` and -0.101 `spd:tap_deficit_by_ticks/84-84/as_rev_s` rev/s; rows under the floor 14.62 `spd:valid_coverage_by_set/all/under_floor_pct` %, under 96 ticks 21.11 `spd:valid_coverage_by_set/all/under_96_ticks_pct` %, window_valid 0 on 9.021 `spd:valid_coverage_by_set/all/window_valid_0_pct` % | rows under the floor 14.62 `spd:valid_coverage_by_set/all/under_floor_pct` %, not valid there |
| 16 | lock 0.85 A, stop 0.5, lap 0.45. The cold line (D V_on - 0.033) / 4.177, warm 10.0 mA per % `s3:closing_table/16` | the size above | not measured. Related: a stalled hold is declared at rest in 55.33 ms, median `vel:rest_declare_ms/band 2 T50/stall_in_hold/median` (band) and 58.73 ms, median `vel:rest_declare_ms/slope 192 T50/stall_in_hold/median` (slope); R(T) is graded to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` only | stage 3's numbers, carried |

### Stage 3 against stage 4 where both carry a bracket

The product form's constants and the awake step's line, derived as a difference over the combined standard error. Stage 3's step line carries no se, so it is stated in stage 4's half brackets.

In [48]:
for k in ("c0_mv", "c_dir_mv", "b_mv_per_rev_s_a"):
    for fit in (PL, PR):
        print(sigma(val("s3", f"v_b_product_form/{k}"),
                    se_of("s3", f"v_b_product_form/{k}"),
                    val("spd", f"{fit}/{k}"), se_of("spd", f"{fit}/{k}"),
                    f"<s3:v_b_product_form/{k}> less <spd:{fit}/{k}>"))
for s3k, czk, u in (("per_v", "awake_step_slope_counts_per_v", "counts/V"),
                    ("c0", "awake_step_intercept_counts", "counts")):
    d = val("s3", f"{CZ_LINE}/{s3k}") - val("cz", czk)
    print(f"derived: awake step {s3k}, <s3:{CZ_LINE}/{s3k}> "
          f"{num(val('s3', f'{CZ_LINE}/{s3k}'))} (no se) less <cz:{czk}> "
          f"{num(val('cz', czk))}: {num(d)} {u}, "
          f"{num(d / se_of('cz', czk))} of stage 4's half bracket")

derived: <s3:v_b_product_form/c0_mv> less <spd:brush_forms/product_ladder_alone/c0_mv>: difference 4.866, combined se 3.287, 1.48 sigma
derived: <s3:v_b_product_form/c0_mv> less <spd:brush_forms/product_ladder_runups/c0_mv>: difference 10.89, combined se 2.547, 4.27 sigma
derived: <s3:v_b_product_form/c_dir_mv> less <spd:brush_forms/product_ladder_alone/c_dir_mv>: difference 1.984, combined se 3.603, 0.55 sigma
derived: <s3:v_b_product_form/c_dir_mv> less <spd:brush_forms/product_ladder_runups/c_dir_mv>: difference -21.65, combined se 2.393, 9.05 sigma
derived: <s3:v_b_product_form/b_mv_per_rev_s_a> less <spd:brush_forms/product_ladder_alone/b_mv_per_rev_s_a>: difference -0.102, combined se 0.0503, 2.04 sigma
derived: <s3:v_b_product_form/b_mv_per_rev_s_a> less <spd:brush_forms/product_ladder_runups/b_mv_per_rev_s_a>: difference -0.0962, combined se 0.0204, 4.72 sigma
derived: awake step per_v, <s3:awake_step_line/per_v> 1.817 (no se) less <cz:awake_step_slope_counts_per_v> 1.667: 0.14

## 4. Where two stage 4 notebooks state the same quantity

Each pair puts the two files' numbers side by side with the conditions each was measured under. A difference over the brackets is derived as a difference over the combined standard error where both carry one, or read against the one bracket there is. A pair outside its brackets is a discrepancy, listed after the pairs with its candidate causes. None is reconciled.

In [49]:
PAIRS = []                              # the pairs, in the order below


def pair(quantity, a, b, conditions, verdict, sig=""):
    """Two notebooks' statements of one quantity."""
    row = {"quantity": quantity, "a": a, "b": b, "conditions": conditions,
           "verdict": verdict, "sigma": sig}
    PAIRS.append({**row, "sources": sources(*row.values())})


def inside(f, key, x):
    """Whether x sits inside the bracket of quantity `key`."""
    v = node(f, key)
    return v["lo"] <= x <= v["hi"]


SR, VS = "rest_bias_by_current", "emf_stopped_rotor_rev_s"
pair("stopped rotor, table form, line on the current",
     f"speed: {Q('spd', SR + '/table/a')} + {Q('spd', SR + '/table/b_per_a')}"
     f", resid sd {Q('spd', SR + '/table/resid_sd')}",
     f"velocity: {Q('vel', VS + '/table/line_a')} + "
     f"{Q('vel', VS + '/table/line_b_per_a')}, resid sd "
     f"{Q('vel', VS + '/table/line_resid_sd')}; "
     f"{Q('vel', VS + '/table/n_holds')} holds, valid from "
     f"{Q('vel', VS + '/table/valid_ma')} mA",
     "speed: the load holds at rest, window means. velocity: the same "
     "holds, the estimate run from the drive's first row, valid rows only",
     "agree within brackets",
     sigma(val("spd", SR + "/table/a"), se_of("spd", SR + "/table/a"),
           val("vel", VS + "/table/line_a"),
           se_of("vel", VS + "/table/line_a"), "intercept") + "; "
     + sigma(val("spd", SR + "/table/b_per_a"),
             se_of("spd", SR + "/table/b_per_a"),
             val("vel", VS + "/table/line_b_per_a"),
             se_of("vel", VS + "/table/line_b_per_a"), "slope"))

In [50]:
pair("stopped rotor, product form, line on the current",
     f"speed: {Q('spd', SR + '/product/a')} + "
     f"{Q('spd', SR + '/product/b_per_a')}",
     f"velocity: {Q('vel', VS + '/product/line_a')} + "
     f"{Q('vel', VS + '/product/line_b_per_a')}",
     "as the table form's line", "agree within brackets",
     sigma(val("spd", SR + "/product/a"), se_of("spd", SR + "/product/a"),
           val("vel", VS + "/product/line_a"),
           se_of("vel", VS + "/product/line_a"), "intercept"))

In [51]:
BANDS = (("(0, 150]", "94"), ("(150, 200]", "165"), ("(200, 250]", "246"),
         ("(250, 1000]", "356"))        # speed's bin, velocity's band, mA
HIGHER = sum(val("vel", f"{VS}/table/by_band_ma/{v}")
             > val("spd", f"{SR}/table/bins/{s}") for s, v in BANDS)
Z_BAND = []
for s, v in BANDS:
    a, b = f"{SR}/table/bins/{s}", f"{VS}/table/by_band_ma/{v}"
    Z_BAND.append(abs(val("spd", a) - val("vel", b))
                  / (se_of("spd", a) ** 2 + se_of("vel", b) ** 2) ** 0.5)
print(f"velocity higher in {HIGHER} of {len(BANDS)} bands; the largest "
      f"band difference {max(Z_BAND):.2f} combined se")
for (s, v), z in zip(BANDS, Z_BAND):
    a, b = f"{SR}/table/bins/{s}", f"{VS}/table/by_band_ma/{v}"
    pair(f"stopped rotor, table form, by current band {s} / {v} mA",
         f"speed: {Q('spd', a)}", f"velocity: {Q('vel', b)}",
         "the same holds by current band (speed's bin, velocity's band "
         "center in mA)",
         ("under " if max(Z_BAND) < SIGMA_FINDING else "not all under ")
         + f"{SIGMA_FINDING:g} sigma each; velocity reads higher in "
         f"{HIGHER} of {len(BANDS)} bands (same holds, so the se overstates "
         "the independence)",
         sigma(val("spd", a), se_of("spd", a), val("vel", b), se_of("vel", b),
               "band"))
pair("stopped rotor's shape in the current",
     f"speed: a line, slope as {Q('spd', SR + '/table/ohm')}; the same "
     f"slope with no brush term {Q('spd', SR + '/plain/b_per_a')}",
     f"velocity: convex, curvature {Q('vel', VS + '/table/curvature_per_a2')}"
     "; under 300 mA alone "
     f"{Q('vel', VS + '/table/curvature_under_300_ma_per_a2')}",
     "the same holds", "not a contradiction: the velocity notebook adds "
     "the curvature, which rests on the top rung")

velocity higher in 4 of 4 bands; the largest band difference 1.46 combined se


The load holds are read two ways. The speed and velocity notebooks read their error under current as a stopped-rotor bias of the back-EMF, with R(T) from the NTC alone. The winding-temperature notebook reads the NTC-alone error on the same holds as the winding's heat, which its one heat node removes. The pair below states both with their numbers. They are two readings of the same holds and are not reconciled here. The speed estimate re-run on those holds with the one-node R(T), an offline analysis, would split them, and section 6 lists it as a gap.

In [52]:
WB = "speed_rms_rev_s_by_current/[0.35, 0.45)"   # winding's rms by bin
pair("the load holds' error under current: a stopped-rotor reading "
     "(speed, velocity) or the winding's heating (winding-temperature)",
     "velocity by band: "
     f"{Q('vel', VS + '/table/by_band_ma/246')}, "
     f"{Q('vel', VS + '/table/by_band_ma/356')}; speed: "
     f"{Q('spd', SR + '/table/bins/(250, 1000]')}",
     "winding-temperature, the NTC alone on the load holds, anchored to the "
     "reads under 0.25 A: " + ", ".join(
         Q("wt", f"speed_bias_ntc_alone_rev_s/{b}") for b in BINS)
     + f"; one node leaves {Q('wt', WB + '/b_one')} rev/s rms at 0.35 to "
     f"0.45 A against {Q('wt', WB + '/a_ntc')} for the NTC alone",
     "the same load holds; speed and velocity with R(T) from the NTC "
     "alone, winding-temperature with the NTC alone and with one node",
     "same holds, same size, two readings; not reconciled: the speed "
     "estimator was not re-run with the one-node R(T)")
DS, DV = "duty_step_response", "emf_direct_path_share/row"
for sk, vk in (("share_rows_0_3", "r0_3"), ("share_rows_4_9", "r4_9"),
               ("share_rows_10_39", "r10_39")):
    x = val("spd", f"{DS}/{sk}")
    pair(f"duty step's direct path per row, {vk}",
         f"speed: {Q('spd', f'{DS}/{sk}')}, no bracket, n "
         f"{Q('spd', DS + '/n')} steps",
         f"velocity: {Q('vel', f'{DV}/{vk}')}, "
         f"{Q('vel', 'emf_direct_path_steps')}",
         "speed: its steps of the larger sizes; velocity: every kept step",
         "inside velocity's bracket" if inside("vel", f"{DV}/{vk}", x)
         else "DISCREPANCY: outside velocity's bracket",
         f"derived: speed less velocity {num(x - val('vel', f'{DV}/{vk}'))}"
         f", {num((x - val('vel', f'{DV}/{vk}')) / se_of('vel', f'{DV}/{vk}'))}"
         " of velocity's se")

In [53]:
EX = node("spd", DS + "/excess_by_step")    # speed's share by step size
for k, v in EX.items():
    print(f"<spd:{DS}/excess_by_step/{k}>: share {num(v['share'])}, n "
          f"{v['n']}, {num(v['rev_s']['value'])} rev/s")
EX_N = sum(v["n"] for v in EX.values())
EX_ALL = sum(v["n"] * v["share"] for v in EX.values()) / EX_N
BIG = [k for k in EX if sum(EX[j]["n"] for j in EX if j != k)
       == val("spd", DS + "/n")]       # the size speed's n leaves out
EX_N39 = sum(v["n"] for k, v in EX.items() if k not in BIG)
EX_39 = sum(v["n"] * v["share"] for k, v in EX.items()
            if k not in BIG) / EX_N39
R10 = f"{DV}/r10_39"
LIKE = (EX_ALL - val("vel", R10)) / se_of("vel", R10)
print(f"derived: speed's n {num(val('spd', DS + '/n'))} leaves out the "
      f"{', '.join(BIG)} steps; n-weighted share over the other "
      f"{EX_N39} steps {num(EX_39)}, over all {EX_N} {num(EX_ALL)}; "
      f"against velocity's {num(val('vel', R10))} +/- "
      f"{num(se_of('vel', R10))}: {num(LIKE)} of velocity's se")
print(f"steps: n {node('spd', 'steps_truth/n')}, under residual "
      f"{node('spd', 'steps_truth/under_residual')}, kept "
      f"{node('spd', 'steps_truth/kept')} <spd:steps_truth>")

<spd:duty_step_response/excess_by_step/2 %>: share 0.0535, n 21, 0.717 rev/s
<spd:duty_step_response/excess_by_step/5 %>: share 0.108, n 19, 3.408 rev/s
<spd:duty_step_response/excess_by_step/10 %>: share 0.116, n 20, 7.266 rev/s
derived: speed's n 39 leaves out the 2 % steps; n-weighted share over the other 39 steps 0.112, over all 60 0.0916; against velocity's 0.0712 +/- 0.0145: 1.414 of velocity's se
steps: n 85, under residual 71, kept 60 <spd:steps_truth>


In [54]:
VC = "valid_coverage_by_set/pot-table"  # speed's coverage, camera sitting
D_PTS = val("spd", VC + "/valid_pct") - 100 * val("vel",
                                                  "emf_valid_share_camera")
pair("valid share of driven rows, camera sitting (pot-table)",
     f"speed: {Q('spd', VC + '/valid_pct')} % of "
     f"{Q('spd', VC + '/driven_rows')} driven rows, "
     f"{Q('spd', VC + '/sweeps')} sweeps",
     f"velocity: {Q('vel', 'emf_valid_share_camera')}",
     "the same sitting under the same rule", f"differ by {num(D_PTS)} "
     "points, no brackets on either: not a finding",
     f"derived: {num(D_PTS)} points")
pair("valid rule's constants",
     f"speed: floor {Q('spd', 'valid_rule/tap_floor_ticks')}, start "
     f"{Q('spd', 'valid_rule/start_rows')}",
     "velocity: reads the speed rule (oscnb.speed), no key of its own",
     "one rule", "same by construction")
RATIO = val("vel", "rest_sigma_row_counts") / val("pos",
                                                  "pot_noise_row_counts")
pair("pot's noise per row at rest",
     f"position: {Q('pos', 'pot_noise_row_counts')}",
     f"velocity: {Q('vel', 'rest_sigma_row_counts')}",
     "the same baselines with the same cut at their start; position: the "
     "sd over the rest windows, bounds the three sources; velocity: the "
     "median of each baseline's sd",
     "inside position's bracket" if inside(
         "pos", "pot_noise_row_counts", val("vel", "rest_sigma_row_counts"))
     else "DISCREPANCY: outside position's bracket, on the same baselines",
     f"derived: ratio {num(RATIO)}; stage 3's holds under torque "
     f"{Q('s3', 'hold_pos_sd_counts')}")
B16 = val("pos", "filter_boxcar16_noise_counts") - val(
    "vel", "rest_sigma16_counts")       # position's value less velocity's
pair("pot's noise at rest through a boxcar of 16",
     f"position: {Q('pos', 'filter_boxcar16_noise_counts')}",
     f"velocity: {Q('vel', 'rest_sigma16_counts')}", "the same baselines",
     f"agree (velocity's median inside its own range, {num(B16)} under "
     "position's value, which is its low bound)",
     f"derived: {num(B16)} counts")

In [55]:
pair("pot's 16-tick pattern",
     f"position: period {Q('pos', 'tick_pattern_period_ticks')}, "
     f"{Q('pos', 'tick_pattern_pp_counts')}",
     "velocity: slopes over a multiple of "
     f"{Q('vel', 'pot_slope_two_point_multiple_of_rows')}",
     "the rest windows", "same")
TZ = "tracked_zero_cost_per_sweep"
D_TZ = (val("spd", TZ + "/design/warm/mean")      # derived: the design
        - val("spd", TZ + "/tracked/warm/mean"))  # less stage 1's tracker
OFF = val("cz", "drive_feed_offset_counts")       # counts
CNT = node("spd", "rev_s_per_count")              # rev/s per count
pair("the tracked zero's cost in the speed",
     f"speed, warm sweeps: stage 1's tracker "
     f"{Q('spd', TZ + '/tracked/warm/mean')}, the design "
     f"{Q('spd', TZ + '/design/warm/mean')}, "
     f"{Q('spd', TZ + '/tracked/warm/sweeps')} sweeps of every rail; a count"
     f" {Q('spd', 'rev_s_per_count')}",
     "velocity, camera crawl at 5 V: the design's zero shifts the crawl's "
     f"bias by {Q('vel', 'emf_bias_crawl_design_zero_shift_rev_s')}",
     "speed: every rail's warm sweeps; velocity: the camera crawl",
     "agree within velocity's range over tiles",
     f"derived: speed's design less tracked {num(D_TZ)} rev/s; "
     f"current-zero's drive-feed offset {Q('cz', 'drive_feed_offset_counts')}"
     f" x the count's worth = {num(OFF * CNT['value'])} to "
     f"{num(OFF * CNT['hi'])} rev/s")
CS = "tracked_zero_cold_start_derived"  # speed's copy of the zero replay
pair("the zero's warm error, stage 1's tracker and the design",
     f"current-zero: {Q('cz', 'warm_error_counts_rms')}, design "
     f"{Q('cz', 'design_warm_rms_counts')}",
     "speed (derived there from the zero replay): "
     f"{Q('spd', CS + '/stage 1, warm/rms_counts')} over "
     f"{Q('spd', CS + '/stage 1, warm/runs')} runs, design "
     f"{Q('spd', CS + '/design, warm/rms_counts')}",
     "the same replay", "same replay; the design's warm rms differs in the "
     f"third digit ({num(val('cz', 'design_warm_rms_counts'))} against "
     f"{num(val('spd', CS + '/design, warm/rms_counts'))}), inside "
     "current-zero's bracket")

In [56]:
DF = "valid_rule/dropout_floor_a"       # speed's dropout floor, 5 sd
pair("the current sample's noise per tick at rest",
     "current-zero, trough column: "
     f"{Q('cz', 'trough_sd_per_tick_counts/narrow_drive')}",
     "speed, crest column on the rest windows: "
     f"{csvcell('spd', 'noise', 'crest_sd_counts', kind='rest', band='rest')}"
     f" counts; the dropout floor {Q('spd', DF)}",
     "current-zero: the trough under a narrow drive; speed: the crest at "
     "rest", "agree",
     "derived: dropout floor / 5 x counts per A = "
     f"{num(val('spd', DF) / 5 * val('spd', 'constants_used/counts_per_a'))}"
     " counts (the floor's unit names its 5 sd)")
TP = "tick_pattern_rest"                # speed's crest pattern at rest
pair("the current channel's tick pattern",
     "current-zero, trough folded on 16 ticks: "
     f"{Q('cz', 'trough_fold16_pp_counts')}",
     "speed, crest at rest under a narrow drive: "
     f"{Q('spd', TP + '/crest_pp_counts')}, period {Q('spd', TP + '/period')}"
     f" (share by modulus 8 {Q('spd', TP + '/share_by_modulus/8')}, 16 "
     f"{Q('spd', TP + '/share_by_modulus/16')})",
     "different columns (trough against crest) and states",
     "not the same quantity: the period is "
     f"{num(val('spd', TP + '/period'))} in speed's key against "
     "current-zero's fold on 16 ticks",
     "derived: " + num(val("spd", TP + "/crest_pp_counts")
                       - val("cz", "trough_fold16_pp_counts")) + " counts")
HS = node("spd", "brush_heldout_scores")


def held(form, zero, band="0-30", set_="holds (free, low)",
         fit="ladder alone"):
    """One held-out window score of the speed file, and its text."""
    i = find("spd", "brush_heldout_scores", form=form, fit=fit, set=set_,
             band=band, zero=zero, level="window")
    r = HS[i]
    return r, (f"{num(r['bias'])} +/- {num(r['se'])} rev/s (n {r['n']} "
               f"windows, {r['sweeps']} sweeps, {num(r['rev_s'])} rev/s, "
               f"{num(1000 * r['i_a'])} mA) <spd:brush_heldout_scores/{i}>")

In [57]:
CRAWL_Z = {}                            # tach to tach, tracked zero
for form in ("table", "product"):
    rt, st = held(form, "tracked")
    _, s_tail = held(form, "tail")
    k = f"emf_bias_crawl_tach_rev_s/{form}"
    CRAWL_Z[form] = abs(rt["bias"] - val("vel", k)) / (
        rt["se"] ** 2 + se_of("vel", k) ** 2) ** 0.5
    pair(f"the crawl's back-EMF bias, {form} form",
         f"speed, free low holds 0-30 rev/s, tach truth, window means: "
         f"tracked zero {st}; coast-tail zero {s_tail}",
         "velocity, camera crawl at 5 V, valid rows, tracked zero: against "
         f"the tach {Q('vel', k)}, against the tag "
         f"{Q('vel', f'emf_bias_crawl_rev_s/{form}')}; stage 3's truths "
         f"differ by {Q('vel', 'tach_minus_tag_rev_s/slow')}",
         "speed: the free low holds on every rail; velocity: the camera "
         "crawl at 5 V",
         ("DISCREPANCY: " if CRAWL_Z[form] >= SIGMA_FINDING else "agree: ")
         + f"{CRAWL_Z[form]:.2f} sigma tach to tach",
         sigma(rt["bias"], rt["se"], val("vel", k), se_of("vel", k),
               "tach to tach, tracked zero"))
CL = "emf_crawl_against_holds_and_lifts_line_rev_s"
pair("the crawl's bias against the line through the stopped holds and the "
     "lifts (velocity's own check)",
     "velocity, the line read at the crawl's current: table "
     f"{Q('vel', CL + '/table/line_at_crawl')}, product "
     f"{Q('vel', CL + '/product/line_at_crawl')}",
     f"velocity, the crawl measured: table "
     f"{Q('vel', CL + '/table/crawl_measured')}, product "
     f"{Q('vel', CL + '/product/crawl_measured')}",
     "one notebook, the line against the crawl",
     ("under " if max(val("vel", CL + f"/{f}/gap_sigma") for f in (
         "table", "product")) < SIGMA_FINDING else "not under ")
     + f"{SIGMA_FINDING:g} sigma: table {Q('vel', CL + '/table/gap_sigma')},"
     f" product {Q('vel', CL + '/product/gap_sigma')} sigma")
for form in ("table", "product"):
    _, st = held(form, "tracked", band="30-100")
    pair(f"free low holds 30-100 rev/s, {form} form (speed only, context)",
         f"speed: tracked zero {st}",
         "velocity: no free-shaft absolute bias key at this band",
         "speed's free low holds", "context")

In [58]:
DIR = "emf_bias_by_direction_rev_s"     # velocity's crawl by direction
pair("the crawl's bias by direction",
     f"velocity, camera crawl: table {Q('vel', DIR + '/table/+')}, "
     f"{Q('vel', DIR + '/table/-')}; product {Q('vel', DIR + '/product/+')},"
     f" {Q('vel', DIR + '/product/-')}",
     "velocity, free shaft against the tach, + less -: table "
     f"{Q('vel', 'emf_direction_split_rev_s/table/under_45')}, product "
     f"{Q('vel', 'emf_direction_split_rev_s/product/under_45')}",
     "one notebook, the camera crawl and the free shaft",
     "same notebook, two sets; consistent in sign and size",
     "derived: camera + less -: table "
     f"{num(val('vel', DIR + '/table/+') - val('vel', DIR + '/table/-'))}, "
     "product "
     f"{num(val('vel', DIR + '/product/+') - val('vel', DIR + '/product/-'))}"
     " rev/s")
NZ = "emf_noise_boxcar_rev_s/crawl valid"
pair("back-EMF noise per row at a crawl, table form",
     "speed: holds 0-30 "
     f"{csvcell('spd', 'noise', 'table_sd', kind='hold', band='0-30')} ["
     f"{csvcell('spd', 'noise', 'table_sd_lo', kind='hold', band='0-30')}, "
     f"{csvcell('spd', 'noise', 'table_sd_hi', kind='hold', band='0-30')}],"
     " holds 30-100 "
     f"{csvcell('spd', 'noise', 'table_sd', kind='hold', band='30-100')}",
     f"velocity: crawl valid {Q('vel', 'emf_noise_row_rev_s/crawl valid')},"
     " steps and reversal "
     f"{Q('vel', 'emf_noise_row_rev_s/steps+reversal slow valid')}",
     "speed: the free low holds; velocity: the camera crawl's valid rows",
     "agree (inside speed's window range)")
for n in (32, 100):
    pair(f"back-EMF noise through a boxcar of {n} at a crawl, table form",
         "speed: " + ", ".join(
             f"holds {b} " + csvcell("spd", "filters", "sd_table",
                                     kind="hold", band=b, filter=f"box {n}")
             for b in ("0-30", "30-100")),
         f"velocity: {Q('vel', f'{NZ} w_tab boxcar {n}')}",
         "speed: the free low holds; velocity: the camera crawl's tiles",
         "agree (inside velocity's range over tiles)")

In [59]:
CM = "emf_commutation_locked_share_crawl"
CM_Z = {b: (csvval("spd", "noise", "table_comm_share", kind="hold", band=b)
            - val("vel", CM)) / se_of("vel", CM) for b in ("0-30", "30-100")}
pair("share of the per-row variance at the commutation rate, crawl",
     "speed: " + ", ".join(
         f"holds {b} " + csvcell("spd", "noise", "table_comm_share",
                                 kind="hold", band=b)
         for b in CM_Z) + ", no bracket",
     f"velocity, split-half covariance on the commutation phase: "
     f"{Q('vel', CM)}",
     "speed: the free low holds by band; velocity: the camera crawl",
     "speed's values sit above velocity's bracket: " + ", ".join(
         f"{b} rev/s {'over' if z >= SIGMA_FINDING else 'under'} "
         f"{SIGMA_FINDING:g} se" for b, z in CM_Z.items())
     + "; the faster band runs above velocity's crawl; different "
     "constructions",
     "derived: " + " and ".join(num(z) for z in CM_Z.values())
     + " of velocity's se")
PT = "product_over_table_noise_by_band/box 16"
pair("product over table noise",
     f"speed: {Q('spd', PT + '/hold 22')} at a crawl, "
     f"{Q('spd', PT + '/303')} at 303 rev/s",
     f"velocity: crawl per row product {Q('vel', NZ + ' w_prod boxcar 1')} "
     f"against table {Q('vel', NZ + ' w_tab boxcar 1')}",
     "speed: by band; velocity: the camera crawl",
     "agree: the same at a crawl")

In [60]:
pair("the carries",
     f"position: codes {Q('pos', 'carry_codes')}, missing "
     f"{Q('pos', 'carry_missing_codes')}, blind "
     f"{Q('pos', 'carry_blind_span_deg')} deg",
     "velocity: dwell at a crawl " + ", ".join(
         Q("vel", f"carry_dwell_ms_crawl/{c}") for c in node(
             "vel", "carry_dwell_ms_crawl"))
     + f"; mask half-width {Q('vel', 'carry_mask_half_counts')}",
     "velocity reads position's codes", "same codes and run (velocity "
     "reads position's)")
pair("holding the pot through a carry's band",
     f"position: {Q('pos', 'carry_hold_near_rms_deg')} against "
     f"{Q('pos', 'carry_near_rms_deg')} with no hold; added delay "
     f"{Q('pos', 'carry_hold_added_delay_ms')}",
     "velocity, two-point at 25.6 ms: hold12 "
     f"{Q('vel', NEAR + '/hold12/near/25.6 ms')} against "
     f"{Q('vel', NEAR + '/none/near/25.6 ms')} with no mask; longest dwell "
     f"on the wide code {Q('vel', 'carry_dwell_max_ms_crawl')}",
     "position: the angle near a carry; velocity: the two-point slope",
     "agree in the verdict (never hold); the two delays are different "
     "quantities (the band's dwell against the wide code's)")

In [61]:
TAU_APART = abs(val("spd", "steps_truth/tau_ms_range")
                / val("vel", "mechanical_tau_ms") - 1)
pair("mechanical time constant of the kept duty steps",
     f"speed: {Q('spd', 'steps_truth/tau_ms_range')}, kept "
     f"{Q('spd', 'steps_truth/kept')}",
     f"velocity: {Q('vel', 'mechanical_tau_ms')}", "the same kept steps",
     "same steps, the two files' medians apart by "
     f"{TAU_APART:.0e} of the value")
KS = "count_kick_start_145ms/count_pot"  # the count less pot over a kick
pair("the pot against the shaft in motion (related, not the same quantity)",
     "position, pot less tag by direction on steady crawl holds: bound "
     f"{Q('pos', 'direction_bound_moving_deg')}, clip means "
     f"{Q('pos', 'direction_offset_clip_min_deg')} to "
     f"{Q('pos', 'direction_offset_clip_max_deg')}",
     f"velocity, commutation count less pot over a kick's start: "
     f"{Q('vel', KS)}; whole pulse pot less tag "
     f"{Q('vel', 'whole_pulse_deg/pot_tag')}; steady holds "
     f"{Q('vel', 'count_steady_hold_100ms/count_pot')}",
     "the count runs through the gear to the motor, the tag sits on the "
     "horn", f"the kick's {num(val('vel', KS))} deg reaches the position "
     f"bound's {num(val('pos', 'direction_bound_moving_deg'))} deg, and "
     "its bracket passes it")
pair("R at the reference temperature",
     f"winding-temperature: {Q('wt', 'ref_r0_ohm')} at "
     f"{Q('wt', 'ref_t0_c')}; locked first row "
     f"{Q('wt', 'locked_first_row_r_ohm')}",
     f"speed: {Q('spd', 'constants_used/r_ohm')} at "
     f"{Q('spd', 'constants_used/t0_c')}; stage 3 "
     f"{Q('s3', 'constants_used/r_c')}", "one constant through the chain",
     "identical (one constant through the chain); winding-temperature "
     f"holds it open: {node('wt', 'stage56_table/3/note')} "
     "<wt:stage56_table/3/note>")
PAIRS_DF = pd.DataFrame(PAIRS)
print(len(PAIRS), "pairs;", int(PAIRS_DF["verdict"].str.startswith(
    "DISCREPANCY").sum()), "outside their brackets")

36 pairs; 3 outside their brackets


In [62]:
show(PAIRS, ["quantity", "a", "b", "conditions", "verdict", "sigma"],
     ["quantity", "A", "B", "conditions", "verdict", "derived"])

| quantity | A | B | conditions | verdict | derived |
|---|---|---|---|---|---|
| stopped rotor, table form, line on the current | speed: -4.663 [-5.633, -3.694] rev/s, +/- se `spd:rest_bias_by_current/table/a` + 43.03 [38.90, 47.16] rev/s per A, +/- se `spd:rest_bias_by_current/table/b_per_a`, resid sd 3.500 `spd:rest_bias_by_current/table/resid_sd` | velocity: -4.543 [-5.727, -3.358] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/line_a` + 42.67 [37.39, 47.95] rev/s per A, +/- se `vel:emf_stopped_rotor_rev_s/table/line_b_per_a`, resid sd 3.627 rev/s `vel:emf_stopped_rotor_rev_s/table/line_resid_sd`; 67 `vel:emf_stopped_rotor_rev_s/table/n_holds` holds, valid from [69.72, 483.7] `vel:emf_stopped_rotor_rev_s/table/valid_ma` mA | speed: the load holds at rest, window means. velocity: the same holds, the estimate run from the drive's first row, valid rows only | agree within brackets | derived: intercept: difference -0.12, combined se 1.531, 0.08 sigma; derived: slope: difference 0.358, combined se 6.701, 0.05 sigma |
| stopped rotor, product form, line on the current | speed: -1.885 [-2.816, -0.954] rev/s, +/- se `spd:rest_bias_by_current/product/a` + 38.82 [34.85, 42.78] rev/s per A, +/- se `spd:rest_bias_by_current/product/b_per_a` | velocity: -2.483 [-3.642, -1.325] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/product/line_a` + 38.67 [33.65, 43.69] rev/s per A, +/- se `vel:emf_stopped_rotor_rev_s/product/line_b_per_a` | as the table form's line | agree within brackets | derived: intercept: difference 0.598, combined se 1.486, 0.40 sigma |
| stopped rotor, table form, by current band (0, 150] / 94 mA | speed: 1.089 [0.698, 1.480] rev/s, +/- se `spd:rest_bias_by_current/table/bins/(0, 150]` | velocity: 1.694 [1.556, 1.831] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/94` | the same holds by current band (speed's bin, velocity's band center in mA) | under 2 sigma each; velocity reads higher in 4 of 4 bands (same holds, so the se overstates the independence) | derived: band: difference -0.605, combined se 0.414, 1.46 sigma |
| stopped rotor, table form, by current band (150, 200] / 165 mA | speed: 1.410 [-0.104, 2.924] rev/s, +/- se `spd:rest_bias_by_current/table/bins/(150, 200]` | velocity: 1.582 [0.337, 2.827] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/165` | the same holds by current band (speed's bin, velocity's band center in mA) | under 2 sigma each; velocity reads higher in 4 of 4 bands (same holds, so the se overstates the independence) | derived: band: difference -0.172, combined se 1.960, 0.09 sigma |
| stopped rotor, table form, by current band (200, 250] / 246 mA | speed: 4.153 [3.860, 4.446] rev/s, +/- se `spd:rest_bias_by_current/table/bins/(200, 250]` | velocity: 4.668 [4.411, 4.925] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/246` | the same holds by current band (speed's bin, velocity's band center in mA) | under 2 sigma each; velocity reads higher in 4 of 4 bands (same holds, so the se overstates the independence) | derived: band: difference -0.515, combined se 0.39, 1.32 sigma |
| stopped rotor, table form, by current band (250, 1000] / 356 mA | speed: 9.569 [8.398, 10.74] rev/s, +/- se `spd:rest_bias_by_current/table/bins/(250, 1000]` | velocity: 11.40 [9.841, 12.97] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/356` | the same holds by current band (speed's bin, velocity's band center in mA) | under 2 sigma each; velocity reads higher in 4 of 4 bands (same holds, so the se overstates the independence) | derived: band: difference -1.835, combined se 1.953, 0.94 sigma |
| stopped rotor's shape in the current | speed: a line, slope as 0.313 ohm `spd:rest_bias_by_current/table/ohm`; the same slope with no brush term 43.32 [39.17, 47.46] rev/s per A, +/- se `spd:rest_bias_by_current/plain/b_per_a` | velocity: convex, curvature 172.1 [146.5, 197.7] rev/s per A^2, +/- se `vel:emf_stopped_rotor_rev_s/table/curvature_per_a2`; under 300 mA alone 179.5 [72.25, 286.8] rev/s per A^2, +/- se, 56 holds under 300 mA `vel:emf_stopped_rotor_rev_s/table/curvature_under_300_ma_per_a2` | the same holds | not a contradiction: the velocity notebook adds the curvature, which rests on the top rung |  |
| the load holds' error under current: a stopped-rotor reading (speed, velocity) or the winding's heating (winding-temperature) | velocity by band: 4.668 [4.411, 4.925] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/246`, 11.40 [9.841, 12.97] rev/s, +/- se `vel:emf_stopped_rotor_rev_s/table/by_band_ma/356`; speed: 9.569 [8.398, 10.74] rev/s, +/- se `spd:rest_bias_by_current/table/bins/(250, 1000]` | winding-temperature, the NTC alone on the load holds, anchored to the reads under 0.25 A: 3.344 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.25, 0.35)`, 6.668 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.35, 0.45)`, 10.72 rev/s `wt:speed_bias_ntc_alone_rev_s/[0.45, 0.6)`; one node leaves 1.845 `wt:speed_rms_rev_s_by_current/[0.35, 0.45)/b_one` rev/s rms at 0.35 to 0.45 A against 7.024 `wt:speed_rms_rev_s_by_current/[0.35, 0.45)/a_ntc` for the NTC alone | the same load holds; speed and velocity with R(T) from the NTC alone, winding-temperature with the NTC alone and with one node | same holds, same size, two readings; not reconciled: the speed estimator was not re-run with the one-node R(T) |  |
| duty step's direct path per row, r0_3 | speed: 0.751 of the volts' step / Ke `spd:duty_step_response/share_rows_0_3`, no bracket, n 39 `spd:duty_step_response/n` steps | velocity: 0.788 [0.74, 0.835] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/row/r0_3`, 60 kept duty steps of steps/capture-1 `vel:emf_direct_path_steps` | speed: its steps of the larger sizes; velocity: every kept step | inside velocity's bracket | derived: speed less velocity -0.0366, -0.774 of velocity's se |
| duty step's direct path per row, r4_9 | speed: 0.299 of the volts' step / Ke `spd:duty_step_response/share_rows_4_9`, no bracket, n 39 `spd:duty_step_response/n` steps | velocity: 0.265 [0.215, 0.314] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/row/r4_9`, 60 kept duty steps of steps/capture-1 `vel:emf_direct_path_steps` | speed: its steps of the larger sizes; velocity: every kept step | inside velocity's bracket | derived: speed less velocity 0.0346, 0.701 of velocity's se |
| duty step's direct path per row, r10_39 | speed: 0.114 of the volts' step / Ke `spd:duty_step_response/share_rows_10_39`, no bracket, n 39 `spd:duty_step_response/n` steps | velocity: 0.0712 [0.0567, 0.0856] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/row/r10_39`, 60 kept duty steps of steps/capture-1 `vel:emf_direct_path_steps` | speed: its steps of the larger sizes; velocity: every kept step | DISCREPANCY: outside velocity's bracket | derived: speed less velocity 0.0425, 2.934 of velocity's se |
| valid share of driven rows, camera sitting (pot-table) | speed: 84.32 `spd:valid_coverage_by_set/pot-table/valid_pct` % of 1149774 `spd:valid_coverage_by_set/pot-table/driven_rows` driven rows, 21 `spd:valid_coverage_by_set/pot-table/sweeps` sweeps | velocity: 0.843 of driven rows `vel:emf_valid_share_camera` | the same sitting under the same rule | differ by 0.042 points, no brackets on either: not a finding | derived: 0.042 points |
| valid rule's constants | speed: floor 84 [72, 96] ticks; bounds: the failing rung below, the first rung with 10 runs or more `spd:valid_rule/tap_floor_ticks`, start 16 rows, 4 L / R rounded up `spd:valid_rule/start_rows` | velocity: reads the speed rule (oscnb.speed), no key of its own | one rule | same by construction |  |
| pot's noise per row at rest | position: 1.031 [1.027, 1.160] counts `pos:pot_noise_row_counts` | velocity: 0.962 counts `vel:rest_sigma_row_counts` | the same baselines with the same cut at their start; position: the sd over the rest windows, bounds the three sources; velocity: the median of each baseline's sd | DISCREPANCY: outside position's bracket, on the same baselines | derived: ratio 0.933; stage 3's holds under torque 0.871 [0.521, 1.524] counts `s3:hold_pos_sd_counts` |
| pot's noise at rest through a boxcar of 16 | position: 0.139 [0.139, 0.161] counts `pos:filter_boxcar16_noise_counts` | velocity: 0.137 [0, 0.165] counts, median [range] over the baselines `vel:rest_sigma16_counts` | the same baselines | agree (velocity's median inside its own range, 0.00204 under position's value, which is its low bound) | derived: 0.00204 counts |
| pot's 16-tick pattern | position: period 16 ticks `pos:tick_pattern_period_ticks`, 3.014 [2.986, 3.025] counts `pos:tick_pattern_pp_counts` | velocity: slopes over a multiple of 16 rows, the board's pattern `vel:pot_slope_two_point_multiple_of_rows` | the rest windows | same |  |
| the tracked zero's cost in the speed | speed, warm sweeps: stage 1's tracker 0.302 [0.283, 0.321] rev/s, +/- se `spd:tracked_zero_cost_per_sweep/tracked/warm/mean`, the design 0.0486 [0.032, 0.0652] rev/s, +/- se `spd:tracked_zero_cost_per_sweep/design/warm/mean`, 134 sweeps `spd:tracked_zero_cost_per_sweep/tracked/warm/sweeps` sweeps of every rail; a count 0.471 [0.471, 0.517] rev/s per count of zero `spd:rev_s_per_count` | velocity, camera crawl at 5 V: the design's zero shifts the crawl's bias by -0.344 [-0.36, -0.25] rev/s, bounds the tiles `vel:emf_bias_crawl_design_zero_shift_rev_s` | speed: every rail's warm sweeps; velocity: the camera crawl | agree within velocity's range over tiles | derived: speed's design less tracked -0.254 rev/s; current-zero's drive-feed offset 0.667 [0.657, 0.698] counts `cz:drive_feed_offset_counts` x the count's worth = 0.314 to 0.345 rev/s |
| the zero's warm error, stage 1's tracker and the design | current-zero: 0.73 [0.684, 0.778] counts `cz:warm_error_counts_rms`, design 0.411 [0.373, 0.447] counts `cz:design_warm_rms_counts` | speed (derived there from the zero replay): 0.73 counts `spd:tracked_zero_cold_start_derived/stage 1, warm/rms_counts` over 911 runs `spd:tracked_zero_cold_start_derived/stage 1, warm/runs` runs, design 0.408 counts `spd:tracked_zero_cold_start_derived/design, warm/rms_counts` | the same replay | same replay; the design's warm rms differs in the third digit (0.411 against 0.408), inside current-zero's bracket |  |
| the current sample's noise per tick at rest | current-zero, trough column: 1.161 [1.044, 1.436] counts `cz:trough_sd_per_tick_counts/narrow_drive` | speed, crest column on the rest windows: 1.132 `spd:.noise.csv[kind=rest,band=rest].crest_sd_counts` counts; the dropout floor 0.00499 A, 5 sd of the rest noise `spd:valid_rule/dropout_floor_a` | current-zero: the trough under a narrow drive; speed: the crest at rest | agree | derived: dropout floor / 5 x counts per A = 1.132 counts (the floor's unit names its 5 sd) |
| the current channel's tick pattern | current-zero, trough folded on 16 ticks: 0.999 [0.967, 1.042] counts `cz:trough_fold16_pp_counts` | speed, crest at rest under a narrow drive: 1.272 counts `spd:tick_pattern_rest/crest_pp_counts`, period 8 ticks `spd:tick_pattern_rest/period` (share by modulus 8 0.0765 of the variance `spd:tick_pattern_rest/share_by_modulus/8`, 16 0.0895 of the variance `spd:tick_pattern_rest/share_by_modulus/16`) | different columns (trough against crest) and states | not the same quantity: the period is 8 in speed's key against current-zero's fold on 16 ticks | derived: 0.274 counts |
| the crawl's back-EMF bias, table form | speed, free low holds 0-30 rev/s, tach truth, window means: tracked zero -0.579 +/- 0.443 rev/s (n 18 windows, 11 sweeps, 23.43 rev/s, 38.53 mA) `spd:brush_heldout_scores/157`; coast-tail zero -0.882 +/- 0.473 rev/s (n 18 windows, 11 sweeps, 23.43 rev/s, 39.03 mA) `spd:brush_heldout_scores/29` | velocity, camera crawl at 5 V, valid rows, tracked zero: against the tach 0.661 [0.571, 0.752] rev/s, +/- se over segments `vel:emf_bias_crawl_tach_rev_s/table`, against the tag 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table`; stage 3's truths differ by -0.122 [-0.187, -0.0568] rev/s, +/- se over segments `vel:tach_minus_tag_rev_s/slow` | speed: the free low holds on every rail; velocity: the camera crawl at 5 V | DISCREPANCY: 2.75 sigma tach to tach | derived: tach to tach, tracked zero: difference -1.240, combined se 0.452, 2.75 sigma |
| the crawl's back-EMF bias, product form | speed, free low holds 0-30 rev/s, tach truth, window means: tracked zero 1.192 +/- 0.447 rev/s (n 18 windows, 11 sweeps, 23.43 rev/s, 38.53 mA) `spd:brush_heldout_scores/155`; coast-tail zero 0.889 +/- 0.466 rev/s (n 18 windows, 11 sweeps, 23.43 rev/s, 39.03 mA) `spd:brush_heldout_scores/27` | velocity, camera crawl at 5 V, valid rows, tracked zero: against the tach 1.504 [1.345, 1.662] rev/s, +/- se over segments `vel:emf_bias_crawl_tach_rev_s/product`, against the tag 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product`; stage 3's truths differ by -0.122 [-0.187, -0.0568] rev/s, +/- se over segments `vel:tach_minus_tag_rev_s/slow` | speed: the free low holds on every rail; velocity: the camera crawl at 5 V | agree: 0.66 sigma tach to tach | derived: tach to tach, tracked zero: difference -0.311, combined se 0.475, 0.66 sigma |
| the crawl's bias against the line through the stopped holds and the lifts (velocity's own check) | velocity, the line read at the crawl's current: table -0.715 [-1.543, 0.112] rev/s, +/- se `vel:emf_crawl_against_holds_and_lifts_line_rev_s/table/line_at_crawl`, product 0.199 [-0.782, 1.180] rev/s, +/- se `vel:emf_crawl_against_holds_and_lifts_line_rev_s/product/line_at_crawl` | velocity, the crawl measured: table 0.54 [0.437, 0.643] rev/s, +/- se over sweeps `vel:emf_crawl_against_holds_and_lifts_line_rev_s/table/crawl_measured`, product 1.382 [1.198, 1.566] rev/s, +/- se over sweeps `vel:emf_crawl_against_holds_and_lifts_line_rev_s/product/crawl_measured` | one notebook, the line against the crawl | under 2 sigma: table 1.505 `vel:emf_crawl_against_holds_and_lifts_line_rev_s/table/gap_sigma`, product 1.186 `vel:emf_crawl_against_holds_and_lifts_line_rev_s/product/gap_sigma` sigma |  |
| free low holds 30-100 rev/s, table form (speed only, context) | speed: tracked zero -0.156 +/- 0.208 rev/s (n 57 windows, 30 sweeps, 35.56 rev/s, 41.53 mA) `spd:brush_heldout_scores/163` | velocity: no free-shaft absolute bias key at this band | speed's free low holds | context |  |
| free low holds 30-100 rev/s, product form (speed only, context) | speed: tracked zero 1.121 +/- 0.223 rev/s (n 57 windows, 30 sweeps, 35.56 rev/s, 41.53 mA) `spd:brush_heldout_scores/161` | velocity: no free-shaft absolute bias key at this band | speed's free low holds | context |  |
| the crawl's bias by direction | velocity, camera crawl: table 0.891 [0.773, 1.009] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/table/+`, 0.109 [0.00347, 0.214] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/table/-`; product 0.219 [0.0885, 0.349] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/product/+`, 2.098 [1.984, 2.212] rev/s, mean of segment means, +/- se `vel:emf_bias_by_direction_rev_s/product/-` | velocity, free shaft against the tach, + less -: table 0.814 [0.746, 0.883] rev/s, +/- se `vel:emf_direction_split_rev_s/table/under_45`, product -1.828 [-1.896, -1.759] rev/s, +/- se `vel:emf_direction_split_rev_s/product/under_45` | one notebook, the camera crawl and the free shaft | same notebook, two sets; consistent in sign and size | derived: camera + less -: table 0.782, product -1.879 rev/s |
| back-EMF noise per row at a crawl, table form | speed: holds 0-30 3.841 `spd:.noise.csv[kind=hold,band=0-30].table_sd` [2.475 `spd:.noise.csv[kind=hold,band=0-30].table_sd_lo`, 5.441 `spd:.noise.csv[kind=hold,band=0-30].table_sd_hi`], holds 30-100 4.861 `spd:.noise.csv[kind=hold,band=30-100].table_sd` | velocity: crawl valid 3.896 rev/s per row, table `vel:emf_noise_row_rev_s/crawl valid`, steps and reversal 4.020 rev/s per row, table `vel:emf_noise_row_rev_s/steps+reversal slow valid` | speed: the free low holds; velocity: the camera crawl's valid rows | agree (inside speed's window range) |  |
| back-EMF noise through a boxcar of 32 at a crawl, table form | speed: holds 0-30 3.690 `spd:.filters.csv[kind=hold,band=0-30,filter=box 32].sd_table`, holds 30-100 4.101 `spd:.filters.csv[kind=hold,band=30-100,filter=box 32].sd_table` | velocity: 3.426 [2.266, 5.916] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_tab boxcar 32` | speed: the free low holds; velocity: the camera crawl's tiles | agree (inside velocity's range over tiles) |  |
| back-EMF noise through a boxcar of 100 at a crawl, table form | speed: holds 0-30 2.611 `spd:.filters.csv[kind=hold,band=0-30,filter=box 100].sd_table`, holds 30-100 2.345 `spd:.filters.csv[kind=hold,band=30-100,filter=box 100].sd_table` | velocity: 2.218 [0.682, 5.096] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_tab boxcar 100` | speed: the free low holds; velocity: the camera crawl's tiles | agree (inside velocity's range over tiles) |  |
| share of the per-row variance at the commutation rate, crawl | speed: holds 0-30 0.756 `spd:.noise.csv[kind=hold,band=0-30].table_comm_share`, holds 30-100 0.77 `spd:.noise.csv[kind=hold,band=30-100].table_comm_share`, no bracket | velocity, split-half covariance on the commutation phase: 0.719 [0.701, 0.738] of the per-row variance, +/- se over segments `vel:emf_commutation_locked_share_crawl` | speed: the free low holds by band; velocity: the camera crawl | speed's values sit above velocity's bracket: 0-30 rev/s under 2 se, 30-100 rev/s over 2 se; the faster band runs above velocity's crawl; different constructions | derived: 1.961 and 2.722 of velocity's se |
| product over table noise | speed: 0.996 ratio `spd:product_over_table_noise_by_band/box 16/hold 22` at a crawl, 1.205 ratio `spd:product_over_table_noise_by_band/box 16/303` at 303 rev/s | velocity: crawl per row product 3.900 [2.514, 6.561] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_prod boxcar 1` against table 3.896 [2.514, 6.536] rev/s about the tile's line, bounds tiles `vel:emf_noise_boxcar_rev_s/crawl valid w_tab boxcar 1` | speed: by band; velocity: the camera crawl | agree: the same at a crawl |  |
| the carries | position: codes [1023, 1279, 2047, 2303, 2559] `pos:carry_codes`, missing [12, 3, 11, 3, 7] `pos:carry_missing_codes`, blind [0.772, 0.139, 0.467, 0.0298, 0.279] `pos:carry_blind_span_deg` deg | velocity: dwell at a crawl 30.70 [19.15, 34.30] ms, median, bounds the crossings `vel:carry_dwell_ms_crawl/1023`, 10.35 [8.650, 11.50] ms, median, bounds the crossings `vel:carry_dwell_ms_crawl/1279`, 21 [8.850, 45.50] ms, median, bounds the crossings `vel:carry_dwell_ms_crawl/2047`, 5.650 [3.500, 11.40] ms, median, bounds the crossings `vel:carry_dwell_ms_crawl/2303`, 15.42 [10.60, 41.80] ms, median, bounds the crossings `vel:carry_dwell_ms_crawl/2559`; mask half-width 12 counts, the longest missing run `vel:carry_mask_half_counts` | velocity reads position's codes | same codes and run (velocity reads position's) |  |
| holding the pot through a carry's band | position: 0.698 deg `pos:carry_hold_near_rms_deg` against 0.201 deg `pos:carry_near_rms_deg` with no hold; added delay 29.57 [-, 108.3] ms `pos:carry_hold_added_delay_ms` | velocity, two-point at 25.6 ms: hold12 31.41 deg/s rms `vel:carry_near_rms_deg_s/hold12/near/25.6 ms` against 10.78 deg/s rms `vel:carry_near_rms_deg_s/none/near/25.6 ms` with no mask; longest dwell on the wide code 45.50 ms, the longest crossing at a crawl `vel:carry_dwell_max_ms_crawl` | position: the angle near a carry; velocity: the two-point slope | agree in the verdict (never hold); the two delays are different quantities (the band's dwell against the wide code's) |  |
| mechanical time constant of the kept duty steps | speed: 18.29 [11.94, 26.66] ms, median, 10th to 90th percentile of the kept `spd:steps_truth/tau_ms_range`, kept 60 `spd:steps_truth/kept` | velocity: 18.29 [11.94, 26.66] ms, median [10-90 %] of the kept steps `vel:mechanical_tau_ms` | the same kept steps | same steps, the two files' medians apart by 5e-09 of the value |  |
| the pot against the shaft in motion (related, not the same quantity) | position, pot less tag by direction on steady crawl holds: bound 0.5 deg `pos:direction_bound_moving_deg`, clip means -0.194 deg `pos:direction_offset_clip_min_deg` to 0.528 deg `pos:direction_offset_clip_max_deg` | velocity, commutation count less pot over a kick's start: 0.483 [0.372, 0.594] deg of horn, +/- 2 se, n 38 `vel:count_kick_start_145ms/count_pot`; whole pulse pot less tag 0.185 [0.0531, 0.317] deg of horn, +/- 2 se `vel:whole_pulse_deg/pot_tag`; steady holds -0.0258 [-0.0776, 0.026] deg of horn, +/- 2 se, n 27 `vel:count_steady_hold_100ms/count_pot` | the count runs through the gear to the motor, the tag sits on the horn | the kick's 0.483 deg reaches the position bound's 0.5 deg, and its bracket passes it |  |
| R at the reference temperature | winding-temperature: 4.177 [4.168, 4.193] ohm `wt:ref_r0_ohm` at 26.26 [26.10, 26.32] C `wt:ref_t0_c`; locked first row 4.180 [4.168, 4.192] ohm (sd 0.0121, n 6) `wt:locked_first_row_r_ohm` | speed: 4.177 ohm `spd:constants_used/r_ohm` at 26.26 C `spd:constants_used/t0_c`; stage 3 4.177 [4.168, 4.193] ohm `s3:constants_used/r_c` | one constant through the chain | identical (one constant through the chain); winding-temperature holds it open: R0 as a cold value open by 1.2 to 2.2 % `wt:stage56_table/3/note` |  |

### The discrepancies

The pairs that differ by more than their brackets, each with its candidate causes and a confidence for each. The causes are hypotheses this notebook does not test, and none is reconciled. The commutation share is listed with its faster band over the threshold and its like-for-like band under it.

In [63]:
DISC = []


def disc(quantity, a, b, sig, causes):
    """One discrepancy: the two sides, the size, the candidate causes."""
    assert all(c in ("low", "low to moderate", "moderate", "high")
               for _, c in causes)      # one confidence vocabulary
    row = {"quantity": quantity, "a": a, "b": b, "sigma": sig,
           "causes": [{"cause": k, "confidence": c} for k, c in causes]}
    DISC.append({**row, "sources": sources(quantity, a, b, sig)})


disc("pot noise per row at rest",
     f"position {Q('pos', 'pot_noise_row_counts')}",
     f"velocity {Q('vel', 'rest_sigma_row_counts')}",
     "velocity under position's low bound; ratio " + num(RATIO)
     + "; the boxcar-16 values agree",
     [("the statistic: a pooled sd against the median of per-baseline sds; "
       "the position notebook's own open discrepancy names the same effect",
       "moderate"),
      ("a different row set behind the two keys", "low"),
      ("the cut at the baselines' start applied differently", "low")])
_, ST = held("table", "tracked")
disc("crawl bias of the back-EMF, table form",
     f"speed {ST}",
     f"velocity {Q('vel', 'emf_bias_crawl_tach_rev_s/table')}",
     f"{CRAWL_Z['table']:.2f} sigma tach to tach; the product form agrees "
     f"at {CRAWL_Z['product']:.2f} sigma",
     [("different sets and rails", "moderate"),
      ("rows under the tap floor inside speed's windows, which velocity's "
       "valid flag leaves out; it would move both forms",
       "low to moderate"),
      ("direction mix; it moves the two forms opposite ways", "low"),
      ("chance among the pairs compared", "low to moderate")])

In [64]:
X39 = val("spd", DS + "/share_rows_10_39")
disc("duty step's direct path, rows 10 to 39",
     f"speed {Q('spd', DS + '/share_rows_10_39')}, n "
     f"{Q('spd', DS + '/n')} steps, no bracket",
     f"velocity {Q('vel', R10)}, {Q('vel', 'emf_direct_path_steps')}",
     f"{num((X39 - val('vel', R10)) / se_of('vel', R10))} of velocity's se "
     f"as the keys stand; speed's own by-step shares weighted over all "
     f"{EX_N} steps give {num(EX_ALL)}, {num(LIKE)} se from velocity, under "
     f"the threshold of {SIGMA_FINDING:g}",
     [("the step set: speed's n leaves out the smallest steps; this is "
       "about half the difference, and the rest is under the threshold",
       "high")])
disc("commutation share of the per-row back-EMF variance at a crawl",
     "speed " + ", ".join(
         csvcell("spd", "noise", "table_comm_share", kind="hold", band=b)
         + f" ({b} rev/s)" for b in CM_Z) + ", no bracket",
     f"velocity {Q('vel', CM)}",
     " and ".join(num(z) for z in CM_Z.values()) + " of velocity's se; the "
     "like-for-like band under the threshold, the second faster than "
     "velocity's crawl",
     [("two constructions of the share", "moderate")])
show([{**d, "causes": "; ".join(f"{c['cause']} ({c['confidence']})"
                                for c in d["causes"])} for d in DISC],
     ["quantity", "a", "b", "sigma", "causes"],
     ["quantity", "A", "B", "size", "candidate causes (confidence)"])

| quantity | A | B | size | candidate causes (confidence) |
|---|---|---|---|---|
| pot noise per row at rest | position 1.031 [1.027, 1.160] counts `pos:pot_noise_row_counts` | velocity 0.962 counts `vel:rest_sigma_row_counts` | velocity under position's low bound; ratio 0.933; the boxcar-16 values agree | the statistic: a pooled sd against the median of per-baseline sds; the position notebook's own open discrepancy names the same effect (moderate); a different row set behind the two keys (low); the cut at the baselines' start applied differently (low) |
| crawl bias of the back-EMF, table form | speed -0.579 +/- 0.443 rev/s (n 18 windows, 11 sweeps, 23.43 rev/s, 38.53 mA) `spd:brush_heldout_scores/157` | velocity 0.661 [0.571, 0.752] rev/s, +/- se over segments `vel:emf_bias_crawl_tach_rev_s/table` | 2.75 sigma tach to tach; the product form agrees at 0.66 sigma | different sets and rails (moderate); rows under the tap floor inside speed's windows, which velocity's valid flag leaves out; it would move both forms (low to moderate); direction mix; it moves the two forms opposite ways (low); chance among the pairs compared (low to moderate) |
| duty step's direct path, rows 10 to 39 | speed 0.114 of the volts' step / Ke `spd:duty_step_response/share_rows_10_39`, n 39 `spd:duty_step_response/n` steps, no bracket | velocity 0.0712 [0.0567, 0.0856] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/row/r10_39`, 60 kept duty steps of steps/capture-1 `vel:emf_direct_path_steps` | 2.934 of velocity's se as the keys stand; speed's own by-step shares weighted over all 60 steps give 0.0916, 1.414 se from velocity, under the threshold of 2 | the step set: speed's n leaves out the smallest steps; this is about half the difference, and the rest is under the threshold (high) |
| commutation share of the per-row back-EMF variance at a crawl | speed 0.756 `spd:.noise.csv[kind=hold,band=0-30].table_comm_share` (0-30 rev/s), 0.77 `spd:.noise.csv[kind=hold,band=30-100].table_comm_share` (30-100 rev/s), no bracket | velocity 0.719 [0.701, 0.738] of the per-row variance, +/- se over segments `vel:emf_commutation_locked_share_crawl` | 1.961 and 2.722 of velocity's se; the like-for-like band under the threshold, the second faster than velocity's crawl | two constructions of the share (moderate) |

## 5. The open choices

Four choices stage 4 leaves open, each with what every option costs on this data and what would decide it. None is chosen here, and no key of the result file names an option of record.

In [65]:
CHOICES = []


def choice(name, options, decided_by):
    """One open choice: its options with their costs, what decides it."""
    row = {"choice": name, "options": [{"option": o, "cost": c}
                                       for o, c in options],
           "decided_by": decided_by}
    CHOICES.append({**row, "sources": sources(
        decided_by, *(t for o in options for t in o))})

In [66]:
LB = "emf_bias_lift_by_speed_rev_s"     # velocity's lift error by speed
SK = node("spd", "table_slopes_share_of_ke/+1/share_of_ke")
choice("brush term: signed speed table against the speed x current product",
       [("table", f"free shaft {sb('ladder (third repeat)', '30-100')}; "
         f"under current {sb('run-up', '160-250')}, "
         f"{sb('run-up', '250-350')}; lifts {sb('lifts', '100-160')}; crawl "
         f"{Q('vel', 'emf_bias_crawl_rev_s/table')}; direction split "
         f"{Q('vel', 'emf_direction_split_rev_s/table/under_45')}; lift error "
         f"by speed {Q('vel', LB + '/table/0-30')} to "
         f"{Q('vel', LB + '/table/100-170')}; costs "
         f"{Q('spd', 'solve_recommendation/steps')} newton steps per row and "
         "a table per direction <spd:brush_forms/table_signed_mv>, steepest "
         f"slope {num(max(SK))} of Ke <spd:table_slopes_share_of_ke/+1/"
         "share_of_ke>"),
        ("product", "the same rows in the cells left; crawl "
         f"{Q('vel', 'emf_bias_crawl_rev_s/product')}; direction split "
         f"{Q('vel', 'emf_direction_split_rev_s/product/under_45')}; lift "
         f"error by speed {Q('vel', LB + '/product/0-30')} to "
         f"{Q('vel', LB + '/product/100-170')}; stopped rotor mean "
         f"{Q('spd', SR + '/mean/product')} against the table's "
         f"{Q('spd', SR + '/mean/table')}; closed form, three constants and "
         "a sign; noise ratio to the table "
         f"{Q('spd', PT + '/303')} at 303 rev/s; c_dir depends on the fit "
         f"set ({Q('spd', PL + '/c_dir_mv')} against "
         f"{Q('spd', PR + '/c_dir_mv')}); crossover against the pot "
         f"{Q('vel', XW + '/crawl_rms_product')} against the table's "
         f"{Q('vel', XW + '/crawl_rms')}")],
       "matched-speed holds at two currents, the tag on the hub (the speed "
       "notebook's brush-form gap, stage 3's, and the README's velocity "
       "gap)")

In [67]:
BS = "blend_seam_rev_s"                 # the blends' seams
choice("one velocity: the switch against the complementary pair",
       [("switch sw5", "crawl bias "
         f"{Q('vel', 'blend_bias_crawl_rev_s/sw5')}, stall over the floor "
         f"{Q('vel', 'blend_bias_stall_over_floor_rev_s/sw5')}, own sd "
         f"{Q('vel', 'blend_own_sd_rev_s/crawl/sw5')}, seams "
         f"{Q('vel', BS + '/sw5_kick/median')} at a start and "
         f"{Q('vel', BS + '/sw5_coast/median')} at a coast, direct path "
         f"{Q('vel', 'emf_direct_path_share/boxcar_32/r10_39')}, no state "
         "of its own; with the rest verdict (swr5) a seam of "
         f"{Q('vel', BS + '/swr5_kick/median')}"),
        ("pair cp5", "crawl bias "
         f"{Q('vel', 'blend_bias_crawl_rev_s/cp5')}, stall over the floor "
         f"{Q('vel', 'blend_bias_stall_over_floor_rev_s/cp5')}, own sd "
         f"{Q('vel', 'blend_own_sd_rev_s/crawl/cp5')}, seams "
         f"{Q('vel', BS + '/cp5_handover/median')} at the handover, "
         f"{Q('vel', BS + '/cp5_coast/median')} at a coast, "
         f"{Q('vel', BS + '/cpr5_kick/median')} at a start with rest on pot "
         "rows, direct path "
         f"{Q('vel', 'emf_direct_path_share/pair_5/r10_39')} and "
         f"{Q('vel', 'emf_direct_path_share/pair_5/r40_199')}, one one-pole "
         f"at {Q('vel', 'blend_corner_rows')} and the pot slope's delay "
         "below it; cp25 under the floor "
         f"{Q('vel', 'blend_own_sd_rev_s/under_floor/cp25')} against cp5 "
         f"{Q('vel', 'blend_own_sd_rev_s/under_floor/cp5')}")],
       "the loop design (stage 5): the back-EMF's bias at DC against the "
       "pot's delay below the corner and a step of "
       f"{Q('vel', BS + '/cpr5_kick/median')} in the feedback at a start")

In [68]:
RD, RS = "rest_declare_ms", "rest_drop_ms"   # the verdict's two times
FM, FM3 = "rest_false_motion_share", "rest_false_motion_stage3_share"
B2, S192 = REST["band"][0], REST["slope"][0]  # the two forms' keys
choice("rest verdict: the band against the slope",
       [("band", f"band of {Q('vel', 'rest_verdict/band/counts')} counts: "
         f"declare {Q('vel', f'{RD}/{B2}/coast_down/median')} (p90 "
         f"{Q('vel', f'{RD}/{B2}/coast_down/p90')}), drop "
         f"{Q('vel', f'{RS}/{B2}/median')}, false motion on coast tails "
         f"{Q('vel', f'{FM}/{B2}/rest_coast')}, on load holds "
         f"{Q('vel', f'{FM3}/{B2}/load holds')}; reads counts, no slope; a "
         "band of 1 count holds "
         f"{Q('vel', f'{FM3}/band 1 T50/stalls resistance')} of the stalls' "
         "rows as motion"),
        ("slope", f"slope of {Q('vel', 'rest_verdict/slope/rows')} rows: "
         f"declare {Q('vel', f'{RD}/{S192}/coast_down/median')} (p90 "
         f"{Q('vel', f'{RD}/{S192}/coast_down/p90')}), drop "
         f"{Q('vel', f'{RS}/{S192}/median')}, false motion on coast tails "
         f"{Q('vel', f'{FM}/{S192}/rest_coast')}, on load holds "
         f"{Q('vel', f'{FM3}/{S192}/load holds')}; keeps the slope's rows; "
         "a 992-row slope holds "
         f"{Q('vel', f'{FM}/slope 992 T75/rest_base')} of the baselines as "
         "motion")],
       f"nothing in this data over {SIGMA_FINDING:g} sigma "
       f"({node('vel', 'rest_verdict/note')} <vel:rest_verdict/note>); a "
       "driven sweep at 2 to 10 deg/s (the velocity notebook's rest-verdict "
       "gap); the veto is a separate option: drop "
       f"{Q('vel', f'{RS}/{B2} + emf/median')}, fires on "
       f"{Q('vel', 'rest_veto/fires_on_stopped')}")

In [69]:
choice("tap floor's rung",
       [(f"{num(FLOOR_Q['value'])} ticks", "floor "
         f"{Q('spd', 'valid_rule/tap_floor_ticks')}; deficit "
         f"{Q('spd', TD + '/84-84/deficit_mv')} on n "
         f"{Q('spd', TD + '/84-84/n')} runs of "
         f"{Q('spd', TD + '/84-84/sweeps')} sweeps; valid share "
         f"{Q('spd', COV + '/valid_pct')} %, under the floor "
         f"{Q('spd', COV + '/under_floor_pct')} %"),
        (f"{num(FLOOR_Q['hi'])} ticks", "deficit "
         f"{Q('spd', TD + '/96-96/deficit_mv')} on n "
         f"{Q('spd', TD + '/96-96/n')} runs; rows under 96 ticks "
         f"{Q('spd', COV + '/under_96_ticks_pct')} %; stage 3's closing "
         f"table: {Q('spd', TS3)}; stage 3's key "
         f"{Q('s3', 'tap_floor_half_ticks')} is the failing rung "
         f"({Q('spd', TD + '/72-72/deficit_mv')}, as speed "
         f"{Q('spd', TD + '/72-72/as_rev_s')} rev/s)")],
       "a hold ladder at rest from 72 to 96 ticks in steps of 4 (the speed "
       "notebook's tap-floor gap, stage 3's tap-floor discrepancy)")
OPTS = [{"choice": c["choice"], "option": o["option"], "cost": o["cost"],
         "decided_by": c["decided_by"]} for c in CHOICES
        for o in c["options"]]
show(OPTS, ["choice", "option", "cost", "decided_by"],
     ["choice", "option", "what it costs", "what decides it"])

| choice | option | what it costs | what decides it |
|---|---|---|---|
| brush term: signed speed table against the speed x current product | table | free shaft ladder (third repeat) 30-100 (94 rev/s, 73 mA): table -0.31, product 0.12 rev/s `spd:stage5_bias_by_band_form/0`; under current run-up 160-250 (218.9 rev/s, 655 mA): table 30, product -1.340 rev/s `spd:stage5_bias_by_band_form/11`, run-up 250-350 (306.2 rev/s, 512 mA): table 27.10, product -1.920 rev/s `spd:stage5_bias_by_band_form/12`; lifts lifts 100-160 (111.5 rev/s, 218 mA): table 7.420, product 3.580 rev/s `spd:stage5_bias_by_band_form/17`; crawl 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table`; direction split 0.814 [0.746, 0.883] rev/s, +/- se `vel:emf_direction_split_rev_s/table/under_45`; lift error by speed 4.033 [3.495, 4.571] rev/s, +/- se over sweeps `vel:emf_bias_lift_by_speed_rev_s/table/0-30` to 7.604 [6.395, 8.812] rev/s, +/- se over sweeps `vel:emf_bias_lift_by_speed_rev_s/table/100-170`; costs 2 `spd:solve_recommendation/steps` newton steps per row and a table per direction `spd:brush_forms/table_signed_mv`, steepest slope 0.655 of Ke `spd:table_slopes_share_of_ke/+1/share_of_ke` | matched-speed holds at two currents, the tag on the hub (the speed notebook's brush-form gap, stage 3's, and the README's velocity gap) |
| brush term: signed speed table against the speed x current product | product | the same rows in the cells left; crawl 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product`; direction split -1.828 [-1.896, -1.759] rev/s, +/- se `vel:emf_direction_split_rev_s/product/under_45`; lift error by speed 4.003 [3.668, 4.339] rev/s, +/- se over sweeps `vel:emf_bias_lift_by_speed_rev_s/product/0-30` to 3.329 [2.408, 4.250] rev/s, +/- se over sweeps `vel:emf_bias_lift_by_speed_rev_s/product/100-170`; stopped rotor mean 6.314 [5.686, 6.943] rev/s, +/- se `spd:rest_bias_by_current/mean/product` against the table's 4.426 [3.747, 5.105] rev/s, +/- se `spd:rest_bias_by_current/mean/table`; closed form, three constants and a sign; noise ratio to the table 1.205 ratio `spd:product_over_table_noise_by_band/box 16/303` at 303 rev/s; c_dir depends on the fit set (12.39 [9.149, 15.63] mV, +/- se `spd:brush_forms/product_ladder_alone/c_dir_mv` against 36.02 [34.22, 37.82] mV, +/- se `spd:brush_forms/product_ladder_runups/c_dir_mv`); crossover against the pot 50.81 [38.96, 64.82] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms_product` against the table's 103.0 [66.15, 115.2] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms` | matched-speed holds at two currents, the tag on the hub (the speed notebook's brush-form gap, stage 3's, and the README's velocity gap) |
| one velocity: the switch against the complementary pair | switch sw5 | crawl bias 0.404 [0.278, 0.53] rev/s, +/- se `vel:blend_bias_crawl_rev_s/sw5`, stall over the floor 1.226 rev/s `vel:blend_bias_stall_over_floor_rev_s/sw5`, own sd 3.426 rev/s `vel:blend_own_sd_rev_s/crawl/sw5`, seams 13.47 rev/s `vel:blend_seam_rev_s/sw5_kick/median` at a start and 6.815 rev/s `vel:blend_seam_rev_s/sw5_coast/median` at a coast, direct path 0.117 [0.103, 0.131] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/boxcar_32/r10_39`, no state of its own; with the rest verdict (swr5) a seam of 44.69 rev/s `vel:blend_seam_rev_s/swr5_kick/median` | the loop design (stage 5): the back-EMF's bias at DC against the pot's delay below the corner and a step of 14.39 rev/s `vel:blend_seam_rev_s/cpr5_kick/median` in the feedback at a start |
| one velocity: the switch against the complementary pair | pair cp5 | crawl bias 0.0442 [9.35e-05, 0.0883] rev/s, +/- se `vel:blend_bias_crawl_rev_s/cp5`, stall over the floor 0.0156 rev/s `vel:blend_bias_stall_over_floor_rev_s/cp5`, own sd 3.517 rev/s `vel:blend_own_sd_rev_s/crawl/cp5`, seams 0.205 rev/s `vel:blend_seam_rev_s/cp5_handover/median` at the handover, 5.408 rev/s `vel:blend_seam_rev_s/cp5_coast/median` at a coast, 14.39 rev/s `vel:blend_seam_rev_s/cpr5_kick/median` at a start with rest on pot rows, direct path 0.153 [0.125, 0.18] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_5/r10_39` and 0.0496 [0.00671, 0.0925] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/pair_5/r40_199`, one one-pole at 1030.5 rows, the pair's one-pole `vel:blend_corner_rows` and the pot slope's delay below it; cp25 under the floor 1.853 rev/s `vel:blend_own_sd_rev_s/under_floor/cp25` against cp5 4.130 rev/s `vel:blend_own_sd_rev_s/under_floor/cp5` | the loop design (stage 5): the back-EMF's bias at DC against the pot's delay below the corner and a step of 14.39 rev/s `vel:blend_seam_rev_s/cpr5_kick/median` in the feedback at a start |
| rest verdict: the band against the slope | band | band of 2 `vel:rest_verdict/band/counts` counts: declare 56.35 ms, median `vel:rest_declare_ms/band 2 T50/coast_down/median` (p90 64.53 ms, p90 `vel:rest_declare_ms/band 2 T50/coast_down/p90`), drop 7.725 ms, median `vel:rest_drop_ms/band 2 T50/median`, false motion on coast tails 0.0193 share of rows `vel:rest_false_motion_share/band 2 T50/rest_coast`, on load holds 0.0058 share of rows `vel:rest_false_motion_stage3_share/band 2 T50/load holds`; reads counts, no slope; a band of 1 count holds 0.0892 share of rows `vel:rest_false_motion_stage3_share/band 1 T50/stalls resistance` of the stalls' rows as motion | nothing in this data over 2 sigma (equal alternatives `vel:rest_verdict/note`); a driven sweep at 2 to 10 deg/s (the velocity notebook's rest-verdict gap); the veto is a separate option: drop 5.750 ms, median `vel:rest_drop_ms/band 2 T50 + emf/median`, fires on 1 of 106 `vel:rest_veto/fires_on_stopped` |
| rest verdict: the band against the slope | slope | slope of 192 `vel:rest_verdict/slope/rows` rows: declare 56.10 ms, median `vel:rest_declare_ms/slope 192 T50/coast_down/median` (p90 57.83 ms, p90 `vel:rest_declare_ms/slope 192 T50/coast_down/p90`), drop 6.500 ms, median `vel:rest_drop_ms/slope 192 T50/median`, false motion on coast tails 0.00777 share of rows `vel:rest_false_motion_share/slope 192 T50/rest_coast`, on load holds 0.0011 share of rows `vel:rest_false_motion_stage3_share/slope 192 T50/load holds`; keeps the slope's rows; a 992-row slope holds 0.0039 share of rows `vel:rest_false_motion_share/slope 992 T75/rest_base` of the baselines as motion | nothing in this data over 2 sigma (equal alternatives `vel:rest_verdict/note`); a driven sweep at 2 to 10 deg/s (the velocity notebook's rest-verdict gap); the veto is a separate option: drop 5.750 ms, median `vel:rest_drop_ms/band 2 T50 + emf/median`, fires on 1 of 106 `vel:rest_veto/fires_on_stopped` |
| tap floor's rung | 84 ticks | floor 84 [72, 96] ticks; bounds: the failing rung below, the first rung with 10 runs or more `spd:valid_rule/tap_floor_ticks`; deficit 10.50 [9.413, 29.59] mV, median [10-90 %] `spd:tap_deficit_by_ticks/84-84/deficit_mv` on n 3 `spd:tap_deficit_by_ticks/84-84/n` runs of 3 `spd:tap_deficit_by_ticks/84-84/sweeps` sweeps; valid share 85.21 `spd:valid_coverage_by_set/all/valid_pct` %, under the floor 14.62 `spd:valid_coverage_by_set/all/under_floor_pct` % | a hold ladder at rest from 72 to 96 ticks in steps of 4 (the speed notebook's tap-floor gap, stage 3's tap-floor discrepancy) |
| tap floor's rung | 96 ticks | deficit -22.81 [-29.34, -10.51] mV, median [10-90 %] `spd:tap_deficit_by_ticks/96-96/deficit_mv` on n 16 `spd:tap_deficit_by_ticks/96-96/n` runs; rows under 96 ticks 21.11 `spd:valid_coverage_by_set/all/under_96_ticks_pct` %; stage 3's closing table: 93 ticks `spd:valid_rule/tap_floor_stage3_closing_ticks`; stage 3's key 72 half-ticks, V_on low with window_valid 1 `s3:tap_floor_half_ticks` is the failing rung (482.5 [341.8, 504.8] mV, median [10-90 %] `spd:tap_deficit_by_ticks/72-72/deficit_mv`, as speed -3.975 `spd:tap_deficit_by_ticks/72-72/as_rev_s` rev/s) | a hold ladder at rest from 72 to 96 ticks in steps of 4 (the speed notebook's tap-floor gap, stage 3's tap-floor discrepancy) |

The coverage of the speed estimate's valid flag by set, at the floor of record and under the upper rung, as the speed file holds it.

In [70]:
COVER = pd.DataFrame(node("spd", "valid_coverage_by_set")).T
print("<spd:valid_coverage_by_set>")
print(COVER.round(3).to_string())

<spd:valid_coverage_by_set>
             sweeps  driven_rows  under_floor_pct  start_up_pct  dropout_pct  valid_pct  window_valid_0_pct  under_96_ticks_pct  after_duty_step_pct
inductance      2.0      11991.0            0.000         0.267        0.000     99.733               0.000               0.000                0.042
ladder        123.0    4231408.0           23.821         0.094        0.023     76.063              20.418              30.628                0.011
load bottom    18.0     420002.0            0.000         0.071        0.000     99.929               0.000               0.000                0.000
load ladder   112.0     729503.0            6.574         0.472        0.000     92.955               3.285               6.574                0.101
load lift      36.0     527979.0            0.000         0.111        0.000     99.889               0.000               0.000                0.000
load steps     55.0    1113753.0            9.874         0.149        0.000  

## 6. The gaps

### As each notebook lists them

The current-zero, position and winding-temperature files and stage 3's file carry a `gaps` key, read here as they stand. The speed and velocity files carry none, so their gaps are written here from those notebooks' closing sections, each with the capture the notebook names. The numbers in the first column are the ones the consolidated table uses.

In [71]:
SPD_GAPS = [   # the speed notebook's closing section: gap, capture
    ("the brush form", "holds at one speed and two currents, at a few "
     "speeds across the crawl and the lower ladder, the tag on the hub"),
    ("the run-ups' direction term", "100 % run-ups on the 3.7 V rail, half "
     "the current and few dropouts, scored row by row with the dropout "
     "rows out"),
    ("the top of the range, the two top rungs still rising inside their "
     "windows", "the 7.4 V top rungs held twice as long"),
    ("the tap floor", "a hold ladder at rest from 72 to 96 ticks in steps "
     "of 4"),
    ("the duty step's residue after the L / R spike", "the steps on the RL "
     "dummy (no EMF) and the same steps with the horn locked"),
    ("the stopped rotor's slope with the current", "the bias against time "
     "inside each hold (existing rows), then holds of 0.2 s against 3 s at "
     "one current"),
    ("the crawl's slow part, under half the commutation rate", "the "
     "per-period tach intervals against the estimate's period means in the "
     "same windows (existing rows)"),
    ("the rows' slow channels, no NTC and no rail", "one sweep's raw "
     "telemetry frames against the loader"),
    ("closed loop, every filter scored open loop", "a closed-loop hold, "
     "stage 5's bench")]

In [72]:
VEL_GAPS = [   # the velocity notebook's closing section: gap, capture
    ("the stopped rotor's law below the lowest stopped hold's current",
     "stopped holds at 20 to 70 mA, the load ladder's lowest rungs "
     "extended down, or a crawl against a brake at 100 to 200 mA"),
    ("the direction split's cause, speed and current collinear on the "
     "sitting", "the same speed at two currents, a crawl against a brake "
     "in both directions"),
    ("the crossover's move against stage 3", "three offline re-scorings of "
     "the same rows, with stage 3's truth, then its zero, then its band"),
    ("the tag as the truth under 100 ms of window", "a lookup fitted on "
     "other chains than it is scored on; a second tag bonded to the output "
     "shaft, or an encoder on it"),
    ("the rest verdict between 2 and 10 deg/s", "a driven sweep at 2 to "
     "10 deg/s"),
    ("the commutation pattern's cause", "a coast with the taps streamed, "
     "as an identification capture"),
    ("the angle-locked motion's shaft", "passes of 40 deg and more at a "
     "steady crawl; a crawl at two duties over the same angles"),
    ("the kick's end: the back-EMF low under negative current, the pot "
     "short of the shaft over a kick", "a kick straight into a coast; kick "
     "ends into holds that keep the current from reversing; a time mark "
     "the camera and the rows both see; kicks of both directions over the "
     "same angles"),
    ("a second sitting", "a second sitting, and a second servo"),
    ("the blend", "a loop design that weighs the switch's bias against the "
     "pair's one-pole and its seams")]

In [73]:
LISTED = []                             # every notebook's gaps, numbered
for i, g in enumerate(node("cz", "gaps"), 1):
    LISTED.append({"notebook": "current-zero", "n": i, "gap": g["gap"],
                   "capture": f"{g['capture']} <cz:gaps/{i - 1}>"})
for i, g in enumerate(node("pos", "gaps"), 1):
    head, _, cap = g.partition(": ")    # 'title: capture'
    LISTED.append({"notebook": "position", "n": i, "gap": head,
                   "capture": f"{cap} <pos:gaps/{i - 1}>"})
for i, g in enumerate(node("wt", "gaps"), 1):
    LISTED.append({"notebook": "winding-temperature", "n": i,
                   "gap": f"{g['gap']} ({g['why']})",
                   "capture": f"{g['capture']} <wt:gaps/{i - 1}>"})
for i, (g, c) in enumerate(SPD_GAPS, 1):
    LISTED.append({"notebook": "speed", "n": i, "gap": g, "capture": c})
for i, (g, c) in enumerate(VEL_GAPS, 1):
    LISTED.append({"notebook": "velocity", "n": i, "gap": g, "capture": c})
for i, g in enumerate(node("s3", "gaps"), 1):
    LISTED.append({"notebook": "stage 3", "n": i,
                   "gap": f"{g} <s3:gaps/{i - 1}>", "capture": "-"})
print(pd.DataFrame(LISTED).groupby("notebook", sort=False).size()
      .to_string())
show(LISTED, ["notebook", "n", "gap", "capture"])

notebook
current-zero           11
position                7
winding-temperature     9
speed                   9
velocity               10
stage 3                 6


| notebook | n | gap | capture |
|---|---|---|---|
| current-zero | 1 | no rail, reference or temperature channel in the rows | a stage 3 schedule re-streamed with vbus, vref_raw and ntc_raw; a torque-off and coast series while the board warms `cz:gaps/0` |
| current-zero | 2 | the step's shape off 4.9 V rests on one day | three sweeps with a coast at 3.7, 5.0 and 7.4 V on another day `cz:gaps/1` |
| current-zero | 3 | the 6.0-6.8 V band sits above the curve | a ladder sweep with coasts at 6.0, 6.4 and 6.8 V `cz:gaps/2` |
| current-zero | 4 | the step under a rail that sags with the load | a narrow drive with a series resistor for half the run, a coast after each half `cz:gaps/3` |
| current-zero | 5 | a drive between two readings adds 0.40 counts | the ladder with a rest in place of every second rung `cz:gaps/4` |
| current-zero | 6 | fast alphas against a far truth | a 0.1 s coast after every narrow rung `cz:gaps/5` |
| current-zero | 7 | stalls and loaded holds scored on the proxy truth | a 0.1 s coast after a stall hold `cz:gaps/6` |
| current-zero | 8 | nothing between 80 and 100 % duty | rungs at 85, 90 and 95 % with a coast after each `cz:gaps/7` |
| current-zero | 9 | the trough's rise under a drive grows with the rail | stage 1's dummy-load trough at two rails, same duty `cz:gaps/8` |
| current-zero | 10 | the 3.3 V laps moved both zeros | the asleep baseline with vref_raw while the supply steps 3.3 to 3.8 V in 50 mV `cz:gaps/9` |
| current-zero | 11 | the zero during and after a wide drive over 2.4 s | holds at 60 % duty for 10 and 60 s, a coast after each `cz:gaps/10` |
| position | 1 | The direction term | crawl segments in both directions over the same span, each direction started from both ends, with a marker driven from the PWM timer in the camera's view so the stamp lead is measured `pos:gaps/0` |
| position | 2 | The high end | rest points every 2 deg from -30 to -16 deg, or one slow pass that runs to the guard `pos:gaps/1` |
| position | 3 | The carries outside the covered codes | one slow pass guard to guard `pos:gaps/2` |
| position | 4 | Between sittings | the opening pass and 20 rest points repeated in a second sitting, graded through this lookup `pos:gaps/3` |
| position | 5 | Temperature | rest points at two bed temperatures on the heated rig `pos:gaps/4` |
| position | 6 | Side load and torque | the load ladder's holds with the tag in view `pos:gaps/5` |
| position | 7 | Wear of the track | the between-sittings capture repeated over the servo's running life `pos:gaps/6` |
| winding-temperature | 1 | a plateau, the total gain winding to ambient (the longest drive is 6.9 s, one of 36.3 s, mean power at most 45.1 mW) | a stall hold on the horn lock at two currents (0.25 and 0.4 A) for 10 to 15 min each, a 100 ms window every 10 s `wt:gaps/0` |
| winding-temperature | 2 | the slower nodes (rotor iron, can) (only the slow node's first seconds are seen, its heat capacity) | the same hold, then the cooling, a 100 ms probe pulse at low duty every 10 to 20 s for 15 min `wt:gaps/1` |
| winding-temperature | 3 | cooling at all (no read at one seat after a rest, every sweep re-seats) | probe pulses after a hold with the seat kept (the lock, no seek, no move at torque off) `wt:gaps/2` |
| winding-temperature | 4 | a hot ambient (the NTC is 24.5 to 28.7 C in every sitting) | a second ambient, on the printer bed or in a warm room `wt:gaps/3` |
| winding-temperature | 5 | the NTC against the can (the NTC sits outside the case and climbs up to 2.18 C after power-up) | a thermocouple on the can, once, through a sitting `wt:gaps/4` |
| winding-temperature | 6 | the seek press (its length is not recorded, 0.16 to 0.43 s) | stream the seek, or chains on the lock with no seek `wt:gaps/5` |
| winding-temperature | 7 | the reference under the construction (the chains are read on warmed copper, the 100 % laps on cold) | partial-duty chains on the horn lock, static load, no seek, 60 s rests `wt:gaps/6` |
| winding-temperature | 8 | the bridged seats' rate (3 holds in one sitting pair) | stopped reads at 20 or more rest angles a few counts apart `wt:gaps/7` |
| winding-temperature | 9 | between the windows of a hold (holds are unstreamed between their windows) | stream the hold at a low rate, or holds of 1 and 2 s `wt:gaps/8` |
| speed | 1 | the brush form | holds at one speed and two currents, at a few speeds across the crawl and the lower ladder, the tag on the hub |
| speed | 2 | the run-ups' direction term | 100 % run-ups on the 3.7 V rail, half the current and few dropouts, scored row by row with the dropout rows out |
| speed | 3 | the top of the range, the two top rungs still rising inside their windows | the 7.4 V top rungs held twice as long |
| speed | 4 | the tap floor | a hold ladder at rest from 72 to 96 ticks in steps of 4 |
| speed | 5 | the duty step's residue after the L / R spike | the steps on the RL dummy (no EMF) and the same steps with the horn locked |
| speed | 6 | the stopped rotor's slope with the current | the bias against time inside each hold (existing rows), then holds of 0.2 s against 3 s at one current |
| speed | 7 | the crawl's slow part, under half the commutation rate | the per-period tach intervals against the estimate's period means in the same windows (existing rows) |
| speed | 8 | the rows' slow channels, no NTC and no rail | one sweep's raw telemetry frames against the loader |
| speed | 9 | closed loop, every filter scored open loop | a closed-loop hold, stage 5's bench |
| velocity | 1 | the stopped rotor's law below the lowest stopped hold's current | stopped holds at 20 to 70 mA, the load ladder's lowest rungs extended down, or a crawl against a brake at 100 to 200 mA |
| velocity | 2 | the direction split's cause, speed and current collinear on the sitting | the same speed at two currents, a crawl against a brake in both directions |
| velocity | 3 | the crossover's move against stage 3 | three offline re-scorings of the same rows, with stage 3's truth, then its zero, then its band |
| velocity | 4 | the tag as the truth under 100 ms of window | a lookup fitted on other chains than it is scored on; a second tag bonded to the output shaft, or an encoder on it |
| velocity | 5 | the rest verdict between 2 and 10 deg/s | a driven sweep at 2 to 10 deg/s |
| velocity | 6 | the commutation pattern's cause | a coast with the taps streamed, as an identification capture |
| velocity | 7 | the angle-locked motion's shaft | passes of 40 deg and more at a steady crawl; a crawl at two duties over the same angles |
| velocity | 8 | the kick's end: the back-EMF low under negative current, the pot short of the shaft over a kick | a kick straight into a coast; kick ends into holds that keep the current from reversing; a time mark the camera and the rows both see; kicks of both directions over the same angles |
| velocity | 9 | a second sitting | a second sitting, and a second servo |
| velocity | 10 | the blend | a loop design that weighs the switch's bias against the pair's one-pole and its seams |
| stage 3 | 1 | PRBS and chirps. The steps only, since no osc-lab verb writes duty on a host clock `s3:gaps/0` | - |
| stage 3 | 2 | The closed-loop hold, stage 5's bench. The open-loop holds here are unstreamed between their two windows `s3:gaps/1` | - |
| stage 3 | 3 | A real pack. The DPS only, the cap read on the rail at rest `s3:gaps/2` | - |
| stage 3 | 4 | The loaded speed estimate at partial duty rests on the lift alone, pot truth, 12 lifts `s3:gaps/3` | - |
| stage 3 | 5 | The brush term's form under load needs matched-speed holds at two currents `s3:gaps/4` | - |
| stage 3 | 6 | The position grade at speed needs the stamp lead fixed `s3:gaps/5` | - |

### Consolidated

One row per gap, de-duplicated across the lists above, with the capture that would close it and the deliverable of stages 5 to 7 it limits. "Listed by" names each list and its number. The last rows are this notebook's own, the offline analysis that would split the load holds' two readings, the stage 3 rate flag that no stage 4 notebook ran, and the cells of section 2 that read "not measured".

In [74]:
CZG = lambda i: f"{node('cz', f'gaps/{i}/gap')} <cz:gaps/{i}/gap>"
GAPS = [   # gap, listed by, the capture that closes it, what it limits
    ("the brush term's form under load", "speed 1, stage 3 5, README",
     "holds at one speed and two currents, at a few speeds across the crawl "
     "and the lower ladder, the tag on the hub", "velocity loop's feedback "
     "(bias under load), thermal and stall limits that read the speed"),
    (f"the run-ups' direction term (c_dir {Q('spd', PL + '/c_dir_mv')} "
     f"against {Q('spd', PR + '/c_dir_mv')})", "speed 2", "100 % run-ups "
     "on the 3.7 V rail, scored row by row with the dropout rows out",
     "velocity loop's feedback at speed under current"),
    (f"the top of the speed range, neither form resolved ({TOP})",
     "speed 3", "the 7.4 V top rungs held twice as long",
     "velocity loop's feedback at the top"),
    (f"the tap floor's rung, between {num(FLOOR_Q['lo'])} and "
     f"{num(FLOOR_Q['hi'])} ticks <spd:valid_rule/tap_floor_ticks>",
     "speed 4, stage 3's tap-floor discrepancy", "a hold ladder at rest "
     "from 72 to 96 ticks in steps of 4", "valid flags for the faults; the "
     "velocity loop's handover to the pot"),
    ("the duty step's residue, "
     f"{Q('spd', DS + '/share_rows_10_39')} after the L / R spike",
     "speed 5", "the steps on the RL dummy and the same steps with the "
     "horn locked", "velocity loop's feedback (the direct path a loop "
     "design carries)"),
    ("the stopped rotor: its slope, its convexity, its law below the "
     f"lowest stopped hold ({Q('vel', VS + '/table/valid_ma')} mA), and "
     "whether it is the winding's heat", "speed 6, velocity 1, the "
     "winding-temperature notebook's open discrepancy, section 4",
     "the bias against time inside each hold (existing rows), holds of "
     "0.2 s against 3 s at one current, stopped holds at 20 to 70 mA or a "
     "crawl against a brake at 100 to 200 mA", "velocity loop's feedback "
     "at a stall, the rest verdict's veto, the stall fault"),
    ("the load holds read two ways, a stopped-rotor bias or the winding's "
     "heat (section 4)", "this notebook", "none: an offline analysis, the "
     "speed estimate re-run on the load holds with the one-node R(T)",
     "velocity loop's feedback at a stall, the rest verdict's veto, "
     "thermal derate and cutoff")]

In [75]:
GAPS += [
    ("the crawl's slow noise and the commutation pattern's cause",
     "speed 7, velocity 6", "per-period tach intervals against the "
     "estimate's period means (existing rows); a coast with the taps "
     "streamed, as an identification capture", "velocity loop's feedback "
     "(what a filter may remove)"),
    (f"no rail, reference or board temperature channel in the rows "
     f"({CZG(0)})", "current-zero 1, speed 8, stage 3 row 14", "one "
     "sweep's raw frames against the loader; a stage 3 schedule "
     "re-streamed with vbus, vref_raw and ntc_raw; a torque-off and coast "
     "series while the board warms", "current loop's feedback (the start "
     "rule before the first drive), thermal derate, the sag cap"),
    ("closed loop: every filter and blend was scored open loop; chirps and "
     "PRBS", "speed 9, velocity 10, stage 3 1 and 2, README", "a "
     "closed-loop hold and chirps, stage 5's bench", "velocity and "
     "position loops' feedback"),
    ("the direction split of the back-EMF at a crawl, speed and current "
     "collinear", "velocity 2", "the same speed at two currents, a crawl "
     "against a brake in both directions", "velocity loop's feedback at a "
     "crawl"),
]

In [76]:
GAPS += [
    ("the crossover's move against stage 3", "velocity 3", "three offline "
     "re-scorings of the same rows (stage 3's truth, then its zero, then "
     "its band)", "velocity loop's feedback (which source at short "
     "windows)"),
    ("the tag as the truth under 100 ms of window", "velocity 4", "a "
     "lookup fitted on other chains than it is scored on; a second tag on "
     "the output shaft or an encoder", "velocity loop's feedback (noise "
     "against delay at loop windows)"),
    ("the rest verdict between 2 and 10 deg/s", "velocity 5", "a driven "
     "sweep at 2 to 10 deg/s", "velocity loop's feedback at a crawl, valid "
     "flags for the faults (stall)"),
    ("the angle-locked motion's shaft, at periods of "
     f"{Q('vel', 'crawl_angle_locked_motion_rev_s/periods')}", "velocity 7",
     "passes of 40 deg and more at a steady crawl; a crawl at two duties "
     "over the same angles", "velocity loop's feedback (the smooth crawl)"),
    ("the kick's end: the back-EMF low under negative current; the "
     "commutation count runs "
     f"{Q('vel', 'count_kick_start_145ms/ratio_count_pot')} of the pot over "
     "a kick", "velocity 8", "a kick straight into a coast; kick ends into "
     "holds that keep the current from reversing; a time mark the camera "
     "and the rows both see; kicks of both directions over the same "
     "angles", "velocity loop's feedback through a hard transient, "
     "position loop's feedback in motion")]

In [77]:
GAPS += [
    ("the direction term of the position and the stamp lead (the grade at "
     "speed)", "position 1, stage 3 6, README, velocity 8", "crawl segments "
     "in both directions over the same span, each started from both ends, "
     "with a marker driven from the PWM timer in the camera's view",
     "position loop's feedback (the lash bound, the grade at speed)"),
    (f"the high end of the lookup, a hole of "
     f"{Q('pos', 'high_end_hole_deg')}", "position 2", "rest points every "
     "2 deg from -30 to -16 deg, or one slow pass to the guard",
     "position loop's feedback near the high stop"),
    ("the carries outside the covered codes", "position 3", "one slow pass "
     "guard to guard", "position and velocity loops' feedback at the ends"),
    ("one sitting, one servo: between sittings, temperature, wear",
     "position 4, 5 and 7, velocity 9", "the opening pass and 20 rest "
     "points in a second sitting; rest points at two bed temperatures; the "
     "same over the servo's life; a second servo", "every deliverable (a "
     "candidate until repeated)"),
    ("side load and torque on the pot", "position 6, stage 3 row 4", "the "
     "load ladder's holds with the tag in view", "position loop's feedback "
     "under load (stage 7's bound)"),
]

In [78]:
GAPS += [
    (f"{CZG(1)}; {CZG(2)}", "current-zero 2 and 3", "three sweeps with a "
     "coast at 3.7, 5.0 and 7.4 V on another day; a ladder sweep with "
     "coasts at 6.0, 6.4 and 6.8 V", "current loop's feedback on a 2S "
     "rail"),
    (f"{CZG(3)}; {CZG(8)}", "current-zero 4 and 9, stage 3 3", "a narrow "
     "drive with a series resistor for half the run; stage 1's dummy-load "
     "trough at two rails", "current loop's feedback on a real pack"),
    (f"{CZG(4)}; {CZG(5)}; {CZG(7)}; {CZG(10)}", "current-zero 5, 6, 8 and "
     "11", "the ladder with a rest in place of every second rung; a 0.1 s "
     "coast after every narrow rung; holds at 60 % for 10 and 60 s; rungs "
     "at 85, 90 and 95 %", "current loop's feedback (the hold bound in a "
     "long wide drive)"),
    (CZG(6), "current-zero 7", "a 0.1 s coast after a stall hold",
     "current loop's feedback at a stall, the stall fault"),
    (CZG(9), "current-zero 10", "the asleep baseline with vref_raw while "
     "the supply steps 3.3 to 3.8 V in 50 mV", "current loop's feedback "
     "near the regulator's dropout")]

In [79]:
WTG = lambda i: f"{node('wt', f'gaps/{i}/gap')} <wt:gaps/{i}/gap>"
GAPS += [
    ("no thermal capture: " + "; ".join(WTG(i) for i in range(5)),
     "winding-temperature 1 to 5, README", "a stall hold on the horn lock "
     "at 0.25 and 0.4 A for 10 to 15 min with a 100 ms window every 10 s, "
     "then the cooling with probe pulses; a second ambient; a thermocouple "
     "on the can", "thermal derate and cutoff (no graded basis past the "
     "graded drive)"),
    ("; ".join(WTG(i) for i in range(5, 9)), "winding-temperature 6 to 9, "
     "stage 3 2", "stream the seek or chains on the lock with no seek; "
     "partial-duty chains on the horn lock; stopped reads at 20 or more "
     "rest angles; stream the hold at a low rate", "thermal derate and "
     "cutoff (R0 open as a cold value), the hold deadband's drift"),
    (f"{node('s3', 'gaps/3')} <s3:gaps/3>", "stage 3 4", "the lift with the "
     "tag on the hub (stage 3's lift discrepancy)", "velocity loop's "
     "feedback under load"),
    ("the dropout rate flag of stage 3 was not run in stage 4", "this "
     "notebook, section 3 row 9", "none needed for the rule; a sitting "
     "where both directions drop out (stage 3's dropout discrepancy)",
     "valid flags for the faults")]
NOT_MEASURED = [   # the cells of section 2 that read 'not measured'
    ("the pot slope's start-up", "velocity loop's feedback (start-up)"),
    ("the share of rows inside the position's valid range",
     "position loop's feedback"),
    ("the winding estimate's start-up error", "thermal derate and cutoff"),
    ("cp25's share of rows by source", "velocity loop's feedback (the "
     "blend)"),
    ("cpr5's delay", "velocity loop's feedback (the blend)")]
GAPS += [(g, "this notebook, section 2", "not named; the cell reads 'not "
          "measured'", lim) for g, lim in NOT_MEASURED]
GAPS = [{"gap": g, "listed_by": s, "capture": c, "limits": d,
         "sources": sources(g)} for g, s, c, d in GAPS]
print(len(GAPS), "gaps")
show(GAPS, ["gap", "listed_by", "capture", "limits"],
     ["gap", "listed by", "capture that closes it", "limits"])

35 gaps


| gap | listed by | capture that closes it | limits |
|---|---|---|---|
| the brush term's form under load | speed 1, stage 3 5, README | holds at one speed and two currents, at a few speeds across the crawl and the lower ladder, the tag on the hub | velocity loop's feedback (bias under load), thermal and stall limits that read the speed |
| the run-ups' direction term (c_dir 12.39 [9.149, 15.63] mV, +/- se `spd:brush_forms/product_ladder_alone/c_dir_mv` against 36.02 [34.22, 37.82] mV, +/- se `spd:brush_forms/product_ladder_runups/c_dir_mv`) | speed 2 | 100 % run-ups on the 3.7 V rail, scored row by row with the dropout rows out | velocity loop's feedback at speed under current |
| the top of the speed range, neither form resolved (ladder (third repeat) 450-600 (456.9 rev/s, 272 mA): table 5.960, product -1.710 rev/s `spd:stage5_bias_by_band_form/5`) | speed 3 | the 7.4 V top rungs held twice as long | velocity loop's feedback at the top |
| the tap floor's rung, between 72 and 96 ticks `spd:valid_rule/tap_floor_ticks` | speed 4, stage 3's tap-floor discrepancy | a hold ladder at rest from 72 to 96 ticks in steps of 4 | valid flags for the faults; the velocity loop's handover to the pot |
| the duty step's residue, 0.114 of the volts' step / Ke `spd:duty_step_response/share_rows_10_39` after the L / R spike | speed 5 | the steps on the RL dummy and the same steps with the horn locked | velocity loop's feedback (the direct path a loop design carries) |
| the stopped rotor: its slope, its convexity, its law below the lowest stopped hold ([69.72, 483.7] `vel:emf_stopped_rotor_rev_s/table/valid_ma` mA), and whether it is the winding's heat | speed 6, velocity 1, the winding-temperature notebook's open discrepancy, section 4 | the bias against time inside each hold (existing rows), holds of 0.2 s against 3 s at one current, stopped holds at 20 to 70 mA or a crawl against a brake at 100 to 200 mA | velocity loop's feedback at a stall, the rest verdict's veto, the stall fault |
| the load holds read two ways, a stopped-rotor bias or the winding's heat (section 4) | this notebook | none: an offline analysis, the speed estimate re-run on the load holds with the one-node R(T) | velocity loop's feedback at a stall, the rest verdict's veto, thermal derate and cutoff |
| the crawl's slow noise and the commutation pattern's cause | speed 7, velocity 6 | per-period tach intervals against the estimate's period means (existing rows); a coast with the taps streamed, as an identification capture | velocity loop's feedback (what a filter may remove) |
| no rail, reference or board temperature channel in the rows (no rail, reference or temperature channel in the rows `cz:gaps/0/gap`) | current-zero 1, speed 8, stage 3 row 14 | one sweep's raw frames against the loader; a stage 3 schedule re-streamed with vbus, vref_raw and ntc_raw; a torque-off and coast series while the board warms | current loop's feedback (the start rule before the first drive), thermal derate, the sag cap |
| closed loop: every filter and blend was scored open loop; chirps and PRBS | speed 9, velocity 10, stage 3 1 and 2, README | a closed-loop hold and chirps, stage 5's bench | velocity and position loops' feedback |
| the direction split of the back-EMF at a crawl, speed and current collinear | velocity 2 | the same speed at two currents, a crawl against a brake in both directions | velocity loop's feedback at a crawl |
| the crossover's move against stage 3 | velocity 3 | three offline re-scorings of the same rows (stage 3's truth, then its zero, then its band) | velocity loop's feedback (which source at short windows) |
| the tag as the truth under 100 ms of window | velocity 4 | a lookup fitted on other chains than it is scored on; a second tag on the output shaft or an encoder | velocity loop's feedback (noise against delay at loop windows) |
| the rest verdict between 2 and 10 deg/s | velocity 5 | a driven sweep at 2 to 10 deg/s | velocity loop's feedback at a crawl, valid flags for the faults (stall) |
| the angle-locked motion's shaft, at periods of 4 deg and longer `vel:crawl_angle_locked_motion_rev_s/periods` | velocity 7 | passes of 40 deg and more at a steady crawl; a crawl at two duties over the same angles | velocity loop's feedback (the smooth crawl) |
| the kick's end: the back-EMF low under negative current; the commutation count runs 1.024 [1.018, 1.030] ratio, +/- 2 se, n 38 `vel:count_kick_start_145ms/ratio_count_pot` of the pot over a kick | velocity 8 | a kick straight into a coast; kick ends into holds that keep the current from reversing; a time mark the camera and the rows both see; kicks of both directions over the same angles | velocity loop's feedback through a hard transient, position loop's feedback in motion |
| the direction term of the position and the stamp lead (the grade at speed) | position 1, stage 3 6, README, velocity 8 | crawl segments in both directions over the same span, each started from both ends, with a marker driven from the PWM timer in the camera's view | position loop's feedback (the lash bound, the grade at speed) |
| the high end of the lookup, a hole of 7.966 deg `pos:high_end_hole_deg` | position 2 | rest points every 2 deg from -30 to -16 deg, or one slow pass to the guard | position loop's feedback near the high stop |
| the carries outside the covered codes | position 3 | one slow pass guard to guard | position and velocity loops' feedback at the ends |
| one sitting, one servo: between sittings, temperature, wear | position 4, 5 and 7, velocity 9 | the opening pass and 20 rest points in a second sitting; rest points at two bed temperatures; the same over the servo's life; a second servo | every deliverable (a candidate until repeated) |
| side load and torque on the pot | position 6, stage 3 row 4 | the load ladder's holds with the tag in view | position loop's feedback under load (stage 7's bound) |
| the step's shape off 4.9 V rests on one day `cz:gaps/1/gap`; the 6.0-6.8 V band sits above the curve `cz:gaps/2/gap` | current-zero 2 and 3 | three sweeps with a coast at 3.7, 5.0 and 7.4 V on another day; a ladder sweep with coasts at 6.0, 6.4 and 6.8 V | current loop's feedback on a 2S rail |
| the step under a rail that sags with the load `cz:gaps/3/gap`; the trough's rise under a drive grows with the rail `cz:gaps/8/gap` | current-zero 4 and 9, stage 3 3 | a narrow drive with a series resistor for half the run; stage 1's dummy-load trough at two rails | current loop's feedback on a real pack |
| a drive between two readings adds 0.40 counts `cz:gaps/4/gap`; fast alphas against a far truth `cz:gaps/5/gap`; nothing between 80 and 100 % duty `cz:gaps/7/gap`; the zero during and after a wide drive over 2.4 s `cz:gaps/10/gap` | current-zero 5, 6, 8 and 11 | the ladder with a rest in place of every second rung; a 0.1 s coast after every narrow rung; holds at 60 % for 10 and 60 s; rungs at 85, 90 and 95 % | current loop's feedback (the hold bound in a long wide drive) |
| stalls and loaded holds scored on the proxy truth `cz:gaps/6/gap` | current-zero 7 | a 0.1 s coast after a stall hold | current loop's feedback at a stall, the stall fault |
| the 3.3 V laps moved both zeros `cz:gaps/9/gap` | current-zero 10 | the asleep baseline with vref_raw while the supply steps 3.3 to 3.8 V in 50 mV | current loop's feedback near the regulator's dropout |
| no thermal capture: a plateau, the total gain winding to ambient `wt:gaps/0/gap`; the slower nodes (rotor iron, can) `wt:gaps/1/gap`; cooling at all `wt:gaps/2/gap`; a hot ambient `wt:gaps/3/gap`; the NTC against the can `wt:gaps/4/gap` | winding-temperature 1 to 5, README | a stall hold on the horn lock at 0.25 and 0.4 A for 10 to 15 min with a 100 ms window every 10 s, then the cooling with probe pulses; a second ambient; a thermocouple on the can | thermal derate and cutoff (no graded basis past the graded drive) |
| the seek press `wt:gaps/5/gap`; the reference under the construction `wt:gaps/6/gap`; the bridged seats' rate `wt:gaps/7/gap`; between the windows of a hold `wt:gaps/8/gap` | winding-temperature 6 to 9, stage 3 2 | stream the seek or chains on the lock with no seek; partial-duty chains on the horn lock; stopped reads at 20 or more rest angles; stream the hold at a low rate | thermal derate and cutoff (R0 open as a cold value), the hold deadband's drift |
| The loaded speed estimate at partial duty rests on the lift alone, pot truth, 12 lifts `s3:gaps/3` | stage 3 4 | the lift with the tag on the hub (stage 3's lift discrepancy) | velocity loop's feedback under load |
| the dropout rate flag of stage 3 was not run in stage 4 | this notebook, section 3 row 9 | none needed for the rule; a sitting where both directions drop out (stage 3's dropout discrepancy) | valid flags for the faults |
| the pot slope's start-up | this notebook, section 2 | not named; the cell reads 'not measured' | velocity loop's feedback (start-up) |
| the share of rows inside the position's valid range | this notebook, section 2 | not named; the cell reads 'not measured' | position loop's feedback |
| the winding estimate's start-up error | this notebook, section 2 | not named; the cell reads 'not measured' | thermal derate and cutoff |
| cp25's share of rows by source | this notebook, section 2 | not named; the cell reads 'not measured' | velocity loop's feedback (the blend) |
| cpr5's delay | this notebook, section 2 | not named; the cell reads 'not measured' | velocity loop's feedback (the blend) |

## 7. What stages 5 to 7 read from here

The README's last table, row by row, with the numbers the result files give each deliverable. The README's own entries are read from the file and sit beside the numbers.

In [80]:
README = (results.ROOT / "stages" / STAGE / "README.md").read_text()
TAIL = README[README.index("## What stages 5 to 7 read from here"):]
README_ROWS = {}                        # deliverable: (from, gap)
for line in TAIL.splitlines():
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) == 3 and cells[0] not in ("Deliverable", "---"):
        README_ROWS[cells[0]] = (cells[1], cells[2])
print(len(README_ROWS), "rows in the README's table")
for k, (f, g) in README_ROWS.items():
    print(f"  {k} | {f} | {g}")

5 rows in the README's table
  current loop's feedback | `current-zero`: the zero's error and its start-up time | no rail, reference or temperature channel in the rows
  velocity loop's feedback | `velocity`: noise against delay, the rest verdict, the crawl | the brush term's form under load; the blend and the rest verdict's form are open
  position loop's feedback | `position`: noise against delay, the held-out residual, the lash as a bound | the grade at speed is bounded by the stamp lead
  thermal derate and cutoff | `winding-temperature` | no thermal capture
  valid flags for the faults | `speed`, `velocity` | the tap floor's rung is open; stage 3's dropout rate flag was not run


In [81]:
ROWS6 = []


def deliverable(name, from_, numbers, gap):
    """One row of the README's table with the files' numbers."""
    readme_from, readme_gap = README_ROWS[name]
    row = {"deliverable": name, "from": from_, "readme_from": readme_from,
           "numbers": numbers, "readme_gap": readme_gap, "gap": gap}
    ROWS6.append({**row, "sources": sources(numbers, gap)})


CZ_GAPS = node("cz", "gaps")
deliverable(
    "current loop's feedback", "current-zero",
    f"zero error all runs {Q('cz', 'design_all_runs_rms_counts')} rms, "
    f"worst {Q('cz', 'design_all_runs_worst_counts')}; warm "
    f"{Q('cz', 'design_warm_rms_counts')}; start-up to half a count "
    f"{Q('cz', 'start_rule_settle_to_half_count_fed_s')} of feed; hold p90 "
    f"at 60 s {Q('cz', 'hold_p90_counts/60s')}; noise per tick "
    f"{Q('cz', 'trough_sd_per_tick_counts/coast')}; counts per A "
    f"{Q('spd', 'constants_used/counts_per_a')}",
    f"the current-zero file lists {len(CZ_GAPS)} <cz:gaps>, among them: "
    f"{CZG(0)}; {CZG(1)}; {CZG(7)}")

In [82]:
deliverable(
    "velocity loop's feedback", "velocity, speed",
    "over the floor at a crawl, 5 ms of delay: back-EMF "
    f"{Q('vel', 'stage5_by_regime/0/noise_rev_s')} rev/s own noise, bias "
    f"table {Q('vel', 'emf_bias_crawl_rev_s/table')}, product "
    f"{Q('vel', 'emf_bias_crawl_rev_s/product')}; the pot's two-point "
    f"{Q('vel', 'stage5_by_regime/2/noise_rev_s')} rev/s; under the floor "
    f"the pot at 50 ms {Q('vel', 'stage5_by_regime/3/noise_rev_s')} rev/s; "
    f"crossover {Q('vel', XW + '/crawl_rms')}; blends sw5 "
    f"{Q('vel', 'blend_bias_crawl_rev_s/sw5')}, "
    f"{Q('vel', 'blend_own_sd_rev_s/crawl/sw5')}; cp5 "
    f"{Q('vel', 'blend_bias_crawl_rev_s/cp5')}, "
    f"{Q('vel', 'blend_own_sd_rev_s/crawl/cp5')}; rest verdict declares "
    f"{Q('vel', f'{RD}/{B2}/coast_down/median')}, drops "
    f"{Q('vel', f'{RS}/{B2}/median')}; crawl's angle-locked motion "
    f"{Q('vel', 'crawl_angle_locked_motion_rev_s/sd')}; mechanical tau "
    f"{Q('vel', 'mechanical_tau_ms')}; direct path "
    f"{Q('vel', 'emf_direct_path_share/row/r0_3')}",
    "the brush term's form under load; also open: the blend, the verdict's "
    "form, the kick's end, the stopped rotor's law")

In [83]:
deliverable(
    "position loop's feedback", "position",
    f"boxcar 16: {Q('pos', 'filter_boxcar16_noise_counts')} at "
    f"{Q('pos', 'filter_boxcar16_delay_ticks')}; per row "
    f"{Q('pos', 'pot_noise_row_counts')}; held-out residual "
    f"{Q('pos', 'table_term_heldout_deg_rms')} (table term), "
    f"{Q('pos', 'heldout_lead_applied_deg_rms')} (nothing out but the "
    f"lead); the lash as a bound {Q('pos', 'direction_bound_moving_deg')} "
    f"in motion, {Q('pos', 'direction_bound_rest_deg')} at rest, not an "
    f"offset; carries blind up to {num(max(SPAN))} deg "
    "<pos:carry_blind_span_deg>",
    "the grade at speed is bounded by the stamp lead "
    f"({Q('pos', 'share_stamp_lead')} of the error)")
deliverable(
    "thermal derate and cutoff", "winding-temperature",
    f"one node {Q('wt', 'one_node_gain_c_per_w')}, "
    f"{Q('wt', 'one_node_tau_s')}, {Q('wt', 'one_node_c_j_per_c')}; rise at "
    f"3 s {Q('wt', 'one_node_rise_at_3s_c_per_w')}; across seats "
    f"{Q('wt', 'floor_across_seats_c')}; graded to {GRADED}; plateau graded "
    f"{Q('wt', 'plateau_graded')}; second node's heat capacity "
    f"{Q('wt', 'two_node_slow_c_j_per_c')}, tau at least "
    f"{Q('wt', 'two_node_slow_tau_s_min')}",
    f"no thermal capture; no graded basis past {GRADED} of drive from a "
    "rest")

In [84]:
deliverable(
    "valid flags for the faults", "speed, velocity",
    f"speed valid on {Q('spd', COV + '/valid_pct')} % of driven rows (floor"
    f" {Q('spd', 'valid_rule/tap_floor_ticks')}, start "
    f"{Q('spd', 'valid_rule/start_rows')}, dropout "
    f"{Q('spd', 'valid_rule/dropout_frac')}); camera sitting "
    f"{Q('vel', 'emf_valid_share_camera')}; dropout rule hit rate "
    f"{Q('spd', DR + '/hit_rate')}, precision {Q('spd', DR + '/precision')}"
    f"; rest verdict: false rest "
    f"{Q('vel', 'rest_false_rest_10_35_deg_s/band 2 50')} at 10 to 35 "
    f"deg/s, a stall declared in "
    f"{Q('vel', f'{RD}/{B2}/stall_in_hold/median')}; zero fed on "
    f"{Q('cz', 'feed_share_awake_rows')} of awake rows",
    f"the tap floor's rung is open ({num(FLOOR_Q['value'])} against "
    f"{num(FLOOR_Q['hi'])} <spd:valid_rule/tap_floor_ticks>); the dropout "
    "rate flag of stage 3 was not run")
assert [r["deliverable"] for r in ROWS6] == list(README_ROWS)
show(ROWS6, ["deliverable", "readme_from", "numbers", "readme_gap", "gap"],
     ["deliverable", "the README's from", "numbers", "the README's gap",
      "the result files' gap"])

| deliverable | the README's from | numbers | the README's gap | the result files' gap |
|---|---|---|---|---|
| current loop's feedback | `current-zero`: the zero's error and its start-up time | zero error all runs 0.408 counts `cz:design_all_runs_rms_counts` rms, worst 1.967 counts `cz:design_all_runs_worst_counts`; warm 0.411 [0.373, 0.447] counts `cz:design_warm_rms_counts`; start-up to half a count 0.0625 [-, 0.303] s `cz:start_rule_settle_to_half_count_fed_s` of feed; hold p90 at 60 s 0.946 [0.877, 0.992] counts `cz:hold_p90_counts/60s`; noise per tick 1.109 [1.038, 1.463] counts `cz:trough_sd_per_tick_counts/coast`; counts per A 1135.0 counts/A `spd:constants_used/counts_per_a` | no rail, reference or temperature channel in the rows | the current-zero file lists 11 `cz:gaps`, among them: no rail, reference or temperature channel in the rows `cz:gaps/0/gap`; the step's shape off 4.9 V rests on one day `cz:gaps/1/gap`; nothing between 80 and 100 % duty `cz:gaps/7/gap` |
| velocity loop's feedback | `velocity`: noise against delay, the rest verdict, the crawl | over the floor at a crawl, 5 ms of delay: back-EMF 1.718 `vel:stage5_by_regime/0/noise_rev_s` rev/s own noise, bias table 0.54 [0.437, 0.643] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/table`, product 1.382 [1.198, 1.566] rev/s, +/- se over segments `vel:emf_bias_crawl_rev_s/product`; the pot's two-point 7.434 `vel:stage5_by_regime/2/noise_rev_s` rev/s; under the floor the pot at 50 ms 1.470 `vel:stage5_by_regime/3/noise_rev_s` rev/s; crossover 103.0 [66.15, 115.2] ms of window, 5-95 % of 200 resamplings, never 0% `vel:crossover_window_ms/crawl_rms`; blends sw5 0.404 [0.278, 0.53] rev/s, +/- se `vel:blend_bias_crawl_rev_s/sw5`, 3.426 rev/s `vel:blend_own_sd_rev_s/crawl/sw5`; cp5 0.0442 [9.35e-05, 0.0883] rev/s, +/- se `vel:blend_bias_crawl_rev_s/cp5`, 3.517 rev/s `vel:blend_own_sd_rev_s/crawl/cp5`; rest verdict declares 56.35 ms, median `vel:rest_declare_ms/band 2 T50/coast_down/median`, drops 7.725 ms, median `vel:rest_drop_ms/band 2 T50/median`; crawl's angle-locked motion 2.236 rev/s `vel:crawl_angle_locked_motion_rev_s/sd`; mechanical tau 18.29 [11.94, 26.66] ms, median [10-90 %] of the kept steps `vel:mechanical_tau_ms`; direct path 0.788 [0.74, 0.835] of the volts' step / Ke, +/- se `vel:emf_direct_path_share/row/r0_3` | the brush term's form under load; the blend and the rest verdict's form are open | the brush term's form under load; also open: the blend, the verdict's form, the kick's end, the stopped rotor's law |
| position loop's feedback | `position`: noise against delay, the held-out residual, the lash as a bound | boxcar 16: 0.139 [0.139, 0.161] counts `pos:filter_boxcar16_noise_counts` at 7.500 ticks `pos:filter_boxcar16_delay_ticks`; per row 1.031 [1.027, 1.160] counts `pos:pot_noise_row_counts`; held-out residual 0.165 [0.16, 0.17] deg `pos:table_term_heldout_deg_rms` (table term), 0.41 [0.376, 0.446] deg `pos:heldout_lead_applied_deg_rms` (nothing out but the lead); the lash as a bound 0.5 deg `pos:direction_bound_moving_deg` in motion, 0.3 deg `pos:direction_bound_rest_deg` at rest, not an offset; carries blind up to 0.772 deg `pos:carry_blind_span_deg` | the grade at speed is bounded by the stamp lead | the grade at speed is bounded by the stamp lead (0.479 share `pos:share_stamp_lead` of the error) |
| thermal derate and cutoff | `winding-temperature` | one node 23.24 [14.27, 23.85] C/W (fold_lo 14.27, fold_hi 23.24, stat_factor 1.012) `wt:one_node_gain_c_per_w`, 2.375 [1.203, 3.227] s (fold_lo 1.400, fold_hi 2.699, stat_factor 1.044) `wt:one_node_tau_s`, 0.102 [0.0625, 0.135] J/C (fold_lo 0.0981, fold_hi 0.117) `wt:one_node_c_j_per_c`; rise at 3 s 16.67 [12.60, 16.67] C/W `wt:one_node_rise_at_3s_c_per_w`; across seats 1.549 [0.769, 2.786] C `wt:floor_across_seats_c`; graded to 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max`; plateau graded False `wt:plateau_graded`; second node's heat capacity 0.761 [0.511, 1.268] J/C `wt:two_node_slow_c_j_per_c`, tau at least 35 s `wt:two_node_slow_tau_s_min` | no thermal capture | no thermal capture; no graded basis past 3.093 [3.092, 3.433] s, the same-seat pairs' span `wt:graded_drive_s_max` of drive from a rest |
| valid flags for the faults | `speed`, `velocity` | speed valid on 85.21 `spd:valid_coverage_by_set/all/valid_pct` % of driven rows (floor 84 [72, 96] ticks; bounds: the failing rung below, the first rung with 10 runs or more `spd:valid_rule/tap_floor_ticks`, start 16 rows, 4 L / R rounded up `spd:valid_rule/start_rows`, dropout 0.25 of the one-pole mean `spd:valid_rule/dropout_frac`); camera sitting 0.843 of driven rows `vel:emf_valid_share_camera`; dropout rule hit rate 1 `spd:dropout_rule_vs_stage3/reference 32 rows/hit_rate`, precision 0.957 `spd:dropout_rule_vs_stage3/reference 32 rows/precision`; rest verdict: false rest 0 share of rows `vel:rest_false_rest_10_35_deg_s/band 2 50` at 10 to 35 deg/s, a stall declared in 55.33 ms, median `vel:rest_declare_ms/band 2 T50/stall_in_hold/median`; zero fed on 0.911 share `cz:feed_share_awake_rows` of awake rows | the tap floor's rung is open; stage 3's dropout rate flag was not run | the tap floor's rung is open (84 against 96 `spd:valid_rule/tap_floor_ticks`); the dropout rate flag of stage 3 was not run |

The README's gap is the short form of the last column: it names the first of each deliverable's gaps, and section 6 holds the full list with the captures. The position loop's row reads the lash as a bound, 0.5 deg in motion and 0.3 at rest, with no offset. The current loop's row names the first of the 11 gaps the current-zero file lists, that the rows carry no rail, reference or temperature channel. The valid flags' row names the tap floor's rung, open between 84 and 96 ticks, and stage 3's dropout rate flag, which was not run.

### The result file

Every number in the result file is one servo on one board, mg90-d on dev-v006-2A-D, and none of it has been repeated on a second servo. The file holds no new measurement. Each key is a table of this notebook, its rows' text carrying the citations, and each row lists its sources. `closing_table` is section 2, `stage3_rows` section 3, `pairs` and `discrepancies` section 4, `open_choices` section 5, `gaps` section 6 and `stages_5_to_7` this section. No key names an option of record for the four open choices. Four tables go to csv files beside the json, the closing table, stage 3's rows, the pairs and the gaps.

In [85]:
QUANTITIES = {"closing_table": ROWS1, "stage3_rows": ROWS3, "pairs": PAIRS,
              "discrepancies": DISC, "open_choices": CHOICES,
              "gaps": GAPS, "stages_5_to_7": ROWS6}
NOTES = (
    "Stage 4 closing table, one servo on one board: mg90-d on "
    "dev-v006-2A-D. No capture of its own and no estimator run: every "
    "value is read from the pair's result files of 3-servo/compensation "
    "(s3) and 4-estimators current-zero (cz), position (pos), speed (spd), "
    "velocity (vel) and winding-temperature (wt), cited in the text as "
    "<file:key>, a key path joined by '/', a csv cell as "
    "<file:.table.csv[column=value].column>. 'derived' marks arithmetic on "
    "two cited keys, used to compare only. Brackets as the files give "
    "them. The state kept, the confidence words, the dispositions, the "
    "candidate causes, the speed and velocity notebooks' gaps and the "
    "captures are this notebook's text. No key names an option of record "
    "for the four open choices. Tables in .closing.csv, .stage3.csv, "
    ".pairs.csv and .gaps.csv.")
path = results.write(STAGE, EXPERIMENT, PAIR_DIR.resolve(), pair=PAIR,
                     notes=NOTES, **QUANTITIES)

In [86]:
def flat(rows):
    """Rows as a frame with list fields joined, for a csv."""
    return pd.DataFrame([{k: "; ".join(map(str, v)) if isinstance(v, list)
                          else v for k, v in r.items()} for r in rows])


base = results.path(STAGE, EXPERIMENT, pair=PAIR)
for suffix, rows in ((".closing.csv", ROWS1), (".stage3.csv", ROWS3),
                     (".pairs.csv", PAIRS), (".gaps.csv", GAPS)):
    flat(rows).to_csv(base.with_suffix(suffix), index=False)
written = results.read(STAGE, EXPERIMENT, pair=PAIR)
print("wrote", path, "with", len(QUANTITIES), "keys, capture",
      written["capture"])
print({k: len(v) for k, v in QUANTITIES.items()}, "rows")

wrote stages/4-estimators/compensation/results/mg90-d__dev-v006-2A-D.json with 7 keys, capture telemetry/mg90-d__dev-v006-2A-D
{'closing_table': 13, 'stage3_rows': 17, 'pairs': 36, 'discrepancies': 4, 'open_choices': 4, 'gaps': 35, 'stages_5_to_7': 5} rows
